## Prepare Psympact Data for LLM Paper

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import patsy
import statsmodels.api as sm

In [2]:
df = pd.read_csv("data/data_psympact_t1_KI_260320.csv")
df.drop(columns = ["lfdn", "v_1024","v_1026", "v_1327"], inplace = True)

In [3]:
df_list1 = df.columns.tolist()

In [4]:
df.drop_duplicates(subset="p_0001", keep="last", inplace=True)

### Adjust Variable Naming

In [5]:
# ---------------------------
# 1) Rename columns
# ---------------------------
rename_map = {
    # IDs / timing
    "p_0001": "id",
    "lfdn": "case_id",  # if present
    "date_of_last_access": "last_access_date",

    # ---------------------------
    # T0 sociodemographics
    # ---------------------------
    "v_12": "age",
    "v_11": "gender",
    "v_13": "education",
    "v_1024": "education_other_text",   # if present
    "v_1025": "vocational_degree",
    "v_1026": "vocational_degree_other_text",  # if present

    # ---------------------------
    # T0 diagnoses / conditions (multi-select)
    # ---------------------------
    "v_1310": "dx_hypertension",
    "v_1311": "dx_allergy",
    "v_1312": "dx_chronic_back_pain",
    "v_1313": "dx_sleep_disorder",
    "v_1314": "dx_joint_disease_arthrosis_rheuma",
    "v_1315": "dx_psych",
    "v_1316": "dx_migraine",
    "v_1317": "dx_heart_disease",
    "v_1318": "dx_chronic_bronchitis",
    "v_1319": "dx_diabetes",
    "v_1320": "dx_osteoporosis",
    "v_1321": "dx_liver_disease_fattyliver_hepatitis_cirrhosis",
    "v_1322": "dx_asthma",
    "v_1323": "dx_stroke",
    "v_1324": "dx_cancer",
    "v_1325": "dx_none_of_these_selected",
    "v_1333": "dx_no_answer_selected",

    # ---------------------------
    # EQ-5D
    # ---------------------------
    "v_1509": "eq5d_usual_activities",
    "v_1512": "eq5d_vas_today",

    # ---------------------------
    # Employment / social
    # ---------------------------
    "v_15": "employ_student",
    "v_16": "employ_parttime",
    "v_17": "employ_fulltime",
    "v_18": "employ_carework",
    "v_19": "employ_unemployed",
    "v_20": "employ_retired",
    "v_21": "employ_work_disabled",
    "v_22": "relationship_status",
    "v_26": "friends_meet_freq",

    # ---------------------------
    # PHQ-9 items
    # ---------------------------
    "v_688": "phq_interest",
    "v_689": "phq_depressed",
    "v_690": "phq_sleep",
    "v_691": "phq_tired",
    "v_692": "phq_appetite",
    "v_693": "phq_failure",
    "v_694": "phq_concentration",
    "v_695": "phq_psychomotor",
    "v_696": "phq_suicidal",

    # ---------------------------
    # GAD-7 items
    # ---------------------------
    "v_697": "gad_nervous",
    "v_698": "gad_control_worry",
    "v_699": "gad_excess_worry",
    "v_700": "gad_relax",
    "v_701": "gad_restless",
    "v_702": "gad_irritable",
    "v_703": "gad_afraid",

    # ---------------------------
    # Mental health service use
    # ---------------------------
    "v_221": "mh_psychotherapy_6m",
    "v_222": "mh_psychiatry_med_6m",
    "v_223": "mh_clinic_psych_selected",
    "v_224": "mh_clinic_psych_days",
    "v_225": "mh_clinic_psychosomatic_selected",
    "v_226": "mh_clinic_psychosomatic_days",
    "v_227": "mh_clinic_dayclinic_selected",
    "v_228": "mh_clinic_dayclinic_days",
    "v_229": "mh_clinic_er_selected",
    "v_230": "mh_clinic_er_days",
    "v_231": "mh_clinic_detox_selected",
    "v_232": "mh_clinic_detox_days",
    "v_233": "mh_clinic_rehab_selected",
    "v_234": "mh_clinic_rehab_days",
    "v_235": "mh_clinic_other_selected",
    "v_236": "mh_clinic_other_text",
    "v_304": "mh_clinic_none_selected",

    # ---------------------------
    # AI use
    # ---------------------------
    "v_1628": "ai_use_any",
    "v_1629": "ai_nouse_no_need",
    "v_1630": "ai_nouse_privacy_security",
    "v_1631": "ai_nouse_no_trust",
    "v_1632": "ai_nouse_no_knowhow",
    "v_1633": "ai_nouse_cost",
    "v_1634": "ai_nouse_ethics",
    "v_1635": "ai_nouse_other_selected",
    "v_1636": "ai_nouse_other_text",

    "v_1637": "ai_prog_chatgpt",
    "v_1638": "ai_prog_gemini",
    "v_1639": "ai_prog_claude",
    "v_1640": "ai_prog_grok",
    "v_1641": "ai_prog_perplexity",
    "v_1642": "ai_prog_llama",
    "v_1643": "ai_prog_deepseek",
    "v_1644": "ai_prog_mistral",
    "v_1645": "ai_prog_copilot",
    "v_1646": "ai_prog_other_selected",

    "v_1647": "ai_use_health",
    "v_1648": "ai_health_nouse_no_need",
    "v_1649": "ai_health_nouse_misinformation_fear",
    "v_1650": "ai_health_nouse_privacy",
    "v_1651": "ai_health_nouse_liability_unclear",
    "v_1652": "ai_health_nouse_no_knowhow",
    "v_1653": "ai_health_nouse_ethics",
    "v_1654": "ai_health_nouse_other_selected",
    "v_1655": "ai_health_nouse_other_text",

    "v_1664": "ai_health_symptom_check",
    "v_1669": "ai_health_explain_med_docs",
    "v_1670": "ai_health_fitness_nutrition",
    "v_1671": "ai_health_psych_emotional",

    "v_1673": "ai_health_avoided_doctor",
    "v_1674": "ai_health_prepared_doctor_visit",
    "v_1675": "ai_health_self_treatment",
    "v_1678": "ai_psych_increased_worry",
    "v_1679": "ai_health_doctor_visit_triggered",
    "v_1683": "ai_health_lifestyle_change",
    "v_1684": "ai_psych_found_comfort",

    "v_1685": "ai_psych_sit_anxiety_worry",
    "v_1686": "ai_psych_sit_self_doubt_low_mood",
    "v_1687": "ai_psych_sit_loneliness",
    "v_1688": "ai_psych_sit_relationship_conflict",
    "v_1689": "ai_psych_sit_self_reflection",
    "v_1690": "ai_psych_sit_grief_loss",
    "v_1691": "ai_psych_sit_impulsive_or_substance",
    "v_1692": "ai_psych_sit_acute_crisis",
    "v_1693": "ai_psych_sit_other_selected",
    "v_1694": "ai_psych_sit_other_text",

    "v_1700": "ai_answer_quality",
    "v_1701": "ai_friends_relationship_change",
    "v_1702": "ai_feel_dependent",
    "v_1703": "ai_hard_to_decide_alone",

    # ---------------------------
    # Additional HiTOP / DSM-5 CC / p-factor source variables
    # ---------------------------
    "v_1010": "dsm5cc_psy_01",
    "v_1011": "dsm5cc_psy_02",
    "v_1012": "dsm5cc_psy_03",
    "v_1013": "dsm5cc_psy_04",
    "v_1015": "dsm5cc_trauma",
    "v_1018": "dsm5cc_dis_01",
    "v_1019": "dsm5cc_dis_02",
    "v_1020": "dsm5cc_dis_03",

    "v_1534": "p_item_01",
    "v_1535": "p_item_02",
    "v_1536": "p_item_03",
    "v_1538": "p_item_04",
    "v_1621": "p_item_05",
    "v_1622": "p_item_06",
    "v_1623": "p_item_07",
    "v_1624": "p_item_08"
}

# PID5BF+M items v_712..v_747 -> pid5bfm_01..pid5bfm_36
pid_vars = [f"v_{i}" for i in range(712, 748)]
rename_map.update({v: f"pid5bfm_{j:02d}" for j, v in enumerate(pid_vars, start=1)})

df = df.rename(columns=rename_map)

# keep ID columns as strings
for col in ["id", "case_id"]:
    if col in df.columns:
        df[col] = df[col].astype(str)

# ---------------------------
# 2) Recode 0 -> NaN
# only where 0 is NOT a meaningful response
# ---------------------------
multi_select_cols = [
    # employment
    "employ_student", "employ_parttime", "employ_fulltime", "employ_carework",
    "employ_unemployed", "employ_retired", "employ_work_disabled",

    # diagnoses / conditions
    "dx_hypertension", "dx_allergy", "dx_chronic_back_pain", "dx_sleep_disorder",
    "dx_joint_disease_arthrosis_rheuma", "dx_psych", "dx_migraine", "dx_heart_disease",
    "dx_chronic_bronchitis", "dx_diabetes", "dx_osteoporosis",
    "dx_liver_disease_fattyliver_hepatitis_cirrhosis", "dx_asthma", "dx_stroke",
    "dx_cancer", "dx_none_of_these_selected", "dx_no_answer_selected",

    # clinic checkboxes
    "mh_clinic_none_selected",
    "mh_clinic_psych_selected", "mh_clinic_psychosomatic_selected", "mh_clinic_dayclinic_selected",
    "mh_clinic_er_selected", "mh_clinic_detox_selected", "mh_clinic_rehab_selected",
    "mh_clinic_other_selected",

    # AI non-use reasons
    "ai_nouse_no_need", "ai_nouse_privacy_security", "ai_nouse_no_trust",
    "ai_nouse_no_knowhow", "ai_nouse_cost", "ai_nouse_ethics", "ai_nouse_other_selected",

    # AI programs used
    "ai_prog_chatgpt", "ai_prog_gemini", "ai_prog_claude", "ai_prog_grok",
    "ai_prog_perplexity", "ai_prog_llama", "ai_prog_deepseek", "ai_prog_mistral",
    "ai_prog_copilot", "ai_prog_other_selected",

    # AI health non-use reasons
    "ai_health_nouse_no_need", "ai_health_nouse_misinformation_fear",
    "ai_health_nouse_privacy", "ai_health_nouse_liability_unclear",
    "ai_health_nouse_no_knowhow", "ai_health_nouse_ethics", "ai_health_nouse_other_selected",
]

# numeric columns except IDs
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
zero_to_na_cols = [c for c in numeric_cols if c not in multi_select_cols]

df[zero_to_na_cols] = df[zero_to_na_cols].mask(df[zero_to_na_cols].eq(0), np.nan)

# ---------------------------
# 3) HiTOP scoring using renamed variables only
# ---------------------------

df= df.copy()

# P factor
p_factor = [
    "p_item_01", "p_item_03", "p_item_04",
    "p_item_05", "p_item_06", "p_item_07", "p_item_08","p_item_02"
]
df["P"] = df[p_factor].mean(axis=1)

# Internalizing
int_items = ["pid5bfm_01", "pid5bfm_07", "pid5bfm_13", "pid5bfm_19", "pid5bfm_25", "pid5bfm_31"]
phq4_items = ["gad_nervous", "gad_control_worry", "phq_interest", "phq_depressed"]

int_pid_mean = df[int_items].mean(axis=1)
int_phq_trauma_mean = df[phq4_items + ["dsm5cc_trauma"]].mean(axis=1)
df["INT"] = pd.concat([int_pid_mean, int_phq_trauma_mean], axis=1).mean(axis=1)

# Detachment
det_items = ["pid5bfm_04", "pid5bfm_10", "pid5bfm_16", "pid5bfm_22", "pid5bfm_28", "pid5bfm_34"]
df["DET"] = df[det_items].mean(axis=1)

# Thought disorder
tho_items = ["pid5bfm_05", "pid5bfm_11", "pid5bfm_17", "pid5bfm_23", "pid5bfm_29", "pid5bfm_35"]
dsm5cc_psy = ["dsm5cc_psy_01", "dsm5cc_psy_02", "dsm5cc_psy_03", "dsm5cc_psy_04"]
df["THO"] = df[tho_items + dsm5cc_psy].mean(axis=1)

# Disinhibited Externalizing
extdis_items = ["pid5bfm_03", "pid5bfm_09", "pid5bfm_15", "pid5bfm_21", "pid5bfm_27", "pid5bfm_33"]
dsm5cc_dis = ["dsm5cc_dis_01", "dsm5cc_dis_02", "dsm5cc_dis_03"]
df["EXTDI"] = df[extdis_items + dsm5cc_dis].mean(axis=1)

# Antagonistic Externalizing
extant_items = ["pid5bfm_02", "pid5bfm_08", "pid5bfm_14", "pid5bfm_20", "pid5bfm_26", "pid5bfm_32"]
df["EXTANT"] = df[extant_items].mean(axis=1)

### Create overview of AI usage 

In [6]:

# --------------------------------------------------
# 0) One row per person
# --------------------------------------------------
df_tab = df.drop_duplicates(subset="id").copy()

general_usage_col = "ai_use_any"   # 1 yes, 2 no, 0/NaN missing
df_tab[general_usage_col] = pd.to_numeric(df_tab[general_usage_col], errors="coerce").fillna(0)

# --------------------------------------------------
# 1) Helpers
# --------------------------------------------------
usage_labels = {1: "Yes", 2: "No", 0: "Missing"}

def add_pcts(tbl, n_total, n_ai_users):
    """Add pct_total and pct_ai_users to a table with column 'n'."""
    if tbl is None or len(tbl) == 0:
        return pd.DataFrame(columns=["section", "item", "n", "pct_total", "pct_ai_users"])

    tbl = tbl.copy()
    if "percent" in tbl.columns:
        tbl = tbl.drop(columns=["percent"])

    tbl["pct_total"] = tbl["n"] / n_total * 100
    tbl["pct_ai_users"] = np.where(n_ai_users > 0, tbl["n"] / n_ai_users * 100, np.nan)
    return tbl[["section", "item", "n", "pct_total", "pct_ai_users"]]


def any_selected_1_4(df_sub, cols, filter_col="ai_use_health"):
    """
    Return:
    - 1 if any item in cols is between 1 and 4
    - 0 if none is between 1 and 4
    - NaN if all items are NaN

    Additional rule:
    - force 0 if filter_col == 2 (e.g., no AI health use → no selection)
    """

    # keep only existing columns
    cols = [c for c in cols if c in df_sub.columns]
    if not cols:
        return pd.Series(np.nan, index=df_sub.index)

    # convert to numeric safely
    tmp = df_sub[cols].apply(pd.to_numeric, errors="coerce")

    # check if any value is between 1 and 4
    selected = tmp.ge(1) & tmp.le(4)
    any_selected = selected.any(axis=1)

    # identify rows where all items are missing
    all_missing = tmp.isna().all(axis=1)

    # base output
    out = pd.Series(
        np.where(any_selected, 1, 0),
        index=df_sub.index,
        dtype="float"
    )

    # set NaN if all items are missing
    out[all_missing] = np.nan

    # apply filter logic: no AI health use → must be 0
    if filter_col in df_sub.columns:
        mask_no_use = df_sub[filter_col] == 2
        out.loc[mask_no_use] = 0

    return out


def in_1_4_mask(df_sub, cols):
    """
    Return a boolean DataFrame indicating whether each item is between 1 and 4.
    Useful for per-item counts.
    """
    cols = [c for c in cols if c in df_sub.columns]
    if not cols:
        return pd.DataFrame(index=df_sub.index)

    return df_sub[cols].apply(lambda s: pd.to_numeric(s, errors="coerce").between(1, 4), axis=0)


def make_usage_table(df_all, df_ai, col, section, n_total, n_ai_users, labels=usage_labels):
    """Create a Yes/No/Missing usage table for total sample and AI users."""
    total = (
        df_all.groupby(col)["id"].nunique()
        .reindex([1, 2, 0], fill_value=0)
        .rename("n")
        .reset_index()
    )

    ai = (
        df_ai.groupby(col)["id"].nunique()
        .reindex([1, 2, 0], fill_value=0)
        .rename("n_ai")
        .reset_index()
    )

    out = total.merge(ai, on=col, how="left").fillna({"n_ai": 0})
    out["section"] = section
    out["item"] = out[col].map(labels)
    out["pct_total"] = out["n"] / n_total * 100
    out["pct_ai_users"] = np.where(n_ai_users > 0, out["n_ai"] / n_ai_users * 100, np.nan)

    return out[["section", "item", "n", "pct_total", "pct_ai_users"]]


def make_reason_table(
    df_nonusers,
    df_ai_nonusers,
    cols,
    labels,
    section,
    n_total,
    n_ai_users
):
    """
    Create a table for binary non-usage reasons.
    Uses total sample denominator for pct_total and AI-user denominator for pct_ai_users.
    """
    cols = [c for c in cols if c in df_nonusers.columns]
    if not cols:
        return pd.DataFrame(columns=["section", "item", "n", "pct_total", "pct_ai_users"])

    tmp = pd.DataFrame({"var": cols})
    tmp["n"] = [(df_nonusers[c] == 1).sum() for c in cols]
    tmp["n_ai"] = [(df_ai_nonusers[c] == 1).sum() for c in cols]
    tmp["section"] = section
    tmp["item"] = tmp["var"].map(labels)
    tmp["pct_total"] = tmp["n"] / n_total * 100
    tmp["pct_ai_users"] = np.where(n_ai_users > 0, tmp["n_ai"] / n_ai_users * 100, np.nan)

    n_miss_total = int(df_nonusers[cols].isna().all(axis=1).sum())
    n_miss_ai = int(df_ai_nonusers[cols].isna().all(axis=1).sum())

    miss_row = pd.DataFrame([{
        "section": section,
        "item": "Missing (all reasons NaN)",
        "n": n_miss_total,
        "pct_total": (n_miss_total / n_total * 100) if n_total > 0 else np.nan,
        "pct_ai_users": (n_miss_ai / n_ai_users * 100) if n_ai_users > 0 else np.nan,
    }])

    return pd.concat(
        [tmp[["section", "item", "n", "pct_total", "pct_ai_users"]], miss_row],
        ignore_index=True
    )


def make_multiselect_table(
    df_sub,
    cols,
    labels,
    section,
    n_total,
    n_ai_users,
    any_label=None,
    missing_label=None
):
    """
    Create a table for multi-select items coded as selected when value is 1-4.
    Optionally adds:
      - row for 'any selected'
      - row for 'all missing'
    """
    cols = [c for c in cols if c in df_sub.columns]
    if not cols:
        return pd.DataFrame(columns=["section", "item", "n", "pct_total", "pct_ai_users"])

    mask = in_1_4_mask(df_sub, cols)
    rows = []

    if any_label is not None:
        rows.append({
            "section": section,
            "item": any_label,
            "n": int(mask.any(axis=1).sum())
        })

    rows += [
        {
            "section": section,
            "item": labels.get(c, c),
            "n": int(mask[c].sum())
        }
        for c in cols
    ]

    if missing_label is not None:
        rows.append({
            "section": section,
            "item": missing_label,
            "n": int(df_sub[cols].isna().all(axis=1).sum())
        })

    return add_pcts(pd.DataFrame(rows), n_total=n_total, n_ai_users=n_ai_users)


# --------------------------------------------------
# 2) Derive health-related AI usage from detailed items
# --------------------------------------------------
medical_cols = [
    "ai_health_symptom_check",
    "ai_health_explain_med_docs",
    "ai_health_fitness_nutrition",
]

psych_sit_cols = [
    "ai_psych_sit_anxiety_worry",
    "ai_psych_sit_self_doubt_low_mood",
    "ai_psych_sit_loneliness",
    "ai_psych_sit_relationship_conflict",
    "ai_psych_sit_self_reflection",
    "ai_psych_sit_grief_loss",
    "ai_psych_sit_impulsive_or_substance",
    "ai_psych_sit_acute_crisis",
    "ai_psych_sit_other_selected",
]

df_tab["medical_ai_usage"] = any_selected_1_4(df_tab, medical_cols)
df_tab["emotional_ai_usage"] = any_selected_1_4(df_tab, psych_sit_cols)

df_tab["med_or_emo_ai_use"] = np.where(
    (df_tab["medical_ai_usage"] == 1) | (df_tab["emotional_ai_usage"] == 1),
    1,
    np.where(
        df_tab["medical_ai_usage"].isna() & df_tab["emotional_ai_usage"].isna(),
        np.nan,
        0
    )
)

# Recode to 1=yes, 2=no, 0=missing for usage table display
df_tab["health_usage_derived"] = np.where(
    df_tab["med_or_emo_ai_use"] == 1, 1,
    np.where(df_tab["med_or_emo_ai_use"] == 0, 2, 0)
)

# --------------------------------------------------
# 3) Denominators
# --------------------------------------------------
n_total = df_tab["id"].nunique()
df_ai_users = df_tab.loc[df_tab[general_usage_col] == 1].copy()
n_ai_users = df_ai_users["id"].nunique()

# --------------------------------------------------
# 4) GENERAL AI tables
# --------------------------------------------------
general_usage_table = make_usage_table(
    df_all=df_tab,
    df_ai=df_ai_users,
    col=general_usage_col,
    section="General AI usage",
    n_total=n_total,
    n_ai_users=n_ai_users,
)

general_nonusage_cols = [
    "ai_nouse_no_need",
    "ai_nouse_privacy_security",
    "ai_nouse_no_trust",
    "ai_nouse_no_knowhow",
    "ai_nouse_cost",
    "ai_nouse_ethics",
    "ai_nouse_other_selected",
]
general_nonusage_labels = {
    "ai_nouse_no_need": "No need / not useful",
    "ai_nouse_privacy_security": "Privacy / security concerns",
    "ai_nouse_no_trust": "Lack of trust in AI",
    "ai_nouse_no_knowhow": "Don’t know how to use",
    "ai_nouse_cost": "Too expensive / paywall",
    "ai_nouse_ethics": "Ethical concerns",
    "ai_nouse_other_selected": "Other (selected)",
}

df_general_nonusers = df_tab.loc[df_tab[general_usage_col] == 2].copy()
df_ai_general_nonusers = df_ai_users.loc[df_ai_users[general_usage_col] == 2].copy()

general_nonusage_table = make_reason_table(
    df_nonusers=df_general_nonusers,
    df_ai_nonusers=df_ai_general_nonusers,
    cols=general_nonusage_cols,
    labels=general_nonusage_labels,
    section="Reasons for general AI non-usage",
    n_total=n_total,
    n_ai_users=n_ai_users,
)

model_cols = [
    "ai_prog_chatgpt",
    "ai_prog_gemini",
    "ai_prog_claude",
    "ai_prog_grok",
    "ai_prog_perplexity",
    "ai_prog_llama",
    "ai_prog_deepseek",
    "ai_prog_mistral",
    "ai_prog_copilot",
    "ai_prog_other_selected",
]
model_labels = {
    "ai_prog_chatgpt": "ChatGPT",
    "ai_prog_gemini": "Gemini",
    "ai_prog_claude": "Claude",
    "ai_prog_grok": "Grok",
    "ai_prog_perplexity": "Perplexity",
    "ai_prog_llama": "Llama",
    "ai_prog_deepseek": "DeepSeek",
    "ai_prog_mistral": "Mistral",
    "ai_prog_copilot": "Copilot",
    "ai_prog_other_selected": "Other model (selected)",
}

model_table = pd.DataFrame(columns=["section", "item", "n", "pct_total", "pct_ai_users"])
model_cols_present = [c for c in model_cols if c in df_ai_users.columns]
if model_cols_present:
    rows = [
        {
            "section": "Models used among general AI users",
            "item": model_labels.get(c, c),
            "n": int((df_ai_users[c] == 1).sum())
        }
        for c in model_cols_present
    ]
    rows.append({
        "section": "Models used among general AI users",
        "item": "Missing (all model vars NaN)",
        "n": int(df_ai_users[model_cols_present].isna().all(axis=1).sum())
    })
    model_table = add_pcts(pd.DataFrame(rows), n_total=n_total, n_ai_users=n_ai_users)

df_general_users = df_ai_users.copy()
for c in ["ai_friends_relationship_change", "ai_feel_dependent", "ai_hard_to_decide_alone"]:
    if c in df_general_users.columns:
        df_general_users[c] = pd.to_numeric(df_general_users[c], errors="coerce")

side_rows = []

col_rel = "ai_friends_relationship_change"
if col_rel in df_general_users.columns:
    s = df_general_users[col_rel]
    side_rows += [
        {"section": "Side effects / social impact (general AI users)", "item": "Friends/relationships: improved (>0)", "n": int((s > 4).sum())},
        {"section": "Side effects / social impact (general AI users)", "item": "Friends/relationships: worsened (<0)", "n": int((s < 4).sum())},
        {"section": "Side effects / social impact (general AI users)", "item": "Friends/relationships: same (0)", "n": int((s == 4).sum())},
        {"section": "Side effects / social impact (general AI users)", "item": "Friends/relationships: missing", "n": int(s.isna().sum())},
    ]

for col, label in [
    ("ai_feel_dependent", "Feels dependent on AI (value > 1)"),
    ("ai_hard_to_decide_alone", "Hard to decide without AI (value > 1)"),
]:
    if col in df_general_users.columns:
        s = df_general_users[col]
        side_rows += [
            {"section": "Side effects / dependency (general AI users)", "item": label, "n": int((s > 1).sum())},
            {"section": "Side effects / dependency (general AI users)", "item": f"{label}: missing", "n": int(s.isna().sum())},
        ]

side_effects_table = add_pcts(pd.DataFrame(side_rows), n_total=n_total, n_ai_users=n_ai_users)

overview_table_1 = pd.concat(
    [general_usage_table, general_nonusage_table, model_table, side_effects_table],
    ignore_index=True
)
overview_table_1["pct_total"] = overview_table_1["pct_total"].round(1)
overview_table_1["pct_ai_users"] = overview_table_1["pct_ai_users"].round(1)

# --------------------------------------------------
# 5) HEALTH AI tables based on DERIVED usage
# --------------------------------------------------
health_usage_table = make_usage_table(
    df_all=df_tab,
    df_ai=df_ai_users,
    col="health_usage_derived",
    section="Health AI usage (derived: medical OR emotional)",
    n_total=n_total,
    n_ai_users=n_ai_users,
)

# users with any derived health-related AI usage
df_health_users = df_tab.loc[df_tab["med_or_emo_ai_use"] == 1].copy()

medical_labels = {
    "ai_health_symptom_check": "Symptom check / interpretation",
    "ai_health_explain_med_docs": "Explain medical documents",
    "ai_health_fitness_nutrition": "Fitness / nutrition",
}
health_med_table = make_multiselect_table(
    df_sub=df_health_users,
    cols=medical_cols,
    labels=medical_labels,
    section="AI for Medical Purposes (derived health AI users)",
    n_total=n_total,
    n_ai_users=n_ai_users,
    any_label="AI for Medical Purposes (any of 3; value 1–4)",
    missing_label="Missing (all medical items NaN)",
)

psych_sit_labels = {
    "ai_psych_sit_anxiety_worry": "Anxiety / worry",
    "ai_psych_sit_self_doubt_low_mood": "Self-doubt / low mood",
    "ai_psych_sit_loneliness": "Loneliness",
    "ai_psych_sit_relationship_conflict": "Relationship conflict",
    "ai_psych_sit_self_reflection": "Self-reflection",
    "ai_psych_sit_grief_loss": "Grief / loss",
    "ai_psych_sit_impulsive_or_substance": "Impulsivity / substance use",
    "ai_psych_sit_acute_crisis": "Acute crisis",
    "ai_psych_sit_other_selected": "Other situation (selected)",
}
health_emotional_table = make_multiselect_table(
    df_sub=df_health_users,
    cols=psych_sit_cols,
    labels=psych_sit_labels,
    section="AI for Emotional Purposes (derived health AI users)",
    n_total=n_total,
    n_ai_users=n_ai_users,
    any_label="Used AI for emotional purposes (any situation item value 1–4)",
    missing_label="Missing (all emotional vars NaN)",
)

downstream_cols = [
    "ai_health_avoided_doctor",
    "ai_health_prepared_doctor_visit",
    "ai_health_self_treatment",
    "ai_health_doctor_visit_triggered",
    "ai_psych_increased_worry",
    "ai_psych_found_comfort",
    "ai_health_lifestyle_change",
]
downstream_labels = {
    "ai_health_avoided_doctor": "Avoided seeing a doctor",
    "ai_health_prepared_doctor_visit": "Prepared better for a doctor visit",
    "ai_health_self_treatment": "Treated myself / changed behavior without a doctor",
    "ai_health_doctor_visit_triggered": "Decided to see a doctor",
    "ai_psych_increased_worry": "Increased worry / anxiety",
    "ai_psych_found_comfort": "Found comfort / reassurance",
    "ai_health_lifestyle_change": "Lifestyle change (e.g., diet/exercise)",
}
downstream_table = make_multiselect_table(
    df_sub=df_health_users,
    cols=downstream_cols,
    labels=downstream_labels,
    section="Downstream consequences (derived health AI users)",
    n_total=n_total,
    n_ai_users=n_ai_users,
    any_label=None,
    missing_label="Missing (all consequence vars NaN)",
)

health_nonusage_cols = [
    "ai_health_nouse_no_need",
    "ai_health_nouse_misinformation_fear",
    "ai_health_nouse_privacy",
    "ai_health_nouse_liability_unclear",
    "ai_health_nouse_no_knowhow",
    "ai_health_nouse_ethics",
    "ai_health_nouse_other_selected",
]
health_nonusage_labels = {
    "ai_health_nouse_no_need": "No need / not useful",
    "ai_health_nouse_misinformation_fear": "Fear of misinformation",
    "ai_health_nouse_privacy": "Privacy concerns",
    "ai_health_nouse_liability_unclear": "Liability / responsibility unclear",
    "ai_health_nouse_no_knowhow": "Don’t know how to use",
    "ai_health_nouse_ethics": "Ethical concerns",
    "ai_health_nouse_other_selected": "Other (selected)",
}

df_health_nonusers = df_tab.loc[df_tab["med_or_emo_ai_use"] == 0].copy()
df_ai_health_nonusers = df_ai_users.loc[df_ai_users["med_or_emo_ai_use"] == 0].copy()

health_nonusage_table = make_reason_table(
    df_nonusers=df_health_nonusers,
    df_ai_nonusers=df_ai_health_nonusers,
    cols=health_nonusage_cols,
    labels=health_nonusage_labels,
    section="Reasons for health AI non-usage (derived)",
    n_total=n_total,
    n_ai_users=n_ai_users,
)

overview_table_2 = pd.concat(
    [health_usage_table, health_med_table, health_emotional_table, downstream_table, health_nonusage_table],
    ignore_index=True
)
overview_table_2["pct_total"] = overview_table_2["pct_total"].round(1)
overview_table_2["pct_ai_users"] = overview_table_2["pct_ai_users"].round(1)

# --------------------------------------------------
# 6) Save
# --------------------------------------------------
overview_table_1.to_csv("general_ai_usage_psympact.csv", index=False)
overview_table_2.to_csv("health_ai_usage_psympact.csv", index=False)

## Overview of Health Ai Usage

In [7]:
# --------------------------------------------------
# 0) one row per person
# --------------------------------------------------
df_tab = df.drop_duplicates(subset="id").copy()

# --------------------------------------------------
# 1) helpers
# --------------------------------------------------
def fmt_n_pct(num, den):
    if pd.isna(den) or den == 0:
        return "—"
    return f"{int(num)}/{int(den)} ({num / den * 100:.1f}%)"


def summarize_sideeffects(df_sub):
    dep = pd.to_numeric(df_sub["ai_feel_dependent"], errors="coerce")
    dec = pd.to_numeric(df_sub["ai_hard_to_decide_alone"], errors="coerce")
    soc = pd.to_numeric(df_sub["ai_friends_relationship_change"], errors="coerce")

    dep_den = int(dep.notna().sum())
    dec_den = int(dec.notna().sum())
    soc_den = int(soc.notna().sum())

    dep_num = int((dep > 1).sum())
    dec_num = int((dec > 1).sum())
    soc_num = int(soc.ne(4).sum())   # unequal to 4

    return {
        "level_n": int(len(df_sub)),
        "Dependency valid n": dep_den,
        "Dependency (%)": fmt_n_pct(dep_num, dep_den),
        "Decision difficulty valid n": dec_den,
        "Decision difficulty (%)": fmt_n_pct(dec_num, dec_den),
        "Social alienation valid n": soc_den,
        "Social alienation (%)": fmt_n_pct(soc_num, soc_den),
    }

def add_group_rows(rows, data, panel, variable, group_col, level_order):
    # total valid N for the variable block in this panel
    n_variable = int(data[group_col].notna().sum())

    for lvl in level_order:
        sub = data.loc[data[group_col] == lvl].copy()
        vals = summarize_sideeffects(sub)
        rows.append({
            "Panel": panel,
            "Variable": variable,
            "n": n_variable,
            "Level": lvl,
            **vals
        })

# --------------------------------------------------
# 2) AI-use variables
# --------------------------------------------------
df_tab["ai_use_any"] = pd.to_numeric(df_tab["ai_use_any"], errors="coerce")
df_tab["ai_use"] = np.where(
    df_tab["ai_use_any"] == 1, 1,
    np.where(df_tab["ai_use_any"] == 2, 0, np.nan)
)

medical_cols = [
    "ai_health_symptom_check",
    "ai_health_explain_med_docs",
    "ai_health_fitness_nutrition",
]
df_tab["medical_ai_usage"] = any_selected_1_4(df_tab, medical_cols)

psych_sit_cols = [
    "ai_psych_sit_anxiety_worry",
    "ai_psych_sit_self_doubt_low_mood",
    "ai_psych_sit_loneliness",
    "ai_psych_sit_relationship_conflict",
    "ai_psych_sit_self_reflection",
    "ai_psych_sit_grief_loss",
    "ai_psych_sit_impulsive_or_substance",
    "ai_psych_sit_acute_crisis",
    "ai_psych_sit_other_selected",
]
df_tab["emotional_ai_usage"] = any_selected_1_4(df_tab, psych_sit_cols)

# Panel B: AI users with medical OR emotional use
df_tab["med_or_emo_ai_use"] = np.where(
    (df_tab["medical_ai_usage"] == 1) | (df_tab["emotional_ai_usage"] == 1),
    1,
    np.where(
        df_tab["medical_ai_usage"].isna() & df_tab["emotional_ai_usage"].isna(),
        np.nan,
        0
    )
)

# --------------------------------------------------
# 3) subgroup variables
# --------------------------------------------------

# Age
df_tab["age"] = pd.to_numeric(df_tab["age"], errors="coerce")
df_tab["age_group"] = pd.cut(
    df_tab["age"],
    bins=[18, 28, 38, 48, 58, 68, 78, np.inf],
    right=False,
    labels=["18-27", "28-37", "38-47", "48-57", "58-67", "68-77", "78+"]
)

# Gender
df_tab["gender"] = pd.to_numeric(df_tab["gender"], errors="coerce")
gender_vals = set(df_tab["gender"].dropna().unique())

if gender_vals.issubset({0, 1, 2}):
    gender_map = {0: "Female", 1: "Male", 2: "Other"}
else:
    gender_map = {1: "Female", 2: "Male", 3: "Other", 4: "Other"}

df_tab["gender_group"] = df_tab["gender"].map(gender_map)

# Self-rated functioning for usual activities
df_tab["eq5d_usual_activities"] = pd.to_numeric(df_tab["eq5d_usual_activities"], errors="coerce")
df_tab["usual_activities_group"] = df_tab["eq5d_usual_activities"].map({
    1: "No problems",
    2: "Slight problems",
    3: "Moderate problems",
    4: "Severe problems",
    5: "Unable"
})

# Lifetime psychiatric diagnosis
df_tab["dx_psych"] = pd.to_numeric(df_tab["dx_psych"], errors="coerce")
if "dx_no_answer_selected" in df_tab.columns:
    no_answer = pd.to_numeric(df_tab["dx_no_answer_selected"], errors="coerce").eq(1)
else:
    no_answer = pd.Series(False, index=df_tab.index)

df_tab["mental_dx_group"] = pd.Series(np.nan, index=df_tab.index, dtype="object")
df_tab.loc[~no_answer & (df_tab["dx_psych"] == 1), "mental_dx_group"] = "Yes"
df_tab.loc[~no_answer & (df_tab["dx_psych"] != 1), "mental_dx_group"] = "No"

# Psychotherapeutic or psychiatric treatment
pt = pd.to_numeric(df_tab["mh_psychotherapy_6m"], errors="coerce")
med = pd.to_numeric(df_tab["mh_psychiatry_med_6m"], errors="coerce")

df_tab["h_psycho_treatment"] = np.where(
    (pt == 1) | (med == 1),
    1,
    np.where(
        pt.isna() & med.isna(),
        np.nan,
        0
    )
)

df_tab["treatment_group"] = pd.Series(np.nan, index=df_tab.index, dtype="object")
df_tab.loc[df_tab["h_psycho_treatment"] == 1, "treatment_group"] = "Yes"
df_tab.loc[df_tab["h_psycho_treatment"] == 0, "treatment_group"] = "No"

# --------------------------------------------------
# 4) panel datasets
# --------------------------------------------------
df_panel_a = df_tab.loc[df_tab["ai_use"] == 1].copy()
df_panel_b = df_tab.loc[
    (df_tab["ai_use"] == 1) &
    (df_tab["med_or_emo_ai_use"] == 1)
].copy()


# --------------------------------------------------
# 5) build Table S2 (PSYMPACT)
# --------------------------------------------------
rows = []

for panel_name, dsub in [
    ("Panel A: All AI users", df_panel_a),
    ("Panel B: AI users with medical/emotional use", df_panel_b),
]:
    add_group_rows(
        rows, dsub, panel_name, "Age", "age_group",
        ["18-27", "28-37", "38-47", "48-57", "58-67", "68-77", "78+"]
    )

    add_group_rows(
        rows, dsub, panel_name, "Gender", "gender_group",
        ["Female", "Male", "Other"]
    )

    add_group_rows(
        rows, dsub, panel_name, "Self-rated functioning for usual activities", "usual_activities_group",
        ["No problems", "Slight problems", "Moderate problems", "Severe problems", "Unable"]
    )

    add_group_rows(
        rows, dsub, panel_name, "Lifetime psychiatric diagnosis", "mental_dx_group",
        ["Yes", "No"]
    )

    add_group_rows(
        rows, dsub, panel_name, "Psychotherapeutic or psychiatric treatment", "treatment_group",
        ["Yes", "No"]
    )

table_s2_psympact = pd.DataFrame(rows)

# remove empty subgroup rows
table_s2_psympact = table_s2_psympact.loc[table_s2_psympact["level_n"] > 0].copy()

# pretty version: show variable n only once per block
table_s2_psympact_print = table_s2_psympact.copy()
table_s2_psympact_print["n"] = table_s2_psympact_print["n"].astype("object")
dup_mask = table_s2_psympact_print.duplicated(subset=["Panel", "Variable"])
table_s2_psympact_print.loc[dup_mask, "n"] = ""


# save
table_s2_psympact.to_csv("table_s2_psympact.csv", index=False)

# --------------------------------------------------
# 6) build Table S3 (PSYMPACT)
# based on Panel B = users with medical/emotional use
# outcomes:
#   - ai_health_self_treatment
#   - ai_psych_found_comfort
#   - ai_health_prepared_doctor_visit
# --------------------------------------------------

def summarize_s3_outcomes(df_sub):
    self_treat = pd.to_numeric(df_sub["ai_health_self_treatment"], errors="coerce")
    comfort = pd.to_numeric(df_sub["ai_psych_found_comfort"], errors="coerce")
    doctor_prep = pd.to_numeric(df_sub["ai_health_prepared_doctor_visit"], errors="coerce")

    self_treat_den = int(self_treat.notna().sum())
    comfort_den = int(comfort.notna().sum())
    doctor_prep_den = int(doctor_prep.notna().sum())

    # assuming values 1-4 indicate endorsement/use, everything else = not endorsed
    self_treat_num = int(self_treat.between(1, 4).sum())
    comfort_num = int(comfort.between(1, 4).sum())
    doctor_prep_num = int(doctor_prep.between(1, 4).sum())

    return {
        "level_n": int(len(df_sub)),

        "Self-treatment valid n": self_treat_den,
        "Self-treatment (%)": fmt_n_pct(self_treat_num, self_treat_den),

        "Found comfort valid n": comfort_den,
        "Found comfort (%)": fmt_n_pct(comfort_num, comfort_den),

        "Prepared doctor visit valid n": doctor_prep_den,
        "Prepared doctor visit (%)": fmt_n_pct(doctor_prep_num, doctor_prep_den),
    }


def add_group_rows_s3(rows, data, variable, group_col, level_order):
    # valid N for subgroup variable itself within Panel B
    n_variable = int(data[group_col].notna().sum())

    for lvl in level_order:
        sub = data.loc[data[group_col] == lvl].copy()
        vals = summarize_s3_outcomes(sub)
        rows.append({
            "Variable": variable,
            "n": n_variable,
            "Level": lvl,
            **vals
        })


rows_s3 = []

add_group_rows_s3(
    rows_s3, df_panel_b, "Age", "age_group",
    ["18-27", "28-37", "38-47", "48-57", "58-67", "68-77", "78+"]
)

add_group_rows_s3(
    rows_s3, df_panel_b, "Gender", "gender_group",
    ["Female", "Male", "Other"]
)

add_group_rows_s3(
    rows_s3, df_panel_b, "Self-rated functioning for usual activities", "usual_activities_group",
    ["No problems", "Slight problems", "Moderate problems", "Severe problems", "Unable"]
)

add_group_rows_s3(
    rows_s3, df_panel_b, "Lifetime psychiatric diagnosis", "mental_dx_group",
    ["Yes", "No"]
)

add_group_rows_s3(
    rows_s3, df_panel_b, "Psychotherapeutic or psychiatric treatment", "treatment_group",
    ["Yes", "No"]
)

table_s3_psympact = pd.DataFrame(rows_s3)

# remove empty subgroup rows
table_s3_psympact = table_s3_psympact.loc[table_s3_psympact["level_n"] > 0].copy()

# pretty version: show variable n only once per block
table_s3_psympact_print = table_s3_psympact.copy()
table_s3_psympact_print["n"] = table_s3_psympact_print["n"].astype("object")
dup_mask = table_s3_psympact_print.duplicated(subset=["Variable"])
table_s3_psympact_print.loc[dup_mask, "n"] = ""

# save
table_s3_psympact.to_csv("table_s3_psympact.csv", index=False)

In [8]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, ttest_ind_from_stats

# =========================================================
# 1) SETTINGS
# =========================================================
# Uncomment this if you want to ensure one row per person:
# if "id" in df.columns:
#     df = df.drop_duplicates(subset="id").copy()

N_DH = 28910
N_PS = df["id"].nunique() if "id" in df.columns else len(df)

medical_cols = [
    "ai_health_symptom_check",
    "ai_health_explain_med_docs",
    "ai_health_fitness_nutrition",
]

emotional_cols = [
    "ai_psych_sit_anxiety_worry",
    "ai_psych_sit_self_doubt_low_mood",
    "ai_psych_sit_loneliness",
    "ai_psych_sit_relationship_conflict",
    "ai_psych_sit_self_reflection",
    "ai_psych_sit_grief_loss",
    "ai_psych_sit_impulsive_or_substance",
    "ai_psych_sit_acute_crisis",
    "ai_psych_sit_other_selected",
]

# =========================================================
# 2) HELPERS
# =========================================================
def any_selected_1_4(df_sub, cols, filter_col="ai_use_health"):
    cols = [c for c in cols if c in df_sub.columns]
    if not cols:
        return pd.Series(np.nan, index=df_sub.index)

    tmp = df_sub[cols].apply(pd.to_numeric, errors="coerce")
    selected = tmp.ge(1) & tmp.le(4)

    out = pd.Series(np.where(selected.any(axis=1), 1, 0), index=df_sub.index, dtype="float")
    out[tmp.isna().all(axis=1)] = np.nan

    if filter_col in df_sub.columns:
        out.loc[pd.to_numeric(df_sub[filter_col], errors="coerce").eq(2)] = 0

    return out


def yes_no_1_2(s):
    s = pd.to_numeric(s, errors="coerce")
    out = pd.Series(np.nan, index=s.index, dtype="float")
    out[s.eq(1)] = 1
    out[s.eq(2)] = 0
    return out


def selected_1_4(s):
    s = pd.to_numeric(s, errors="coerce")
    out = pd.Series(np.nan, index=s.index, dtype="float")
    out[s.between(1, 4)] = 1
    out[s.eq(5)] = 0
    return out


def present_1_2_3(s):
    s = pd.to_numeric(s, errors="coerce")
    out = pd.Series(np.nan, index=s.index, dtype="float")
    out[s.isin([1, 2, 3])] = 1
    out[s.eq(0)] = 0
    return out


def any_binary_yes(df_sub, cols):
    cols = [c for c in cols if c in df_sub.columns]
    if not cols:
        return pd.Series(np.nan, index=df_sub.index)

    tmp = df_sub[cols].apply(pd.to_numeric, errors="coerce")

    out = pd.Series(np.nan, index=tmp.index, dtype="float")
    out[tmp.eq(1).any(axis=1)] = 1
    out[tmp.isna().all(axis=1)] = np.nan
    out[(~tmp.eq(1).any(axis=1)) & (~tmp.isna().all(axis=1))] = 0

    return out


def bin_stat(s, base_n, count_only=False):
    s = pd.to_numeric(s, errors="coerce")
    yes = int((s == 1).sum())
    miss = int(s.isna().sum())

    if count_only:
        stat = f"{yes}"
    else:
        stat = f"{yes} ({100 * yes / base_n:.1f}%)" if base_n > 0 else "—"

    miss_txt = f"{miss} ({100 * miss / base_n:.1f}%)" if base_n > 0 else "—"
    return stat, miss_txt, yes, miss


def cont_stat(s, base_n):
    s = pd.to_numeric(s, errors="coerce")
    miss = int(s.isna().sum())
    x = s.dropna()

    stat = f"{x.mean():.1f} ({x.std(ddof=1):.1f})" if len(x) > 0 else "—"
    miss_txt = f"{miss} ({100 * miss / base_n:.1f}%)" if base_n > 0 else "—"
    return stat, miss_txt, x


def cramers_v_2x2(a_yes, a_base, a_miss, b_yes, b_base, b_miss):
    a_no = a_base - a_miss - a_yes
    b_no = b_base - b_miss - b_yes

    tab = np.array([[a_yes, a_no], [b_yes, b_no]])

    if (tab < 0).any() or (tab.sum(axis=0) == 0).any() or (tab.sum(axis=1) == 0).any():
        return "", ""

    chi2, p, _, _ = chi2_contingency(tab, correction=False)
    v = np.sqrt(chi2 / tab.sum())

    p_txt = "< .001" if p < .001 else f"= {p:.3f}"
    return f"χ²(1) = {chi2:.2f}, p {p_txt}", f"V = {v:.3f}"


def hedges_g(m1, sd1, n1, m2, sd2, n2):
    sp = np.sqrt(((n1 - 1) * sd1**2 + (n2 - 1) * sd2**2) / (n1 + n2 - 2))
    if sp == 0:
        return 0.0
    d = (m1 - m2) / sp
    j = 1 - (3 / (4 * (n1 + n2) - 9))
    return j * d


# =========================================================
# 3) PREPARE PSYMPACT DATA
# =========================================================
df = df.copy()

for c in medical_cols + emotional_cols + [
    "ai_use_any",
    "ai_use_health",
    "ai_answer_quality",
    "ai_friends_relationship_change",
    "ai_feel_dependent",
    "ai_hard_to_decide_alone",
]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Main usage variables
df["med_use"] = any_selected_1_4(df, medical_cols)
df["emo_use"] = any_selected_1_4(df, emotional_cols)
df["any_ai_use"] = yes_no_1_2(df["ai_use_any"])
df["health_related_ai_use"] = any_binary_yes(df, ["med_use", "emo_use"])

# ---------------------------------------------------------
# Side effects
# ---------------------------------------------------------
df["ai_friends_relationship_change_negative"] = df["ai_friends_relationship_change"].map({
    1: 3,
    2: 2,
    3: 1,
    4: 0,
    5: 0,
    6: 0,
    7: 0
})

df["ai_feel_dependent_neg"] = df["ai_feel_dependent"] - 1
df["ai_hard_to_decide_alone_neg"] = df["ai_hard_to_decide_alone"] - 1

df["ai_friends_relationship_change_negative_bin"] = present_1_2_3(df["ai_friends_relationship_change_negative"])
df["ai_feel_dependent_neg_bin"] = present_1_2_3(df["ai_feel_dependent_neg"])
df["ai_hard_to_decide_alone_neg_bin"] = present_1_2_3(df["ai_hard_to_decide_alone_neg"])

df["any_side_effect"] = any_binary_yes(df, [
    "ai_friends_relationship_change_negative_bin",
    "ai_feel_dependent_neg_bin",
    "ai_hard_to_decide_alone_neg_bin"
])

# Masks
m_med = df["med_use"].eq(1)
m_emo = df["emo_use"].eq(1)
m_ai = df["any_ai_use"].eq(1)
m_health = df["health_related_ai_use"].eq(1)

# =========================================================
# 4) FIXED DIGIHERO VALUES
# =========================================================
dh = {
    "Physical Health AI Use, n (%)":                  dict(stat="8727",         miss="",              yes=8727,  base=N_DH,  miss_n=np.nan),
    "Symptom Check":                                  dict(stat="6368 (73.0%)", miss="12 (0.1%)",    yes=6368,  base=8727,  miss_n=12),
    "Explain Medical Documents":                      dict(stat="5417 (62.1%)", miss="12 (0.1%)",    yes=5417,  base=8727,  miss_n=12),
    "Fitness, Nutrition":                             dict(stat="4922 (56.4%)", miss="14 (0.2%)",    yes=4922,  base=8727,  miss_n=14),

    "Mental Health AI Use, n (%)":                    dict(stat="5097",         miss="",              yes=5097,  base=N_DH,  miss_n=np.nan),
    "Anxiety, Worry":                                 dict(stat="2202 (43.2%)", miss="14 (0.3%)",    yes=2202,  base=5097,  miss_n=14),
    "Low Mood, Self-Doubt":                           dict(stat="1362 (26.7%)", miss="17 (0.3%)",    yes=1362,  base=5097,  miss_n=17),
    "Loneliness":                                     dict(stat="648 (12.7%)",  miss="16 (0.3%)",    yes=648,   base=5097,  miss_n=16),
    "Relationship Conflict":                          dict(stat="1581 (31.0%)", miss="15 (0.3%)",    yes=1581,  base=5097,  miss_n=15),
    "Self-Reflection":                                dict(stat="2842 (55.8%)", miss="17 (0.3%)",    yes=2842,  base=5097,  miss_n=17),
    "Grief, Loss, Major Life Event":                  dict(stat="1493 (29.3%)", miss="19 (0.4%)",    yes=1493,  base=5097,  miss_n=19),
    "Impulsivity, Substance Use":                     dict(stat="326 (6.4%)",   miss="19 (0.4%)",    yes=326,   base=5097,  miss_n=19),
    "Acute Crisis":                                   dict(stat="748 (14.7%)",  miss="22 (0.4%)",    yes=748,   base=5097,  miss_n=22),
    "Other":                                          dict(stat="2457 (48.2%)", miss="20 (0.4%)",    yes=2457,  base=5097,  miss_n=20),

    "Side Effects of AI users, n (%)":                dict(stat="15058",        miss="",              yes=15058, base=N_DH,  miss_n=np.nan),
    "Dependency":                                     dict(stat="1499 (10.0%)", miss="169 (1.1%)",   yes=1499,  base=15058, miss_n=169),
    "Relationship Deterioration":                     dict(stat="51 (0.3%)",    miss="470 (3.1%)",   yes=51,    base=15058, miss_n=470),
    "Reduced Decision Autonomy":                      dict(stat="696 (4.6%)",   miss="167 (1.1%)",   yes=696,   base=15058, miss_n=167),

    "Side Effects of Health-related AI users, n (%)": dict(stat="9507",         miss="",              yes=9507,  base=N_DH,  miss_n=np.nan),
    "Dependency_health":                              dict(stat="1216 (12.8%)", miss="120 (1.3%)",   yes=1216,  base=9507,  miss_n=120),
    "Relationship Deterioration_health":              dict(stat="41 (0.4%)",    miss="187 (2.0%)",   yes=41,    base=9507,  miss_n=187),
    "Reduced Decision Autonomy_health":               dict(stat="616 (6.5%)",   miss="113 (1.2%)",   yes=616,   base=9507,  miss_n=113),

    "Trust in AI of Health-related AI users, M (SD)": dict(stat="50.4 (23.8)",  miss="996 (10.5%)",  mean=50.4, sd=23.8, base=9507, miss_n=996),
}

# =========================================================
# 5) BUILD TABLE
# =========================================================
rows = []

def add_header(label):
    rows.append([label, "", "", "", "", "", ""])

def add_bin_row(label, s, base_n, dh_key=None, count_only=False):
    ps_stat, ps_miss, ps_yes, ps_miss_n = bin_stat(s, base_n, count_only=count_only)
    d = dh[dh_key or label]

    if pd.isna(d["miss_n"]):
        test, eff = "", ""
    else:
        test, eff = cramers_v_2x2(
            d["yes"], d["base"], d["miss_n"],
            ps_yes, base_n, ps_miss_n
        )

    rows.append([
        label,
        d["stat"],
        d["miss"],
        ps_stat,
        ps_miss,
        test,
        eff
    ])

def add_cont_row(label, s, base_n):
    ps_stat, ps_miss, x = cont_stat(s, base_n)
    d = dh[label]

    n1 = int(d["base"] - d["miss_n"])
    n2 = len(x)

    if n2 < 2:
        test, eff = "", ""
    else:
        t, p = ttest_ind_from_stats(
            mean1=d["mean"], std1=d["sd"], nobs1=n1,
            mean2=x.mean(), std2=x.std(ddof=1), nobs2=n2,
            equal_var=False
        )
        g = hedges_g(d["mean"], d["sd"], n1, x.mean(), x.std(ddof=1), n2)
        p_txt = "< .001" if p < .001 else f"= {p:.3f}"
        test = f"t = {t:.2f}, p {p_txt}"
        eff = f"g = {g:.3f}"

    rows.append([
        label,
        d["stat"],
        d["miss"],
        ps_stat,
        ps_miss,
        test,
        eff
    ])

# ---------------------------------------------------------
# Health-related AI usage
# ---------------------------------------------------------
add_header("Health-related AI Usage")

add_bin_row("Physical Health AI Use, n (%)", df["med_use"], N_PS, count_only=True)
add_bin_row("Symptom Check", selected_1_4(df.loc[m_med, "ai_health_symptom_check"]), int(m_med.sum()))
add_bin_row("Explain Medical Documents", selected_1_4(df.loc[m_med, "ai_health_explain_med_docs"]), int(m_med.sum()))
add_bin_row("Fitness, Nutrition", selected_1_4(df.loc[m_med, "ai_health_fitness_nutrition"]), int(m_med.sum()))

add_bin_row("Mental Health AI Use, n (%)", df["emo_use"], N_PS, count_only=True)
add_bin_row("Anxiety, Worry", selected_1_4(df.loc[m_emo, "ai_psych_sit_anxiety_worry"]), int(m_emo.sum()))
add_bin_row("Low Mood, Self-Doubt", selected_1_4(df.loc[m_emo, "ai_psych_sit_self_doubt_low_mood"]), int(m_emo.sum()))
add_bin_row("Loneliness", selected_1_4(df.loc[m_emo, "ai_psych_sit_loneliness"]), int(m_emo.sum()))
add_bin_row("Relationship Conflict", selected_1_4(df.loc[m_emo, "ai_psych_sit_relationship_conflict"]), int(m_emo.sum()))
add_bin_row("Self-Reflection", selected_1_4(df.loc[m_emo, "ai_psych_sit_self_reflection"]), int(m_emo.sum()))
add_bin_row("Grief, Loss, Major Life Event", selected_1_4(df.loc[m_emo, "ai_psych_sit_grief_loss"]), int(m_emo.sum()))
add_bin_row("Impulsivity, Substance Use", selected_1_4(df.loc[m_emo, "ai_psych_sit_impulsive_or_substance"]), int(m_emo.sum()))
add_bin_row("Acute Crisis", selected_1_4(df.loc[m_emo, "ai_psych_sit_acute_crisis"]), int(m_emo.sum()))
add_bin_row("Other", selected_1_4(df.loc[m_emo, "ai_psych_sit_other_selected"]), int(m_emo.sum()))

# ---------------------------------------------------------
# Side effects among AI users
# ---------------------------------------------------------
add_bin_row("Side Effects of AI users, n (%)", df["any_ai_use"], N_PS, count_only=True)
add_bin_row("Dependency", df.loc[m_ai, "ai_feel_dependent_neg_bin"], int(m_ai.sum()))
add_bin_row("Relationship Deterioration", df.loc[m_ai, "ai_friends_relationship_change_negative_bin"], int(m_ai.sum()))
add_bin_row("Reduced Decision Autonomy", df.loc[m_ai, "ai_hard_to_decide_alone_neg_bin"], int(m_ai.sum()))

# ---------------------------------------------------------
# Side effects among health-related AI users
# ---------------------------------------------------------
add_bin_row("Side Effects of Health-related AI users, n (%)", df["health_related_ai_use"], N_PS, count_only=True)
add_bin_row("Dependency", df.loc[m_health, "ai_feel_dependent_neg_bin"], int(m_health.sum()), dh_key="Dependency_health")
add_bin_row("Relationship Deterioration", df.loc[m_health, "ai_friends_relationship_change_negative_bin"], int(m_health.sum()), dh_key="Relationship Deterioration_health")
add_bin_row("Reduced Decision Autonomy", df.loc[m_health, "ai_hard_to_decide_alone_neg_bin"], int(m_health.sum()), dh_key="Reduced Decision Autonomy_health")

# ---------------------------------------------------------
# Trust in AI among health-related AI users
# ---------------------------------------------------------
add_cont_row("Trust in AI of Health-related AI users, M (SD)", df.loc[m_health, "ai_answer_quality"], int(m_health.sum()))

# =========================================================
# 6) FINAL TABLE
# =========================================================
table = pd.DataFrame(rows, columns=[
    "Variable",
    "DigiHero Statistic",
    "DigiHero Missing or NA n (%)",
    "PSYMPACT Statistic",
    "PSYMPACT Missing or NA n (%)",
    "Test statistic",
    "Effect size"
])

table
table.to_excel("ai_usage_sideeffects_table.xlsx", index=False)

In [9]:


# =========================================================
# 1) SETTINGS
# =========================================================
df_tab = df.drop_duplicates(subset="id").copy() if "id" in df.columns else df.copy()

N_DH = 28910
N_PS = df_tab["id"].nunique() if "id" in df_tab.columns else len(df_tab)

medical_cols = [
    "ai_health_symptom_check",
    "ai_health_explain_med_docs",
    "ai_health_fitness_nutrition",
]

emotional_cols = [
    "ai_psych_sit_anxiety_worry",
    "ai_psych_sit_self_doubt_low_mood",
    "ai_psych_sit_loneliness",
    "ai_psych_sit_relationship_conflict",
    "ai_psych_sit_self_reflection",
    "ai_psych_sit_grief_loss",
    "ai_psych_sit_impulsive_or_substance",
    "ai_psych_sit_acute_crisis",
    "ai_psych_sit_other_selected",
]

model_cols = [
    "ai_prog_chatgpt",
    "ai_prog_gemini",
    "ai_prog_claude",
    "ai_prog_grok",
    "ai_prog_perplexity",
    "ai_prog_llama",
    "ai_prog_deepseek",
    "ai_prog_mistral",
    "ai_prog_copilot",
    "ai_prog_other_selected",
]

general_nonuse_cols = [
    "ai_nouse_no_need",
    "ai_nouse_privacy_security",
    "ai_nouse_no_trust",
    "ai_nouse_no_knowhow",
    "ai_nouse_cost",
    "ai_nouse_ethics",
    "ai_nouse_other_selected",
]

health_nonuse_cols = [
    "ai_health_nouse_no_need",
    "ai_health_nouse_privacy",
    "ai_health_nouse_misinformation_fear",
    "ai_health_nouse_no_knowhow",
    "ai_health_nouse_liability_unclear",
    "ai_health_nouse_ethics",
    "ai_health_nouse_other_selected",
]

downstream_cols = [
    "ai_health_prepared_doctor_visit",
    "ai_health_avoided_doctor",
    "ai_health_doctor_visit_triggered",
    "ai_health_self_treatment",
    "ai_health_lifestyle_change",
    "ai_psych_found_comfort",
    "ai_psych_increased_worry",
]

for c in (
    medical_cols
    + emotional_cols
    + model_cols
    + general_nonuse_cols
    + health_nonuse_cols
    + downstream_cols
    + ["ai_use_any", "ai_use_health"]
):
    if c in df_tab.columns:
        df_tab[c] = pd.to_numeric(df_tab[c], errors="coerce")

# =========================================================
# 2) HELPERS
# =========================================================
def any_selected_1_4(df_sub, cols, filter_col="ai_use_health"):
    cols = [c for c in cols if c in df_sub.columns]
    if not cols:
        return pd.Series(np.nan, index=df_sub.index)

    tmp = df_sub[cols].apply(pd.to_numeric, errors="coerce")
    selected = tmp.ge(1) & tmp.le(4)

    out = pd.Series(np.where(selected.any(axis=1), 1, 0), index=df_sub.index, dtype="float")
    out[tmp.isna().all(axis=1)] = np.nan

    if filter_col in df_sub.columns:
        out.loc[pd.to_numeric(df_sub[filter_col], errors="coerce").eq(2)] = 0

    return out


def yes_no_1_2(s):
    s = pd.to_numeric(s, errors="coerce")
    out = pd.Series(np.nan, index=s.index, dtype="float")
    out[s.eq(1)] = 1
    out[s.eq(2)] = 0
    return out


def any_binary_yes(df_sub, cols):
    cols = [c for c in cols if c in df_sub.columns]
    if not cols:
        return pd.Series(np.nan, index=df_sub.index)

    tmp = df_sub[cols].apply(pd.to_numeric, errors="coerce")
    out = pd.Series(np.nan, index=tmp.index, dtype="float")
    out[tmp.eq(1).any(axis=1)] = 1
    out[tmp.isna().all(axis=1)] = np.nan
    out[(~tmp.eq(1).any(axis=1)) & (~tmp.isna().all(axis=1))] = 0
    return out


def selected_binary_01(s):
    s = pd.to_numeric(s, errors="coerce")
    out = pd.Series(np.nan, index=s.index, dtype="float")
    out[s.eq(1)] = 1
    out[s.eq(0)] = 0
    return out


def selected_1_4(s):
    s = pd.to_numeric(s, errors="coerce")
    out = pd.Series(np.nan, index=s.index, dtype="float")
    out[s.between(1, 4)] = 1
    out[s.eq(5)] = 0
    return out


def bin_stat(s, base_n, count_only=False):
    s = pd.to_numeric(s, errors="coerce")
    yes = int((s == 1).sum())
    miss = int(s.isna().sum())

    stat = f"{yes}" if count_only else (f"{yes} ({100 * yes / base_n:.1f}%)" if base_n > 0 else "—")
    miss_txt = f"{miss} ({100 * miss / base_n:.1f}%)" if base_n > 0 else "—"
    return stat, miss_txt, yes, miss


def all_missing_stat(df_sub, cols):
    cols = [c for c in cols if c in df_sub.columns]
    if not cols:
        return 0
    return int(df_sub[cols].isna().all(axis=1).sum())


def fmt_missing(n_miss, base_n):
    return f"{n_miss} ({100 * n_miss / base_n:.1f}%)" if base_n > 0 else "—"


def cramers_v_2x2(a_yes, a_base, a_miss, b_yes, b_base, b_miss):
    a_no = a_base - a_miss - a_yes
    b_no = b_base - b_miss - b_yes

    tab = np.array([[a_yes, a_no], [b_yes, b_no]])

    if (tab < 0).any() or (tab.sum(axis=0) == 0).any() or (tab.sum(axis=1) == 0).any():
        return "", ""

    chi2, p, _, _ = chi2_contingency(tab, correction=False)
    v = np.sqrt(chi2 / tab.sum())

    p_txt = "< .001" if p < .001 else f"= {p:.3f}"
    return f"χ²(1) = {chi2:.2f}, p {p_txt}", f"V = {v:.3f}"


def add_header(rows, label):
    rows.append([label, "", "", "", "", "", ""])


def add_bin_row(rows, fixed, label, s, base_n, fixed_key=None, count_only=False, allow_test=True):
    ps_stat, ps_miss, ps_yes, ps_miss_n = bin_stat(s, base_n, count_only=count_only)
    d = fixed[fixed_key or label]

    if allow_test and d["miss_n"] is not None:
        test, eff = cramers_v_2x2(d["yes"], d["base"], d["miss_n"], ps_yes, base_n, ps_miss_n)
    else:
        test, eff = "", ""

    rows.append([
        label,
        d["stat"],
        d["miss"],
        ps_stat,
        ps_miss,
        test,
        eff
    ])


def add_group_count_row(rows, label, dh_stat, dh_miss, ps_yes, ps_base, ps_miss=None):
    ps_stat = f"{ps_yes}"
    ps_miss_txt = "—" if ps_miss is None else fmt_missing(ps_miss, ps_base)

    rows.append([
        label,
        dh_stat,
        dh_miss,
        ps_stat,
        ps_miss_txt,
        "",
        ""
    ])


def add_fixed_only_row(rows, fixed, label, fixed_key=None):
    d = fixed[fixed_key or label]
    rows.append([
        label,
        d["stat"],
        d["miss"],
        "—",
        "—",
        "",
        ""
    ])

# =========================================================
# 3) DERIVE PSYMPACT VARIABLES
# =========================================================
df_tab["med_use"] = any_selected_1_4(df_tab, medical_cols)
df_tab["emo_use"] = any_selected_1_4(df_tab, emotional_cols)
df_tab["any_ai_use"] = yes_no_1_2(df_tab["ai_use_any"])
df_tab["health_related_ai_use"] = any_binary_yes(df_tab, ["med_use", "emo_use"])

m_general = df_tab["any_ai_use"].eq(1)
m_general_non = df_tab["any_ai_use"].eq(0)

# Table S2: within general AI users
m_health = m_general & df_tab["health_related_ai_use"].eq(1)
m_health_non = m_general & df_tab["health_related_ai_use"].eq(0)

n_general = int(m_general.sum())
n_general_non = int(m_general_non.sum())
n_health = int(m_health.sum())
n_health_non = int(m_health_non.sum())

ps_general_nonuse_missing = all_missing_stat(df_tab.loc[m_general_non], general_nonuse_cols)
ps_health_nonuse_missing = all_missing_stat(df_tab.loc[m_health_non], health_nonuse_cols)

# =========================================================
# 4) FIXED DIGIHERO VALUES
# =========================================================
dh_s1 = {
    "General AI Usage, n (%)":      dict(stat="15058",         miss="",              yes=15058, base=N_DH,   miss_n=0),

    "ChatGPT":                      dict(stat="11077 (73.6%)", miss="",              yes=11077, base=15058,  miss_n=0),
    "Gemini":                       dict(stat="6346 (42.1%)",  miss="",              yes=6346,  base=15058,  miss_n=0),
    "Claude":                       dict(stat="517 (3.4%)",    miss="",              yes=517,   base=15058,  miss_n=0),
    "Grok":                         dict(stat="257 (1.7%)",    miss="",              yes=257,   base=15058,  miss_n=0),
    "Perplexity":                   dict(stat="1474 (9.8%)",   miss="",              yes=1474,  base=15058,  miss_n=0),
    "Llama":                        dict(stat="199 (1.3%)",    miss="",              yes=199,   base=15058,  miss_n=0),
    "DeepSeek":                     dict(stat="298 (2.0%)",    miss="",              yes=298,   base=15058,  miss_n=0),
    "Mistral":                      dict(stat="334 (2.2%)",    miss="",              yes=334,   base=15058,  miss_n=0),
    "Copilot":                      dict(stat="3315 (22.0%)",  miss="",              yes=3315,  base=15058,  miss_n=0),
    "Other model":                  dict(stat="1616 (10.7%)",  miss="",              yes=1616,  base=15058,  miss_n=0),

    # 23 appears to be the all-missing count for the general non-use reason block
    "Reasons for Non-Usage, n (%)": dict(stat="13852",         miss="23 (0.2%)",     yes=13852, base=N_DH,   miss_n=None),
    "No benefit":                   dict(stat="8953 (64.7%)",  miss="23 (0.2%)",     yes=8953,  base=13852,  miss_n=23),
    "Privacy":                      dict(stat="6885 (50.2%)",  miss="23 (0.2%)",     yes=6885,  base=13852,  miss_n=23),
    "Mistrust":                     dict(stat="5516 (39.9%)",  miss="23 (0.2%)",     yes=5516,  base=13852,  miss_n=23),
    "Tech Literacy":                dict(stat="2538 (18.4%)",  miss="23 (0.2%)",     yes=2538,  base=13852,  miss_n=23),
    "Responsibility":               dict(stat="2289 (16.6%)",  miss="23 (0.2%)",     yes=2289,  base=13852,  miss_n=23),
    "Ethics":                       dict(stat="2215 (16.0%)",  miss="23 (0.2%)",     yes=2215,  base=13852,  miss_n=23),
    "Other":                        dict(stat="977 (7.1%)",    miss="23 (0.2%)",     yes=977,   base=13852,  miss_n=23),
}

dh_s2 = {
    # within general AI users
    "Downstream consequences, n (%)":          dict(stat="9507",         miss="",                yes=9507, base=15058, miss_n=None),
    "Prepared for Medical Appointment":        dict(stat="3870 (41.4%)", miss="160 (1.7%)",     yes=3870, base=9507,  miss_n=160),
    "Avoided Medical Appointment":             dict(stat="1022 (10.9%)", miss="159 (1.7%)",     yes=1022, base=9507,  miss_n=159),
    "Induced Medical Appointment":             dict(stat="1541 (16.5%)", miss="147 (1.5%)",     yes=1541, base=9507,  miss_n=147),
    "Self-Treatment":                          dict(stat="2702 (28.9%)", miss="161 (1.7%)",     yes=2702, base=9507,  miss_n=161),
    "Induces Lifestyle Change":                dict(stat="2078 (22.2%)", miss="150 (1.6%)",     yes=2078, base=9507,  miss_n=150),
    "Emotional Reassurance":                   dict(stat="2036 (21.9%)", miss="192 (2.0%)",     yes=2036, base=9507,  miss_n=192),
    "Increased Worry":                         dict(stat="375 (4.0%)",   miss="197 (2.1%)",     yes=375,  base=9507,  miss_n=197),

    # 2374 appears to be the all-missing count for the health non-use reason block
    "Reasons for Non-Usage for Health, n (%)": dict(stat="5551",         miss="2374 (42.8%)",   yes=5551, base=15058, miss_n=None),
    "No benefit_health":                       dict(stat="1791 (56.4%)", miss="2374 (42.8%)",   yes=1791, base=5551,  miss_n=2374),
    "Privacy_health":                          dict(stat="1787 (56.2%)", miss="2374 (42.8%)",   yes=1787, base=5551,  miss_n=2374),
    "Mistrust_health":                         dict(stat="1803 (56.8%)", miss="2374 (42.8%)",   yes=1803, base=5551,  miss_n=2374),
    "Tech Literacy_health":                    dict(stat="390 (12.3%)",  miss="2374 (42.8%)",   yes=390,  base=5551,  miss_n=2374),
    "Responsibility_health":                   dict(stat="342 (10.8%)",  miss="2374 (42.8%)",   yes=342,  base=5551,  miss_n=2374),
    "Ethics_health":                           dict(stat="962 (30.3%)",  miss="2374 (42.8%)",   yes=962,  base=5551,  miss_n=2374),
    "Other_health":                            dict(stat="245 (7.7%)",   miss="2374 (42.8%)",   yes=245,  base=5551,  miss_n=2374),
}

# =========================================================
# 5) TABLE S1
# =========================================================
rows_s1 = []

add_header(rows_s1, "General AI Usage")

add_bin_row(
    rows_s1, dh_s1, "General AI Usage, n (%)",
    df_tab["any_ai_use"], N_PS,
    count_only=True
)

add_header(rows_s1, "AI Tools Used, n (%)")

add_bin_row(rows_s1, dh_s1, "ChatGPT",     selected_binary_01(df_tab.loc[m_general, "ai_prog_chatgpt"]),        n_general)
add_bin_row(rows_s1, dh_s1, "Gemini",      selected_binary_01(df_tab.loc[m_general, "ai_prog_gemini"]),         n_general)
add_bin_row(rows_s1, dh_s1, "Claude",      selected_binary_01(df_tab.loc[m_general, "ai_prog_claude"]),         n_general)
add_bin_row(rows_s1, dh_s1, "Grok",        selected_binary_01(df_tab.loc[m_general, "ai_prog_grok"]),           n_general)
add_bin_row(rows_s1, dh_s1, "Perplexity",  selected_binary_01(df_tab.loc[m_general, "ai_prog_perplexity"]),     n_general)
add_bin_row(rows_s1, dh_s1, "Llama",       selected_binary_01(df_tab.loc[m_general, "ai_prog_llama"]),          n_general)
add_bin_row(rows_s1, dh_s1, "DeepSeek",    selected_binary_01(df_tab.loc[m_general, "ai_prog_deepseek"]),       n_general)
add_bin_row(rows_s1, dh_s1, "Mistral",     selected_binary_01(df_tab.loc[m_general, "ai_prog_mistral"]),        n_general)
add_bin_row(rows_s1, dh_s1, "Copilot",     selected_binary_01(df_tab.loc[m_general, "ai_prog_copilot"]),        n_general)
add_bin_row(rows_s1, dh_s1, "Other model", selected_binary_01(df_tab.loc[m_general, "ai_prog_other_selected"]), n_general)

add_group_count_row(
    rows_s1,
    "Reasons for Non-Usage, n (%)",
    dh_stat=dh_s1["Reasons for Non-Usage, n (%)"]["stat"],
    dh_miss=dh_s1["Reasons for Non-Usage, n (%)"]["miss"],
    ps_yes=n_general_non,
    ps_base=N_PS,
    ps_miss=ps_general_nonuse_missing
)

add_bin_row(rows_s1, dh_s1, "No benefit",     selected_binary_01(df_tab.loc[m_general_non, "ai_nouse_no_need"]),            n_general_non)
add_bin_row(rows_s1, dh_s1, "Privacy",        selected_binary_01(df_tab.loc[m_general_non, "ai_nouse_privacy_security"]),   n_general_non)
add_bin_row(rows_s1, dh_s1, "Mistrust",       selected_binary_01(df_tab.loc[m_general_non, "ai_nouse_no_trust"]),           n_general_non)
add_bin_row(rows_s1, dh_s1, "Tech Literacy",  selected_binary_01(df_tab.loc[m_general_non, "ai_nouse_no_knowhow"]),         n_general_non)

# No true PSYMPACT analogue for DigiHero's general-AI "Responsibility" row:
add_fixed_only_row(rows_s1, dh_s1, "Responsibility")

add_bin_row(rows_s1, dh_s1, "Ethics",         selected_binary_01(df_tab.loc[m_general_non, "ai_nouse_ethics"]),             n_general_non)
add_bin_row(rows_s1, dh_s1, "Other",          selected_binary_01(df_tab.loc[m_general_non, "ai_nouse_other_selected"]),     n_general_non)

table_s1 = pd.DataFrame(rows_s1, columns=[
    "Variable",
    f"DigiHero Statistic (N={N_DH})",
    "DigiHero Missing or NA n (%)",
    f"PSYMPACT Statistic (N={N_PS})",
    "PSYMPACT Missing or NA n (%)",
    "Test statistic",
    "Effect size"
])

# =========================================================
# 6) TABLE S2
# =========================================================
rows_s2 = []

add_header(rows_s2, "Health-Related AI Usage")

add_group_count_row(
    rows_s2,
    "Downstream consequences, n (%)",
    dh_stat=dh_s2["Downstream consequences, n (%)"]["stat"],
    dh_miss=dh_s2["Downstream consequences, n (%)"]["miss"],
    ps_yes=n_health,
    ps_base=n_general,
    ps_miss=None
)

add_bin_row(rows_s2, dh_s2, "Prepared for Medical Appointment", selected_1_4(df_tab.loc[m_health, "ai_health_prepared_doctor_visit"]),  n_health)
add_bin_row(rows_s2, dh_s2, "Avoided Medical Appointment",      selected_1_4(df_tab.loc[m_health, "ai_health_avoided_doctor"]),         n_health)
add_bin_row(rows_s2, dh_s2, "Induced Medical Appointment",      selected_1_4(df_tab.loc[m_health, "ai_health_doctor_visit_triggered"]), n_health)
add_bin_row(rows_s2, dh_s2, "Self-Treatment",                   selected_1_4(df_tab.loc[m_health, "ai_health_self_treatment"]),         n_health)
add_bin_row(rows_s2, dh_s2, "Induces Lifestyle Change",         selected_1_4(df_tab.loc[m_health, "ai_health_lifestyle_change"]),       n_health)
add_bin_row(rows_s2, dh_s2, "Emotional Reassurance",            selected_1_4(df_tab.loc[m_health, "ai_psych_found_comfort"]),           n_health)
add_bin_row(rows_s2, dh_s2, "Increased Worry",                  selected_1_4(df_tab.loc[m_health, "ai_psych_increased_worry"]),        n_health)

add_group_count_row(
    rows_s2,
    "Reasons for Non-Usage for Health, n (%)",
    dh_stat=dh_s2["Reasons for Non-Usage for Health, n (%)"]["stat"],
    dh_miss=dh_s2["Reasons for Non-Usage for Health, n (%)"]["miss"],
    ps_yes=n_health_non,
    ps_base=n_general,
    ps_miss=ps_health_nonuse_missing
)

add_bin_row(rows_s2, dh_s2, "No benefit",      selected_binary_01(df_tab.loc[m_health_non, "ai_health_nouse_no_need"]),              n_health_non, fixed_key="No benefit_health")
add_bin_row(rows_s2, dh_s2, "Privacy",         selected_binary_01(df_tab.loc[m_health_non, "ai_health_nouse_privacy"]),              n_health_non, fixed_key="Privacy_health")
add_bin_row(rows_s2, dh_s2, "Mistrust",        selected_binary_01(df_tab.loc[m_health_non, "ai_health_nouse_misinformation_fear"]),  n_health_non, fixed_key="Mistrust_health")
add_bin_row(rows_s2, dh_s2, "Tech Literacy",   selected_binary_01(df_tab.loc[m_health_non, "ai_health_nouse_no_knowhow"]),           n_health_non, fixed_key="Tech Literacy_health")
add_bin_row(rows_s2, dh_s2, "Responsibility",  selected_binary_01(df_tab.loc[m_health_non, "ai_health_nouse_liability_unclear"]),    n_health_non, fixed_key="Responsibility_health")
add_bin_row(rows_s2, dh_s2, "Ethics",          selected_binary_01(df_tab.loc[m_health_non, "ai_health_nouse_ethics"]),               n_health_non, fixed_key="Ethics_health")
add_bin_row(rows_s2, dh_s2, "Other",           selected_binary_01(df_tab.loc[m_health_non, "ai_health_nouse_other_selected"]),       n_health_non, fixed_key="Other_health")

table_s2 = pd.DataFrame(rows_s2, columns=[
    "Variable",
    "DigiHero Statistic",
    "DigiHero Missing or NA n (%)",
    "PSYMPACT Statistic",
    "PSYMPACT Missing or NA n (%)",
    "Test statistic",
    "Effect size"
])

# =========================================================
# 7) SAVE
# =========================================================
with pd.ExcelWriter("tables_S1_S2_ai_usage_corrected.xlsx", engine="openpyxl") as writer:
    table_s1.to_excel(writer, sheet_name="Table_S1", index=False)
    table_s2.to_excel(writer, sheet_name="Table_S2", index=False)

table_s1, table_s2

(                        Variable DigiHero Statistic (N=28910)  \
 0               General AI Usage                                
 1        General AI Usage, n (%)                        15058   
 2           AI Tools Used, n (%)                                
 3                        ChatGPT                11077 (73.6%)   
 4                         Gemini                 6346 (42.1%)   
 5                         Claude                   517 (3.4%)   
 6                           Grok                   257 (1.7%)   
 7                     Perplexity                  1474 (9.8%)   
 8                          Llama                   199 (1.3%)   
 9                       DeepSeek                   298 (2.0%)   
 10                       Mistral                   334 (2.2%)   
 11                       Copilot                 3315 (22.0%)   
 12                   Other model                 1616 (10.7%)   
 13  Reasons for Non-Usage, n (%)                        13852   
 14       

### Adjust/Map Sociodemographics

#### 1. Age and Gender

In [10]:
# ============================================================
# Harmonized columns (PSYMPACT -> DigiHero-combinable)
# ============================================================

# --- gender (4cat) ---
# PSYMPACT v_11: 1 weiblich, 2 männlich, 3 non-binär, 4 keine Angabe :contentReference[oaicite:28]{index=28}
gender_map = {1: "female", 2: "male", 3: "other", 4: "not_specified"}
df["h_gender_4cat"] = df["gender"].map(gender_map)

# --- age ---
df["h_age_years"] = pd.to_numeric(df["age"], errors="coerce")

#### 2. Employment

In [11]:
df = df.copy()
# --- employment: keep dummies + derive binary employment status ---
emp_cols = [
    "employ_student",
    "employ_parttime",
    "employ_fulltime",
    "employ_carework",
    "employ_unemployed",
    "employ_retired",
    "employ_work_disabled",
]

for c in emp_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# harmonized copies (optional)
df["h_emp_student"]    = df.get("employ_student")
df["h_emp_parttime"]   = df.get("employ_parttime")
df["h_emp_fulltime"]   = df.get("employ_fulltime")
df["h_emp_care"]       = df.get("employ_carework")
df["h_emp_unemployed"] = df.get("employ_unemployed")
df["h_emp_retired"]    = df.get("employ_retired")
df["h_emp_disabled"]   = df.get("employ_work_disabled")

# masks for classification
employed_mask = (
    (df["h_emp_fulltime"] == 1) |
    (df["h_emp_parttime"] == 1) 
)

not_employed_mask = (
    (df["h_emp_student"] == 1) |
    (df["h_emp_care"] == 1) |
    (df["h_emp_unemployed"] == 1) |
    (df["h_emp_retired"] == 1) |
    (df["h_emp_disabled"] == 1)
)

# NEW: define "has any observed employment info" (at least one non-NaN dummy)
has_any_emp_info = df[["h_emp_student","h_emp_parttime","h_emp_fulltime",
                       "h_emp_care","h_emp_unemployed","h_emp_retired","h_emp_disabled"]].notna().any(axis=1)

# start as NaN, only fill where we have any info
df["h_employed"] = np.nan
df.loc[has_any_emp_info & employed_mask, "h_employed"] = 1
df.loc[has_any_emp_info & (~employed_mask) & not_employed_mask, "h_employed"] = 0

# keep as float with NaN
df["h_employed"] = df["h_employed"].astype(float)

In [12]:
df.groupby("h_employed")["id"].nunique()

h_employed
0.0     832
1.0    1481
Name: id, dtype: int64

#### 3. Mental Health Diagnosis 

In [13]:
df = df.copy()
# --------------------------------------------------
# --- mental diagnosis ever (binary) ---
# NEW naming: v_1315 -> "dx_psych"
# (0 = not selected, 1 = selected; keep as 0/1)
# --------------------------------------------------
df["dx_psych"] = pd.to_numeric(df.get("dx_psych"), errors="coerce")
df["h_dx_mental_ever"] = df["dx_psych"]

# --------------------------------------------------
# --- PHQ-4 bridge from PHQ-9 + GAD-7 ---
# NEW naming: PHQ items -> phq_interest, phq_depressed, ...
#             GAD items -> gad_nervous, gad_control_worry, ...
# --------------------------------------------------
df["h_phq4_i1"] = df["phq_interest"] - 1          # interest
df["h_phq4_i2"] = df["phq_depressed"] - 1         # depressed mood
df["h_phq4_i3"] = df["gad_nervous"] - 1           # nervous
df["h_phq4_i4"] = df["gad_control_worry"] - 1     # uncontrollable worry

phq4_items = ["h_phq4_i1", "h_phq4_i2", "h_phq4_i3", "h_phq4_i4"]
df["h_phq4_total"] = df[phq4_items].sum(axis=1, min_count=1)
df["h_phq4_depr"]  = df[["h_phq4_i1", "h_phq4_i2"]].sum(axis=1, min_count=1)
df["h_phq4_anx"]   = df[["h_phq4_i3", "h_phq4_i4"]].sum(axis=1, min_count=1)



#### 4. Somatic Diagnosis

In [14]:
somatic_dx_cols = [
    "dx_hypertension", "dx_allergy", "dx_chronic_back_pain", "dx_sleep_disorder",
    "dx_joint_disease_arthrosis_rheuma", "dx_migraine", "dx_heart_disease",
    "dx_chronic_bronchitis", "dx_diabetes", "dx_osteoporosis",
    "dx_liver_disease_fattyliver_hepatitis_cirrhosis", "dx_asthma", "dx_stroke",
    "dx_cancer"
]

# ensure numeric (0/1/NaN)
for c in somatic_dx_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

cols_present = [c for c in somatic_dx_cols if c in df.columns]

# all NaN -> NaN; else 1 if any ==1; else 0 (includes all 0)
all_nan = df[cols_present].isna().all(axis=1)

any_selected = (df[cols_present] == 1).any(axis=1)

df["h_dx_somatic_any"] = np.where(
    all_nan,
    np.nan,
    np.where(any_selected, 1, 0)
).astype(float)

In [15]:
df.groupby("h_dx_somatic_any")["id"].nunique()

h_dx_somatic_any
0.0     688
1.0    1543
Name: id, dtype: int64

In [16]:
df.h_dx_somatic_any.isna().sum()

82

#### 5. Education

In [17]:
import numpy as np
import pandas as pd

# ensure numeric
df["education"] = pd.to_numeric(df["education"], errors="coerce")
df["vocational_degree"] = pd.to_numeric(df["vocational_degree"], errors="coerce")

# --- choose how to treat PSYMPACT education==6 (Abitur OR FHR) ---
ASSUME_ABITUR_FOR_PSYMPACT = True  # False -> 12 years; True -> 13 years

# -----------------------------
# 1) school_eq from PSYMPACT education
# -----------------------------
df["school_eq"] = np.nan
df.loc[df["education"].isin([2, 3]), "school_eq"] = 0          # no certificate / <=7 years
df.loc[df["education"] == 4, "school_eq"] = 1                  # Haupt / POS 8-9
df.loc[df["education"] == 5, "school_eq"] = 2                  # Realschule / POS 10
df.loc[df["education"] == 6, "school_eq"] = 4 if ASSUME_ABITUR_FOR_PSYMPACT else 3

# education == 1 ("still in school") -> NaN
# education == 7 ("other") -> NaN

# -----------------------------
# 2) occupa_eq from PSYMPACT vocational_degree
# -----------------------------
df["occupa_eq"] = np.nan
df.loc[df["vocational_degree"] == 3, "occupa_eq"] = 0          # apprenticeship
df.loc[df["vocational_degree"] == 4, "occupa_eq"] = 2          # school vocational
df.loc[df["vocational_degree"] == 5, "occupa_eq"] = 3          # Meister/Fachschule/Techniker
df.loc[df["vocational_degree"] == 6, "occupa_eq"] = 4          # university degree
df.loc[df["vocational_degree"].isin([7, 8]), "occupa_eq"] = 5  # phd / habil etc.

# no vocational degree and not in training -> +0 years
df.loc[df["vocational_degree"] == 2, "occupa_eq"] = -1

# vocational_degree == 1 ("in training") -> NaN
# vocational_degree == 9 ("other") -> NaN

# -----------------------------
# 3) xBILZEIT = schooling years + occupational add
# -----------------------------
df["xBILZEIT"] = np.nan
df.loc[df["school_eq"] == 0, "xBILZEIT"] = 7
df.loc[df["school_eq"] == 1, "xBILZEIT"] = 9
df.loc[df["school_eq"] == 2, "xBILZEIT"] = 10
df.loc[df["school_eq"] == 3, "xBILZEIT"] = 12
df.loc[df["school_eq"] == 4, "xBILZEIT"] = 13

occ_add = pd.Series(0.0, index=df.index)
occ_add.loc[df["occupa_eq"] == 0] = 1.5
occ_add.loc[df["occupa_eq"] == 1] = 1.5
occ_add.loc[df["occupa_eq"] == 2] = 2.0
occ_add.loc[df["occupa_eq"] == 3] = 3.0
occ_add.loc[df["occupa_eq"] == 4] = 5.0
occ_add.loc[df["occupa_eq"] == 5] = 8.0
occ_add.loc[df["occupa_eq"] == -1] = 0.0

mask_add = df["xBILZEIT"].notna() & (df["xBILZEIT"] >= 7)
df.loc[mask_add, "xBILZEIT"] = df.loc[mask_add, "xBILZEIT"] + occ_add.loc[mask_add]

# -----------------------------
# 4) edu5 categorical variable aligned with your later regression coding
# -----------------------------
def edu5_from_psympact(education, vocational_degree):
    # doctorate
    if vocational_degree in [7, 8]:
        return "doctorate"
    
    # advanced tertiary / higher vocational
    if vocational_degree in [5, 6]:
        return "tertiary_adv"
    
    # still in training -> classify from school level only
    if vocational_degree == 1:
        if education in [2, 3]:
            return "low"
        elif education in [4, 5]:
            return "medium"
        elif education == 6:
            return "high"
        else:
            return "unknown"
    
    # completed basic vocational degree
    # keep this aligned with your earlier edu5 logic:
    # low school + basic vocational -> medium
    if vocational_degree in [3, 4]:
        if education in [2, 3]:
            return "medium"
        elif education in [4, 5]:
            return "medium"
        elif education == 6:
            return "high"
        else:
            return "medium"   # vocational only, school unknown
    
    # no vocational degree
    if vocational_degree == 2:
        if education in [2, 3]:
            return "low"
        elif education in [4, 5]:
            return "medium"
        elif education == 6:
            return "high"
        else:
            return "unknown"
    
    # fallback if vocational is missing/other
    if education in [2, 3]:
        return "low"
    elif education in [4, 5]:
        return "medium"
    elif education == 6:
        return "high"
    else:
        return "unknown"

edu5_order = ["low", "medium", "high", "tertiary_adv", "doctorate", "unknown"]

df["edu5"] = [
    edu5_from_psympact(e, v)
    for e, v in zip(df["education"], df["vocational_degree"])
]

df["edu5"] = pd.Categorical(df["edu5"], categories=edu5_order, ordered=True)

In [18]:
df["edu5"]

0               high
1               high
2             medium
3       tertiary_adv
4               high
            ...     
2308    tertiary_adv
2309    tertiary_adv
2310    tertiary_adv
2311          medium
2312    tertiary_adv
Name: edu5, Length: 2313, dtype: category
Categories (6, object): ['low' < 'medium' < 'high' < 'tertiary_adv' < 'doctorate' < 'unknown']

In [19]:
# Schulabschluss: schülerin aktuell nan, weiß nicht nan, keine angabe nan, anderer schulabschluss=10
# Occupational: anderen berufl abschluss nan, weiß nicht nan, keine angabe nan 
# Unterscheidung zw. Bachelor Master/ Promotion 

#### 6. Mental Health Treatment

In [20]:
df["h_psycho_treatment"] = np.where(
    (df["mh_psychotherapy_6m"] == 1) | (df["mh_psychiatry_med_6m"] == 1),
    1,
    np.where(
        df["mh_psychotherapy_6m"].isna() & df["mh_psychiatry_med_6m"].isna(),
        np.nan,
        0))

In [21]:
df.groupby("h_psycho_treatment")["id"].nunique()

h_psycho_treatment
0.0     923
1.0    1341
Name: id, dtype: int64

#### 7. General Functioning

In [22]:
df.groupby("eq5d_usual_activities")["id"].nunique()

eq5d_usual_activities
1.0    693
2.0    583
3.0    397
4.0    275
5.0     31
Name: id, dtype: int64

#### 8. HITOP

In [23]:
df.EXTANT.describe()

count    2313.000000
mean        1.555772
std         0.478425
min         1.000000
25%         1.166667
50%         1.500000
75%         1.833333
max         4.000000
Name: EXTANT, dtype: float64

## Regression Analysis

In [24]:
# -----------------------------
# Variable names (YOUR naming)
# -----------------------------
ki_col = "ai_use_any"       # 1 = yes, 2 = no
gender_col = "gender"       # 1=female, 2=male, 3=other, 4=not specified
age_col = "h_age_years"     # already age in years
employed_col = "h_employed" # 1 = employed, 0 = not employed
edu_col = "xBILZEIT"        # education years; -2 = still in school

df = df.copy()

# ensure numeric where needed
for col in [ki_col, gender_col, age_col, employed_col, edu_col]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# -----------------------------
# 1) Exclusions
# -----------------------------
n_total = len(df)

miss_gender = df[gender_col].isna()
miss_age = df[age_col].isna()

# valid AI answer = 1/2 only
valid_ai = df[ki_col].isin([1, 2])
other_ki = df[ki_col].notna() & (~valid_ai)

n_miss_gender = int(miss_gender.sum())
n_miss_age = int(miss_age.sum())
n_other_ki = int(other_ki.sum())

mask_keep = (~miss_gender) & (~miss_age) & valid_ai
n_excluded = int(n_total - mask_keep.sum())

print("Exclusions (reasons can overlap):")
print(f"  Missing gender:               {n_miss_gender:>6}")
print(f"  Missing age:                  {n_miss_age:>6}")
print(f"  Non-1/2 on AI question:       {n_other_ki:>6}")
print(f"  Total excluded:               {n_excluded:>6}")
print(f"  Analysis sample (kept):       {int(mask_keep.sum()):>6}")


Exclusions (reasons can overlap):
  Missing gender:                   48
  Missing age:                      48
  Non-1/2 on AI question:            0
  Total excluded:                   48
  Analysis sample (kept):         2265


In [25]:


medical_cols = [
    "ai_health_symptom_check",
    "ai_health_explain_med_docs",
    "ai_health_fitness_nutrition",
]
emotional_cols = [    "ai_psych_sit_anxiety_worry",
    "ai_psych_sit_self_doubt_low_mood",
    "ai_psych_sit_loneliness",
    "ai_psych_sit_relationship_conflict",
    "ai_psych_sit_self_reflection",
    "ai_psych_sit_grief_loss",
    "ai_psych_sit_impulsive_or_substance",
    "ai_psych_sit_acute_crisis",
   "ai_psych_sit_other_selected",] 

df["med_use"] = any_selected_1_4(df, medical_cols)
df["emo_use"] = any_selected_1_4(df, emotional_cols)


In [26]:
# --------------------------------------------------
# 2) Side-effect coding (numeric only)
# --------------------------------------------------
side_effect_cols = {
    "ai_feel_dependent": "AI dependency",
    "ai_hard_to_decide_alone": "Decision difficulty",
    "ai_friends_relationship_change": "Social impact",
}

SOCIAL_YES_NUM = {1, 2, 3}

def to_yes_no(v, col):
    if pd.isna(v):
        return np.nan

    num = pd.to_numeric(v, errors="coerce")
    if pd.isna(num):
        return np.nan

    # -----------------------------
    # SPECIAL CASE: social variable
    # -----------------------------
    if col == "ai_friends_relationship_change":
        return 1 if int(num) in SOCIAL_YES_NUM else 0

    # -----------------------------
    # GENERAL CASE: agreement = 2–4
    # -----------------------------
    return 1 if 2 <= num <= 4 else 0


# apply coding
for col in side_effect_cols:
    ycol = f"{col}_yes"
    if col in df.columns:
        df[ycol] = df[col].apply(lambda x: to_yes_no(x, col))
    else:
        df[ycol] = np.nan

# -----------------------------
# Counts per side-effect item
# -----------------------------
print("Per-side-effect Yes/No/Missing counts:")

for col, label in side_effect_cols.items():
    ycol = f"{col}_yes"

    n_yes = int((df[ycol] == 1).sum())
    n_no = int((df[ycol] == 0).sum())
    n_miss = int(df[ycol].isna().sum())

    n_ans = n_yes + n_no
    pct_yes = (100 * n_yes / n_ans) if n_ans > 0 else np.nan

    if n_ans > 0:
        print(f"  {label:<35} Yes={n_yes:>6}  No={n_no:>6}  Missing={n_miss:>6}  Yes%={pct_yes:.1f}")
    else:
        print(f"  {label:<35} Yes={n_yes:>6}  No={n_no:>6}  Missing={n_miss:>6}  Yes%=n/a")


# --------------------------------------------------
# 3) Person-level outcome: ANY side effect
# --------------------------------------------------
ycols = [f"{c}_yes" for c in side_effect_cols.keys()]

answered_any = df[ycols].notna().any(axis=1)
any_yes = df[ycols].eq(1).any(axis=1)

df["side_effect_any"] = np.where(
    ~answered_any,
    np.nan,
    np.where(any_yes, 1, 0)
)

Per-side-effect Yes/No/Missing counts:
  AI dependency                       Yes=   496  No=  1354  Missing=   463  Yes%=26.8
  Decision difficulty                 Yes=   308  No=  1542  Missing=   463  Yes%=16.6
  Social impact                       Yes=    35  No=  1815  Missing=   463  Yes%=1.9


#### Calculate Side Effects for Table (Recoding to Map INEP)

In [28]:
df_emo_use = df[df["emo_use"] ==1]

In [29]:

df_emo_use["ai_friends_relationship_change_negative"] = df_emo_use["ai_friends_relationship_change"].map({1: 3,2: 2,3: 1,4: 0,5: 0, 6: 0,7: 0})

df_emo_use["ai_feel_dependent"] = df_emo_use["ai_feel_dependent"] - 1
df_emo_use["ai_hard_to_decide_alone"] = df_emo_use["ai_hard_to_decide_alone"] - 1

In [34]:
side_effect_cols = [
    "ai_friends_relationship_change_negative",
    "ai_feel_dependent",
    "ai_hard_to_decide_alone"
]

mask_neg = df_emo_use["ai_friends_relationship_change_negative"].gt(0)

n_neg = mask_neg.sum()
pct_neg = mask_neg.mean() * 100

print(f"n = {n_neg}")
print(f"percentage = {pct_neg:.2f}%")

n = 27
percentage = 2.29%


In [35]:
mask_neg = df_emo_use["ai_feel_dependent"].gt(0)

n_neg = mask_neg.sum()
pct_neg = mask_neg.mean() * 100

print(f"n = {n_neg}")
print(f"percentage = {pct_neg:.2f}%")

n = 386
percentage = 32.71%


In [36]:
mask_neg = df_emo_use["ai_hard_to_decide_alone"].gt(0)

n_neg = mask_neg.sum()
pct_neg = mask_neg.mean() * 100

print(f"n = {n_neg}")
print(f"percentage = {pct_neg:.2f}%")

n = 247
percentage = 20.93%


In [37]:
df_emo_use.id.nunique()

1180

## Regression Analysis

In [ ]:
# -----------------------------
# 2) Keep only cases with non  missing age and gender
# -----------------------------

df_reg = df.loc[mask_keep].copy()

# -----------------------------
# 2) Build regression variables
# -----------------------------
# Outcome: AI use (1 = yes, 0 = no)
df_reg["ai_use"] = np.where(
    df_reg[ki_col] == 1, 1,
    np.where(df_reg[ki_col] == 2, 0, np.nan)
)

# Age: already age in years
df_reg["age_years"] = pd.to_numeric(df_reg[age_col], errors="coerce")

# Gender: keep only female/male, set all else to NaN
df_reg["gender_cat_label"] = df_reg[gender_col].map({
    1: "female",
    2: "male"
})

# optional: binary coding for regression
df_reg["gender_binary"] = df_reg[gender_col].map({
    1: 0,   # female
    2: 1    # male
})

# Employment: already binary
df_reg["employed"] = pd.to_numeric(df_reg[employed_col], errors="coerce")

# Education years
df_reg["education_years"] = pd.to_numeric(df_reg[edu_col], errors="coerce")

# xBILZEIT special code: still in school -> missing for regression
df_reg.loc[df_reg["education_years"] < 0, "education_years"] = np.nan


# -----------------------------
# 3) Optional: inspect missingness
# -----------------------------
model_vars = ["ai_use", "age_years", "gender_cat_label", "employed", "education_years"]

print("\nMissingness in model variables:")
print(df_reg[model_vars].isna().sum())

complete_case_mask = df_reg[model_vars].notna().all(axis=1)
print(f"\nComplete-case sample for regression: {int(complete_case_mask.sum())}")

df_reg_cc = df_reg.loc[complete_case_mask].copy()

### AI use based on sociodemographics

In [ ]:
# -----------------------------
# 3) Strict complete-case regression sample
# -----------------------------

# ensure correct types first
df_reg['ai_use'] = pd.to_numeric(df_reg['ai_use'], errors='coerce')
df_reg['age_years'] = pd.to_numeric(df_reg['age_years'], errors='coerce')
df_reg['education_years'] = pd.to_numeric(df_reg['education_years'], errors='coerce')
df_reg['employed'] = pd.to_numeric(df_reg['employed'], errors='coerce')


# safety: xBILZEIT-derived "still in school" / special negative codes should not enter regression
df_reg.loc[df_reg['education_years'] < 0, 'education_years'] = np.nan

required = ['ai_use', 'age_years', 'gender_cat_label', 'education_years', 'employed']

print(f'  Starting N (base predictors):   {len(df_reg):>6}')

remaining = df_reg.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f'  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})')

print(f'  Final complete cases:           {len(remaining):>6}')
print()

df_reg = remaining.copy()

# final types
df_reg['ai_use'] = df_reg['ai_use'].astype(int)
df_reg['employed'] = df_reg['employed'].astype(int)
df_reg['gender_cat_label'] = df_reg['gender_cat_label'].astype('category')

print(f'  Analysis N (final complete cases): {len(df_reg):,}')

# -----------------------------
# 4) Logistic regression
# -----------------------------
preds = ['age_years', 'gender_cat_label', 'education_years', 'employed']

print()
print('Model: ai_use ~ age_years + C(gender_cat_label) + education_years + employed')

if len(df_reg) < 10:
    print('Too few complete cases to fit model safely.')
elif df_reg['ai_use'].nunique() < 2:
    print('Outcome has only one class after filtering; model not fit.')
else:
    keep_preds = [p for p in preds if df_reg[p].nunique(dropna=True) > 1]
    dropped = [p for p in preds if p not in keep_preds]

    if dropped:
        print('Dropped constant predictor(s): ' + ', '.join(dropped))

    if not keep_preds:
        print('All predictors are constant; model not fit.')
    else:
        formula_terms = []
        for p in keep_preds:
            if p == 'gender_cat_label':
                formula_terms.append('C(gender_cat_label)')
            else:
                formula_terms.append(p)

        formula_fit = 'ai_use ~ ' + ' + '.join(formula_terms)
        print(f'Fitting: {formula_fit}')

        result = smf.logit(formula_fit, data=df_reg).fit(disp=0)
        print(result.summary())

        coef = result.params
        conf = result.conf_int()
        conf.columns = ['CI_low', 'CI_high']

        report = pd.DataFrame({
            'Coef': coef,
            'OR': np.exp(coef),
            'OR_95%_CI_low': np.exp(conf['CI_low']),
            'OR_95%_CI_high': np.exp(conf['CI_high']),
            'p': result.pvalues,
        })

        print()
        print('Odds ratios and 95% CI:')
        print(report.round(4).to_string())

### AI use based on sociodemographics + clinical

In [ ]:
# start from the regression-prepared dataset BEFORE complete-case restriction
df_ext = df_reg.copy()

# -----------------------------
# Build additional predictors
# -----------------------------
# already available from your harmonized/prepared data
df_ext["phq4_sum"] = pd.to_numeric(df_ext["h_phq4_total"], errors="coerce")
df_ext["behandlung"] = pd.to_numeric(df_ext["h_psycho_treatment"], errors="coerce")
df_ext["diagnose_selbst"] = pd.to_numeric(df_ext["h_dx_mental_ever"], errors="coerce")
df_ext["eq5d_daily"] = pd.to_numeric(df_ext["eq5d_usual_activities"], errors="coerce")

# keep only female/male in gender_cat_label; everything else becomes NaN
df_ext["gender_cat_label"] = df_ext["gender_cat_label"].where(
    df_ext["gender_cat_label"].isin(["female", "male"]),
    np.nan
)

# -----------------------------
# Complete-case regression sample
# -----------------------------
required = [
    "ai_use", "age_years", "gender_cat_label", "education_years", "employed",
    "phq4_sum", "behandlung", "diagnose_selbst", "eq5d_daily",
]

print("Diagnostic: row loss for extended AI-use regression")
print(f"  Starting N (base predictors):   {len(df_ext):>6}")

remaining = df_ext.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")

print(f"  Final complete cases:           {len(remaining):>6}")
print()

df_ext = df_ext.dropna(subset=required).copy()

# ensure types
df_ext["ai_use"] = pd.to_numeric(df_ext["ai_use"], errors="coerce").astype(int)
df_ext["age_years"] = pd.to_numeric(df_ext["age_years"], errors="coerce")
df_ext["education_years"] = pd.to_numeric(df_ext["education_years"], errors="coerce")
df_ext["employed"] = pd.to_numeric(df_ext["employed"], errors="coerce").astype(int)
df_ext["phq4_sum"] = pd.to_numeric(df_ext["phq4_sum"], errors="coerce")
df_ext["behandlung"] = pd.to_numeric(df_ext["behandlung"], errors="coerce").astype(int)
df_ext["diagnose_selbst"] = pd.to_numeric(df_ext["diagnose_selbst"], errors="coerce")
df_ext["eq5d_daily"] = pd.to_numeric(df_ext["eq5d_daily"], errors="coerce")

# female/male only
df_ext["gender_cat_label"] = pd.Categorical(
    df_ext["gender_cat_label"],
    categories=["female", "male"]
)

print(f"  Analysis N (final complete cases): {len(df_ext):,}")
print("  Gender counts:")
print(df_ext["gender_cat_label"].value_counts(dropna=False))

# -----------------------------
# Logistic regression
# -----------------------------
preds = [
    "age_years", "gender_cat_label", "education_years", "employed",
    "phq4_sum", "behandlung", "diagnose_selbst", "eq5d_daily"
]

print()
print("Model: ai_use ~ age_years + C(gender_cat_label) + education_years + employed + phq4_sum + behandlung + diagnose_selbst + eq5d_daily")

if len(df_ext) < 10:
    print("Too few complete cases to fit model safely.")
elif df_ext["ai_use"].nunique() < 2:
    print("Outcome has only one class after filtering; model not fit.")
else:
    keep_preds = [p for p in preds if df_ext[p].nunique(dropna=True) > 1]
    dropped = [p for p in preds if p not in keep_preds]

    if dropped:
        print("Dropped constant predictor(s): " + ", ".join(dropped))

    if not keep_preds:
        print("All predictors are constant; model not fit.")
    else:
        formula_terms = []
        for p in keep_preds:
            if p == "gender_cat_label":
                formula_terms.append("C(gender_cat_label)")
            else:
                formula_terms.append(p)

        formula_fit = "ai_use ~ " + " + ".join(formula_terms)
        print(f"Fitting: {formula_fit}")

        result = smf.logit(formula_fit, data=df_ext).fit(disp=0)
        print(result.summary())

        coef = result.params
        conf = result.conf_int()
        conf.columns = ["CI_low", "CI_high"]

        report = pd.DataFrame({
            "Coef": coef,
            "OR": np.exp(coef),
            "OR_95%_CI_low": np.exp(conf["CI_low"]),
            "OR_95%_CI_high": np.exp(conf["CI_high"]),
            "p": result.pvalues,
        })

        print()
        print("Odds ratios and 95% CI:")
        print(report.round(4).to_string())

### AI use for Medical Purposes

In [ ]:

df_reg_med = df_reg.loc[df_reg["ai_use_any"] == 1].copy()
# --------------------------------------------------
# 3) Build additional predictors
# --------------------------------------------------
df_reg_med["phq4_sum"] = pd.to_numeric(df_reg_med["h_phq4_total"], errors="coerce")
df_reg_med["behandlung"] = pd.to_numeric(df_reg_med["h_psycho_treatment"], errors="coerce")
df_reg_med["diagnose_selbst"] = pd.to_numeric(df_reg_med["h_dx_mental_ever"], errors="coerce")
df_reg_med["eq5d_daily"] = pd.to_numeric(df_reg_med["eq5d_usual_activities"], errors="coerce")

# keep only female/male; everything else -> NaN
df_reg_med["gender_cat_label"] = df_reg_med["gender_cat_label"].where(
    df_reg_med["gender_cat_label"].isin(["female", "male"]),
    np.nan
)

# convert predictors BEFORE complete-case restriction
for c in [
    "med_use", "age_years", "education_years", "employed",
    "phq4_sum", "behandlung", "diagnose_selbst", "eq5d_daily"
]:
    df_reg_med[c] = pd.to_numeric(df_reg_med[c], errors="coerce")

# --------------------------------------------------
# 4) Complete-case regression sample
# --------------------------------------------------
required = [
    "med_use", "age_years", "gender_cat_label", "education_years", "employed",
]

print("Diagnostic: row loss for med_use extended regression (AI users only)")
print(f"  Starting N (AI users):          {len(df_reg_med):>6}")

remaining = df_reg_med.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")

print(f"  Final complete cases:           {len(remaining):>6}")
print()

df_reg_med = df_reg_med.dropna(subset=required).copy()

# final types after CC restriction
df_reg_med["med_use"] = df_reg_med["med_use"].astype(int)
df_reg_med["employed"] = df_reg_med["employed"].astype(int)

df_reg_med["gender_cat_label"] = df_reg_med["gender_cat_label"].astype("category")
df_reg_med["gender_cat_label"] = df_reg_med["gender_cat_label"].cat.remove_unused_categories()

print(f"  Analysis N (final complete cases): {len(df_reg_med):,}")
print("  med_use counts:")
print(df_reg_med["med_use"].value_counts(dropna=False).sort_index())
print("  gender counts:")
print(df_reg_med["gender_cat_label"].value_counts(dropna=False))
print()

# --------------------------------------------------
# 5) Logistic regression
# --------------------------------------------------
preds = [
    "age_years", "gender_cat_label", "education_years", "employed",

]

print("Model: med_use ~ age_years + C(gender_cat_label) + education_years + employed")

if len(df_reg_med) < 10:
    print("Too few complete cases to fit model safely.")

elif df_reg_med["med_use"].nunique() < 2:
    print("med_use has only one class after filtering; model not fit.")

else:
    # drop constant raw predictors first
    keep_preds = [p for p in preds if df_reg_med[p].nunique(dropna=True) > 1]
    dropped_raw = [p for p in preds if p not in keep_preds]

    if dropped_raw:
        print("Dropped constant raw predictor(s): " + ", ".join(dropped_raw))

    if not keep_preds:
        print("All predictors are constant; model not fit.")

    else:
        formula_terms = []
        for p in keep_preds:
            if p == "gender_cat_label":
                formula_terms.append("C(gender_cat_label)")
            else:
                formula_terms.append(p)

        formula_fit = "med_use ~ " + " + ".join(formula_terms)
        print(f"Fitting: {formula_fit}")

        # --------------------------------------------------
        # Build design matrix explicitly
        # --------------------------------------------------
        y, X = patsy.dmatrices(formula_fit, data=df_reg_med, return_type="dataframe")
        y = np.ravel(y)

        # drop constant design-matrix columns except intercept
        const_cols = [c for c in X.columns if c != "Intercept" and X[c].nunique(dropna=False) <= 1]
        if const_cols:
            print("Dropped constant design-matrix column(s): " + ", ".join(const_cols))
            X = X.drop(columns=const_cols)

        # drop exact duplicate columns
        dup_cols = []
        cols = X.columns.tolist()
        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                if X.iloc[:, i].equals(X.iloc[:, j]):
                    dup_cols.append(cols[j])

        dup_cols = list(dict.fromkeys(dup_cols))
        if dup_cols:
            print("Dropped duplicate design-matrix column(s): " + ", ".join(dup_cols))
            X = X.drop(columns=dup_cols)

        print(f"  Design matrix shape: {X.shape}")
        print(f"  Design matrix rank:  {np.linalg.matrix_rank(X.to_numpy())}")

        # --------------------------------------------------
        # Try standard logit first
        # --------------------------------------------------
        fit_type = None
        result_med = None

        try:
            model = sm.Logit(y, X)
            result_med = model.fit(disp=0, maxiter=200)
            fit_type = "standard"
            print("Standard logistic regression fit succeeded.\n")

        except Exception as e:
            print(f"Standard logistic regression failed: {type(e).__name__}: {e}")
            print("Trying penalized logistic regression instead.\n")

            # penalized fallback avoids singular-matrix crash
            model = sm.Logit(y, X)
            result_med = model.fit_regularized(method="l1", alpha=0.01, disp=0)
            fit_type = "penalized"

        # --------------------------------------------------
        # Output
        # --------------------------------------------------
        try:
            print(result_med.summary())
        except Exception:
            print("Model fit completed, but no printable summary was available.")
            print()

        coef = result_med.params

        report = pd.DataFrame({
            "Coef": coef,
            "OR": np.exp(coef),
        })

        # add CI and p-values where available
        try:
            conf = result_med.conf_int()
            if isinstance(conf, pd.DataFrame):
                conf.columns = ["CI_low", "CI_high"]
                report["OR_95%_CI_low"] = np.exp(conf["CI_low"])
                report["OR_95%_CI_high"] = np.exp(conf["CI_high"])
            else:
                report["OR_95%_CI_low"] = np.exp(conf[:, 0])
                report["OR_95%_CI_high"] = np.exp(conf[:, 1])
        except Exception:
            report["OR_95%_CI_low"] = np.nan
            report["OR_95%_CI_high"] = np.nan

        try:
            report["p"] = result_med.pvalues
        except Exception:
            report["p"] = np.nan

        print(f"Odds ratios and 95% CI (med_use model; {fit_type} fit):")
        print(report.round(4).to_string())

### Medical Use based on sociodemographic + clinical variables

In [ ]:

df_reg_med = df_reg.loc[df_reg["ai_use_any"] == 1].copy()
# --------------------------------------------------
# 3) Build additional predictors
# --------------------------------------------------
df_reg_med["phq4_sum"] = pd.to_numeric(df_reg_med["h_phq4_total"], errors="coerce")
df_reg_med["behandlung"] = pd.to_numeric(df_reg_med["h_psycho_treatment"], errors="coerce")
df_reg_med["diagnose_selbst"] = pd.to_numeric(df_reg_med["h_dx_mental_ever"], errors="coerce")
df_reg_med["eq5d_daily"] = pd.to_numeric(df_reg_med["eq5d_usual_activities"], errors="coerce")

# keep only female/male; everything else -> NaN
df_reg_med["gender_cat_label"] = df_reg_med["gender_cat_label"].where(
    df_reg_med["gender_cat_label"].isin(["female", "male"]),
    np.nan
)

# convert predictors BEFORE complete-case restriction
for c in [
    "med_use", "age_years", "education_years", "employed",
    "phq4_sum", "behandlung", "diagnose_selbst", "eq5d_daily"
]:
    df_reg_med[c] = pd.to_numeric(df_reg_med[c], errors="coerce")

# --------------------------------------------------
# 4) Complete-case regression sample
# --------------------------------------------------
required = [
    "med_use", "age_years", "gender_cat_label", "education_years", "employed",
    "phq4_sum", "behandlung", "diagnose_selbst", "eq5d_daily",
]

print("Diagnostic: row loss for med_use extended regression (AI users only)")
print(f"  Starting N (AI users):          {len(df_reg_med):>6}")

remaining = df_reg_med.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")

print(f"  Final complete cases:           {len(remaining):>6}")
print()

df_reg_med = df_reg_med.dropna(subset=required).copy()

# final types after CC restriction
df_reg_med["med_use"] = df_reg_med["med_use"].astype(int)
df_reg_med["employed"] = df_reg_med["employed"].astype(int)
df_reg_med["behandlung"] = df_reg_med["behandlung"].astype(int)

df_reg_med["gender_cat_label"] = df_reg_med["gender_cat_label"].astype("category")
df_reg_med["gender_cat_label"] = df_reg_med["gender_cat_label"].cat.remove_unused_categories()

print(f"  Analysis N (final complete cases): {len(df_reg_med):,}")
print("  med_use counts:")
print(df_reg_med["med_use"].value_counts(dropna=False).sort_index())
print("  gender counts:")
print(df_reg_med["gender_cat_label"].value_counts(dropna=False))
print()

# --------------------------------------------------
# 5) Logistic regression
# --------------------------------------------------
preds = [
    "age_years", "gender_cat_label", "education_years", "employed",
    "phq4_sum", "behandlung", "diagnose_selbst", "eq5d_daily"
]

print("Model: med_use ~ age_years + C(gender_cat_label) + education_years + employed + phq4_sum + behandlung + diagnose_selbst + eq5d_daily")

if len(df_reg_med) < 10:
    print("Too few complete cases to fit model safely.")

elif df_reg_med["med_use"].nunique() < 2:
    print("med_use has only one class after filtering; model not fit.")

else:
    # drop constant raw predictors first
    keep_preds = [p for p in preds if df_reg_med[p].nunique(dropna=True) > 1]
    dropped_raw = [p for p in preds if p not in keep_preds]

    if dropped_raw:
        print("Dropped constant raw predictor(s): " + ", ".join(dropped_raw))

    if not keep_preds:
        print("All predictors are constant; model not fit.")

    else:
        formula_terms = []
        for p in keep_preds:
            if p == "gender_cat_label":
                formula_terms.append("C(gender_cat_label)")
            else:
                formula_terms.append(p)

        formula_fit = "med_use ~ " + " + ".join(formula_terms)
        print(f"Fitting: {formula_fit}")

        # --------------------------------------------------
        # Build design matrix explicitly
        # --------------------------------------------------
        y, X = patsy.dmatrices(formula_fit, data=df_reg_med, return_type="dataframe")
        y = np.ravel(y)

        # drop constant design-matrix columns except intercept
        const_cols = [c for c in X.columns if c != "Intercept" and X[c].nunique(dropna=False) <= 1]
        if const_cols:
            print("Dropped constant design-matrix column(s): " + ", ".join(const_cols))
            X = X.drop(columns=const_cols)

        # drop exact duplicate columns
        dup_cols = []
        cols = X.columns.tolist()
        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                if X.iloc[:, i].equals(X.iloc[:, j]):
                    dup_cols.append(cols[j])

        dup_cols = list(dict.fromkeys(dup_cols))
        if dup_cols:
            print("Dropped duplicate design-matrix column(s): " + ", ".join(dup_cols))
            X = X.drop(columns=dup_cols)

        print(f"  Design matrix shape: {X.shape}")
        print(f"  Design matrix rank:  {np.linalg.matrix_rank(X.to_numpy())}")
        print()

        # helpful diagnostics for possible separation
        for c in ["gender_cat_label", "employed", "behandlung", "diagnose_selbst"]:
            if c in df_reg_med.columns and c in keep_preds:
                print(f"Crosstab: {c} x med_use")
                print(pd.crosstab(df_reg_med[c], df_reg_med["med_use"], dropna=False))
                print()

        # --------------------------------------------------
        # Try standard logit first
        # --------------------------------------------------
        fit_type = None
        result_med = None

        try:
            model = sm.Logit(y, X)
            result_med = model.fit(disp=0, maxiter=200)
            fit_type = "standard"
            print("Standard logistic regression fit succeeded.\n")

        except Exception as e:
            print(f"Standard logistic regression failed: {type(e).__name__}: {e}")
            print("Trying penalized logistic regression instead.\n")

            # penalized fallback avoids singular-matrix crash
            model = sm.Logit(y, X)
            result_med = model.fit_regularized(method="l1", alpha=0.01, disp=0)
            fit_type = "penalized"

        # --------------------------------------------------
        # Output
        # --------------------------------------------------
        try:
            print(result_med.summary())
        except Exception:
            print("Model fit completed, but no printable summary was available.")
            print()

        coef = result_med.params

        report = pd.DataFrame({
            "Coef": coef,
            "OR": np.exp(coef),
        })

        # add CI and p-values where available
        try:
            conf = result_med.conf_int()
            if isinstance(conf, pd.DataFrame):
                conf.columns = ["CI_low", "CI_high"]
                report["OR_95%_CI_low"] = np.exp(conf["CI_low"])
                report["OR_95%_CI_high"] = np.exp(conf["CI_high"])
            else:
                report["OR_95%_CI_low"] = np.exp(conf[:, 0])
                report["OR_95%_CI_high"] = np.exp(conf[:, 1])
        except Exception:
            report["OR_95%_CI_low"] = np.nan
            report["OR_95%_CI_high"] = np.nan

        try:
            report["p"] = result_med.pvalues
        except Exception:
            report["p"] = np.nan

        print(f"Odds ratios and 95% CI (med_use model; {fit_type} fit):")
        print(report.round(4).to_string())

In [ ]:
df.groupby("ai_use_any")["id"].nunique()

### AI Use for Emotional Purposes

In [ ]:

df_reg_emo = df_reg.loc[df_reg["ai_use_any"] == 1].copy()
# --------------------------------------------------
# 3) Build additional predictors
# --------------------------------------------------
df_reg_emo["phq4_sum"] = pd.to_numeric(df_reg_emo["h_phq4_total"], errors="coerce")
df_reg_emo["behandlung"] = pd.to_numeric(df_reg_emo["h_psycho_treatment"], errors="coerce")
df_reg_emo["diagnose_selbst"] = pd.to_numeric(df_reg_emo["h_dx_mental_ever"], errors="coerce")
df_reg_emo["eq5d_daily"] = pd.to_numeric(df_reg_emo["eq5d_usual_activities"], errors="coerce")

# keep only female/male; everything else -> NaN
df_reg_emo["gender_cat_label"] = df_reg_emo["gender_cat_label"].where(
    df_reg_emo["gender_cat_label"].isin(["female", "male"]),
    np.nan
)

# convert predictors BEFORE complete-case restriction
for c in [
    "emo_use", "age_years", "education_years", "employed",
]:
    df_reg_emo[c] = pd.to_numeric(df_reg_emo[c], errors="coerce")

# --------------------------------------------------
# 4) Complete-case regression sample
# --------------------------------------------------
required = [
    "emo_use", "age_years", "gender_cat_label", "education_years", "employed",

]

print("Diagnostic: row loss for emo_use extended regression (AI users only)")
print(f"  Starting N (AI users):          {len(df_reg_emo):>6}")

remaining = df_reg_emo.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")

print(f"  Final complete cases:           {len(remaining):>6}")
print()

df_reg_emo = df_reg_emo.dropna(subset=required).copy()

# final types after CC restriction
df_reg_emo["emo_use"] = df_reg_emo["emo_use"].astype(int)
df_reg_emo["employed"] = df_reg_emo["employed"].astype(int)

df_reg_emo["gender_cat_label"] = df_reg_emo["gender_cat_label"].astype("category")
df_reg_emo["gender_cat_label"] = df_reg_emo["gender_cat_label"].cat.remove_unused_categories()

print(f"  Analysis N (final complete cases): {len(df_reg_emo):,}")
print("  emo_use counts:")
print(df_reg_emo["emo_use"].value_counts(dropna=False).sort_index())
print("  gender counts:")
print(df_reg_emo["gender_cat_label"].value_counts(dropna=False))
print()

# --------------------------------------------------
# 5) Logistic regression
# --------------------------------------------------
preds = [
    "age_years", "gender_cat_label", "education_years", "employed",

]

print("Model: emo_use ~ age_years + C(gender_cat_label) + education_years + employed")

if len(df_reg_emo) < 10:
    print("Too few complete cases to fit model safely.")

elif df_reg_emo["emo_use"].nunique() < 2:
    print("emo_use has only one class after filtering; model not fit.")

else:
    # drop constant raw predictors first
    keep_preds = [p for p in preds if df_reg_emo[p].nunique(dropna=True) > 1]
    dropped_raw = [p for p in preds if p not in keep_preds]

    if dropped_raw:
        print("Dropped constant raw predictor(s): " + ", ".join(dropped_raw))

    if not keep_preds:
        print("All predictors are constant; model not fit.")

    else:
        formula_terms = []
        for p in keep_preds:
            if p == "gender_cat_label":
                formula_terms.append("C(gender_cat_label)")
            else:
                formula_terms.append(p)

        formula_fit = "emo_use ~ " + " + ".join(formula_terms)
        print(f"Fitting: {formula_fit}")

        # --------------------------------------------------
        # Build design matrix explicitly
        # --------------------------------------------------
        y, X = patsy.dmatrices(formula_fit, data=df_reg_emo, return_type="dataframe")
        y = np.ravel(y)

        # drop constant design-matrix columns except intercept
        const_cols = [c for c in X.columns if c != "Intercept" and X[c].nunique(dropna=False) <= 1]
        if const_cols:
            print("Dropped constant design-matrix column(s): " + ", ".join(const_cols))
            X = X.drop(columns=const_cols)

        # drop exact duplicate columns
        dup_cols = []
        cols = X.columns.tolist()
        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                if X.iloc[:, i].equals(X.iloc[:, j]):
                    dup_cols.append(cols[j])

        dup_cols = list(dict.fromkeys(dup_cols))
        if dup_cols:
            print("Dropped duplicate design-matrix column(s): " + ", ".join(dup_cols))
            X = X.drop(columns=dup_cols)

        print(f"  Design matrix shape: {X.shape}")
        print(f"  Design matrix rank:  {np.linalg.matrix_rank(X.to_numpy())}")

        # --------------------------------------------------
        # Try standard logit first
        # --------------------------------------------------
        fit_type = None
        result_med = None

        try:
            model = sm.Logit(y, X)
            result_med = model.fit(disp=0, maxiter=200)
            fit_type = "standard"
            print("Standard logistic regression fit succeeded.\n")

        except Exception as e:
            print(f"Standard logistic regression failed: {type(e).__name__}: {e}")
            print("Trying penalized logistic regression instead.\n")

            # penalized fallback avoids singular-matrix crash
            model = sm.Logit(y, X)
            result_med = model.fit_regularized(method="l1", alpha=0.01, disp=0)
            fit_type = "penalized"

        # --------------------------------------------------
        # Output
        # --------------------------------------------------
        try:
            print(result_med.summary())
        except Exception:
            print("Model fit completed, but no printable summary was available.")
            print()

        coef = result_med.params

        report = pd.DataFrame({
            "Coef": coef,
            "OR": np.exp(coef),
        })

        # add CI and p-values where available
        try:
            conf = result_med.conf_int()
            if isinstance(conf, pd.DataFrame):
                conf.columns = ["CI_low", "CI_high"]
                report["OR_95%_CI_low"] = np.exp(conf["CI_low"])
                report["OR_95%_CI_high"] = np.exp(conf["CI_high"])
            else:
                report["OR_95%_CI_low"] = np.exp(conf[:, 0])
                report["OR_95%_CI_high"] = np.exp(conf[:, 1])
        except Exception:
            report["OR_95%_CI_low"] = np.nan
            report["OR_95%_CI_high"] = np.nan

        try:
            report["p"] = result_med.pvalues
        except Exception:
            report["p"] = np.nan

        print(f"Odds ratios and 95% CI (emo_use model; {fit_type} fit):")
        print(report.round(4).to_string())

In [ ]:
# --------------------------------------------------
# 4) Complete-case regression sample
# --------------------------------------------------
required = [
    "emo_use", "age_years", "gender_cat_label", "education_years", "employed",
    "phq4_sum", "behandlung", "diagnose_selbst", "eq5d_daily",
]

print("Diagnostic: row loss for emo_use extended regression (AI users only)")
print(f"  Starting N (AI users):          {len(df_reg_emo):>6}")

remaining = df_reg_emo.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")

print(f"  Final complete cases:           {len(remaining):>6}")
print()

df_reg_emo = df_reg_emo.dropna(subset=required).copy()

# final types after CC restriction
df_reg_emo["emo_use"] = df_reg_emo["emo_use"].astype(int)
df_reg_emo["employed"] = df_reg_emo["employed"].astype(int)
df_reg_emo["behandlung"] = df_reg_emo["behandlung"].astype(int)

df_reg_emo["gender_cat_label"] = df_reg_emo["gender_cat_label"].astype("category")
df_reg_emo["gender_cat_label"] = df_reg_emo["gender_cat_label"].cat.remove_unused_categories()

print(f"  Analysis N (final complete cases): {len(df_reg_emo):,}")
print("  emo_use counts:")
print(df_reg_emo["emo_use"].value_counts(dropna=False).sort_index())
print("  gender counts:")
print(df_reg_emo["gender_cat_label"].value_counts(dropna=False))
print()

# --------------------------------------------------
# 5) Logistic regression
# --------------------------------------------------
preds = [
    "age_years", "gender_cat_label", "education_years", "employed",
    "phq4_sum", "behandlung", "diagnose_selbst", "eq5d_daily",

]

print("Model: emo_use ~ age_years + C(gender_cat_label) + education_years + employed + phq4_sum + behandlung + diagnose_selbst + eq5d daily")

if len(df_reg_emo) < 10:
    print("Too few complete cases to fit model safely.")

elif df_reg_emo["emo_use"].nunique() < 2:
    print("emo_use has only one class after filtering; model not fit.")

else:
    # drop constant raw predictors first
    keep_preds = [p for p in preds if df_reg_emo[p].nunique(dropna=True) > 1]
    dropped_raw = [p for p in preds if p not in keep_preds]

    if dropped_raw:
        print("Dropped constant raw predictor(s): " + ", ".join(dropped_raw))

    if not keep_preds:
        print("All predictors are constant; model not fit.")

    else:
        formula_terms = []
        for p in keep_preds:
            if p == "gender_cat_label":
                formula_terms.append("C(gender_cat_label)")
            else:
                formula_terms.append(p)

        formula_fit = "emo_use ~ " + " + ".join(formula_terms)
        print(f"Fitting: {formula_fit}")

        # --------------------------------------------------
        # Build design matrix explicitly
        # --------------------------------------------------
        y, X = patsy.dmatrices(formula_fit, data=df_reg_emo, return_type="dataframe")
        y = np.ravel(y)

        # drop constant design-matrix columns except intercept
        const_cols = [c for c in X.columns if c != "Intercept" and X[c].nunique(dropna=False) <= 1]
        if const_cols:
            print("Dropped constant design-matrix column(s): " + ", ".join(const_cols))
            X = X.drop(columns=const_cols)

        # drop exact duplicate columns
        dup_cols = []
        cols = X.columns.tolist()
        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                if X.iloc[:, i].equals(X.iloc[:, j]):
                    dup_cols.append(cols[j])

        dup_cols = list(dict.fromkeys(dup_cols))
        if dup_cols:
            print("Dropped duplicate design-matrix column(s): " + ", ".join(dup_cols))
            X = X.drop(columns=dup_cols)

        print(f"  Design matrix shape: {X.shape}")
        print(f"  Design matrix rank:  {np.linalg.matrix_rank(X.to_numpy())}")

        # --------------------------------------------------
        # Try standard logit first
        # --------------------------------------------------
        fit_type = None
        result_med = None

        try:
            model = sm.Logit(y, X)
            result_med = model.fit(disp=0, maxiter=200)
            fit_type = "standard"
            print("Standard logistic regression fit succeeded.\n")

        except Exception as e:
            print(f"Standard logistic regression failed: {type(e).__name__}: {e}")
            print("Trying penalized logistic regression instead.\n")

            # penalized fallback avoids singular-matrix crash
            model = sm.Logit(y, X)
            result_med = model.fit_regularized(method="l1", alpha=0.01, disp=0)
            fit_type = "penalized"

        # --------------------------------------------------
        # Output
        # --------------------------------------------------
        try:
            print(result_med.summary())
        except Exception:
            print("Model fit completed, but no printable summary was available.")
            print()

        coef = result_med.params

        report = pd.DataFrame({
            "Coef": coef,
            "OR": np.exp(coef),
        })

        # add CI and p-values where available
        try:
            conf = result_med.conf_int()
            if isinstance(conf, pd.DataFrame):
                conf.columns = ["CI_low", "CI_high"]
                report["OR_95%_CI_low"] = np.exp(conf["CI_low"])
                report["OR_95%_CI_high"] = np.exp(conf["CI_high"])
            else:
                report["OR_95%_CI_low"] = np.exp(conf[:, 0])
                report["OR_95%_CI_high"] = np.exp(conf[:, 1])
        except Exception:
            report["OR_95%_CI_low"] = np.nan
            report["OR_95%_CI_high"] = np.nan

        try:
            report["p"] = result_med.pvalues
        except Exception:
            report["p"] = np.nan

        print(f"Odds ratios and 95% CI (emo_use model; {fit_type} fit):")
        print(report.round(4).to_string())

### Prediction of Side Effects based on Sociodemographics

In [ ]:

# --------------------------------------------------
# 1) Base sample: AI users
# --------------------------------------------------
df_sub = df_reg.loc[df_reg["ai_use"] == 1].copy()

# --------------------------------------------------
# 2) Side-effect coding (numeric only)
# --------------------------------------------------
side_effect_cols = {
    "ai_feel_dependent": "AI dependency",
    "ai_hard_to_decide_alone": "Decision difficulty",
    "ai_friends_relationship_change": "Social impact",
}

SOCIAL_YES_NUM = {1, 2, 3}

def to_yes_no(v, col):
    if pd.isna(v):
        return np.nan

    num = pd.to_numeric(v, errors="coerce")
    if pd.isna(num):
        return np.nan

    # -----------------------------
    # SPECIAL CASE: social variable
    # -----------------------------
    if col == "ai_friends_relationship_change":
        return 1 if int(num) in SOCIAL_YES_NUM else 0

    # -----------------------------
    # GENERAL CASE: agreement = 2–4
    # -----------------------------
    return 1 if 2 <= num <= 4 else 0


# apply coding
for col in side_effect_cols:
    ycol = f"{col}_yes"
    if col in df_sub.columns:
        df_sub[ycol] = df_sub[col].apply(lambda x: to_yes_no(x, col))
    else:
        df_sub[ycol] = np.nan

# -----------------------------
# Counts per side-effect item
# -----------------------------
print("Per-side-effect Yes/No/Missing counts:")

for col, label in side_effect_cols.items():
    ycol = f"{col}_yes"

    n_yes = int((df_sub[ycol] == 1).sum())
    n_no = int((df_sub[ycol] == 0).sum())
    n_miss = int(df_sub[ycol].isna().sum())

    n_ans = n_yes + n_no
    pct_yes = (100 * n_yes / n_ans) if n_ans > 0 else np.nan

    if n_ans > 0:
        print(f"  {label:<35} Yes={n_yes:>6}  No={n_no:>6}  Missing={n_miss:>6}  Yes%={pct_yes:.1f}")
    else:
        print(f"  {label:<35} Yes={n_yes:>6}  No={n_no:>6}  Missing={n_miss:>6}  Yes%=n/a")


# --------------------------------------------------
# 3) Person-level outcome: ANY side effect
# --------------------------------------------------
ycols = [f"{c}_yes" for c in side_effect_cols.keys()]

answered_any = df_sub[ycols].notna().any(axis=1)
any_yes = df_sub[ycols].eq(1).any(axis=1)

df_sub["side_effect_any"] = np.where(
    ~answered_any,
    np.nan,
    np.where(any_yes, 1, 0)
)

print("Outcome distribution:")
print(df_sub["side_effect_any"].value_counts(dropna=False))


# --------------------------------------------------
# 4) Gender: ONLY female/male
# --------------------------------------------------
df_sub["gender_cat_label"] = df_sub["gender_cat_label"].where(
    df_sub["gender_cat_label"].isin(["female", "male"])
)


# --------------------------------------------------
# 5) Complete-case regression sample
# --------------------------------------------------
required = [
    "side_effect_any",
    "age_years",
    "gender_cat_label",
    "education_years",
    "employed",
]

print("\nDiagnostic: row loss for side_effect regression")
print(f"  Starting N (AI users):          {len(df_sub):>6}")

remaining = df_sub.copy()
for c in required:
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} lost: {lost:>6}")

print(f"  Final complete cases:           {len(remaining):>6}")
print()

df_reg_se = df_sub.dropna(subset=required).copy()


# --------------------------------------------------
# 6) Ensure types
# --------------------------------------------------
df_reg_se["side_effect_any"] = df_reg_se["side_effect_any"].astype(int)
df_reg_se["employed"] = pd.to_numeric(df_reg_se["employed"], errors="coerce").astype(int)
df_reg_se["age_years"] = pd.to_numeric(df_reg_se["age_years"], errors="coerce")
df_reg_se["education_years"] = pd.to_numeric(df_reg_se["education_years"], errors="coerce")
df_reg_se["phq4_sum"] = pd.to_numeric(df_reg_se["h_phq4_total"], errors="coerce")
df_reg_se["behandlung"] = pd.to_numeric(df_reg_se["h_psycho_treatment"], errors="coerce")
df_reg_se["diagnose_selbst"] = pd.to_numeric(df_reg_se["h_dx_mental_ever"], errors="coerce")
df_reg_se["eq5d_daily"] = pd.to_numeric(df_reg_se["eq5d_usual_activities"], errors="coerce")

# --------------------------------------------------
# 7) Drop constant predictors
# --------------------------------------------------
preds = ["age_years", "gender_cat_label", "education_years", "employed"]
preds = [p for p in preds if df_reg_se[p].nunique(dropna=True) > 1]


# --------------------------------------------------
# 8) Build formula
# --------------------------------------------------
formula_terms = []
for p in preds:
    if p == "gender_cat_label":
        formula_terms.append("C(gender_cat_label)")
    else:
        formula_terms.append(p)

formula_fit = "side_effect_any ~ " + " + ".join(formula_terms)

print(f"Fitting: {formula_fit}")
print(f"N = {len(df_reg_se)}")


# --------------------------------------------------
# 9) Logistic regression (robust)
# --------------------------------------------------
try:
    result_se = smf.logit(formula_fit, data=df_reg_se).fit(disp=0)
    print(result_se.summary())

    coef = result_se.params
    conf = result_se.conf_int()
    conf.columns = ["CI_low", "CI_high"]

    report = pd.DataFrame({
        "Coef": coef,
        "OR": np.exp(coef),
        "OR_95%_CI_low": np.exp(conf["CI_low"]),
        "OR_95%_CI_high": np.exp(conf["CI_high"]),
        "p": result_se.pvalues,
    })

    print("\nOdds ratios and 95% CI:")
    print(report.round(4).to_string())

except Exception as e:
    print("Model failed:", e)

    # Debugging (if singular matrix etc.)
    print("\nDEBUG outcome distribution:")
    print(df_reg_se["side_effect_any"].value_counts())

    print("\nDEBUG cross-tabs:")
    print(pd.crosstab(df_reg_se["gender_cat_label"], df_reg_se["side_effect_any"]))


# --------------------------------------------------
# 5) Complete-case regression sample
# --------------------------------------------------
required = [
    "side_effect_any",
    "age_years",
    "gender_cat_label",
    "education_years",
    "employed",
    "phq4_sum", 
    "behandlung", 
    "diagnose_selbst", 
    "eq5d_daily"]

print("\nDiagnostic: row loss for side_effect regression")
print(f"  Starting N (AI users):          {len(df_reg_se):>6}")

remaining = df_reg_se.copy()
for c in required:
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} lost: {lost:>6}")

print(f"  Final complete cases:           {len(remaining):>6}")
print()

df_reg_se = df_reg_se.dropna(subset=required).copy()


# --------------------------------------------------
# 6) Ensure types
# --------------------------------------------------
df_reg_se["side_effect_any"] = df_reg_se["side_effect_any"].astype(int)
df_reg_se["employed"] = pd.to_numeric(df_reg_se["employed"], errors="coerce").astype(int)
df_reg_se["age_years"] = pd.to_numeric(df_reg_se["age_years"], errors="coerce")
df_reg_se["education_years"] = pd.to_numeric(df_reg_se["education_years"], errors="coerce")




# --------------------------------------------------
# 7) Drop constant predictors
# --------------------------------------------------
preds = ["age_years", "gender_cat_label", "education_years", "employed",
          "phq4_sum", "behandlung", "diagnose_selbst", "eq5d_daily",]
preds = [p for p in preds if df_reg_se[p].nunique(dropna=True) > 1]


# --------------------------------------------------
# 8) Build formula
# --------------------------------------------------
formula_terms = []
for p in preds:
    if p == "gender_cat_label":
        formula_terms.append("C(gender_cat_label)")
    else:
        formula_terms.append(p)

formula_fit = "side_effect_any ~ " + " + ".join(formula_terms)

print(f"Fitting: {formula_fit}")
print(f"N = {len(df_reg_se)}")


# --------------------------------------------------
# 9) Logistic regression (robust)
# --------------------------------------------------
try:
    result_se = smf.logit(formula_fit, data=df_reg_se).fit(disp=0)
    print(result_se.summary())

    coef = result_se.params
    conf = result_se.conf_int()
    conf.columns = ["CI_low", "CI_high"]

    report = pd.DataFrame({
        "Coef": coef,
        "OR": np.exp(coef),
        "OR_95%_CI_low": np.exp(conf["CI_low"]),
        "OR_95%_CI_high": np.exp(conf["CI_high"]),
        "p": result_se.pvalues,
    })

    print("\nOdds ratios and 95% CI:")
    print(report.round(4).to_string())

except Exception as e:
    print("Model failed:", e)

    # Debugging (if singular matrix etc.)
    print("\nDEBUG outcome distribution:")
    print(df_reg_se["side_effect_any"].value_counts())

    print("\nDEBUG cross-tabs:")
    print(pd.crosstab(df_reg_se["gender_cat_label"], df_reg_se["side_effect_any"]))

### Mediation Analysis

In [ ]:

# -----------------------------
# 1) Base sample (AI users)
# -----------------------------
df_mediation = df_reg.loc[df_reg["ai_use"] == 1].copy()



# optional diagnostics
print("Per-side-effect Yes/No/Missing counts:")
for col, label in side_effect_cols.items():
    yc = f"{col}_yes"
    n_yes = int((df_mediation[yc] == 1).sum())
    n_no = int((df_mediation[yc] == 0).sum())
    n_miss = int(df_mediation[yc].isna().sum())
    n_ans = n_yes + n_no
    pct_yes = (100 * n_yes / n_ans) if n_ans > 0 else np.nan

    if n_ans > 0:
        print(f"  {label:<35} Yes={n_yes:>6}  No={n_no:>6}  Missing={n_miss:>6}  Yes%={pct_yes:.1f}")
    else:
        print(f"  {label:<35} Yes={n_yes:>6}  No={n_no:>6}  Missing={n_miss:>6}  Yes%=n/a")

print("\nPerson-level outcome side_effect_any:")
print(df_mediation["side_effect_any"].value_counts(dropna=False).to_string())


# =========================================================
# 3) Complete-case sample
# =========================================================
X = ["age_years", "gender_cat_label", "education_years", "employed"]
needed = ["emo_use", "side_effect_any"] + X

dfm = df_mediation.dropna(subset=needed).copy()

# enforce types
dfm["emo_use"] = pd.to_numeric(dfm["emo_use"], errors="coerce")
dfm["side_effect_any"] = pd.to_numeric(dfm["side_effect_any"], errors="coerce")
dfm["employed"] = pd.to_numeric(dfm["employed"], errors="coerce")
dfm["age_years"] = pd.to_numeric(dfm["age_years"], errors="coerce")
dfm["education_years"] = pd.to_numeric(dfm["education_years"], errors="coerce")
dfm["phq4_sum"] = pd.to_numeric(dfm["h_phq4_total"], errors="coerce")
dfm["behandlung"] = pd.to_numeric(dfm["h_psycho_treatment"], errors="coerce")
dfm["diagnose_selbst"] = pd.to_numeric(dfm["h_dx_mental_ever"], errors="coerce")
dfm["eq5d_daily"] = pd.to_numeric(dfm["eq5d_usual_activities"], errors="coerce")

dfm = dfm.dropna(subset=needed).copy()

dfm["emo_use"] = dfm["emo_use"].astype(int)
dfm["side_effect_any"] = dfm["side_effect_any"].astype(int)
dfm["employed"] = dfm["employed"].astype(int)

# clean gender
dfm["gender_cat_label"] = pd.Categorical(dfm["gender_cat_label"])
dfm["gender_cat_label"] = dfm["gender_cat_label"].cat.remove_unused_categories()

print(f"\nComplete-case N: {len(dfm)}")
print("Gender distribution:")
print(dfm["gender_cat_label"].value_counts())


# =========================================================
# 4) Drop problematic predictors
# =========================================================
base_preds = ["age_years", "education_years", "employed"]

# keep only predictors with variation
keep_preds = [p for p in base_preds if dfm[p].nunique(dropna=True) > 1]

# gender only if >1 category present
if dfm["gender_cat_label"].nunique(dropna=True) > 1:
    keep_preds.append("C(gender_cat_label)")
else:
    print("Dropping gender (no variation)")

print("\nFinal predictors:", keep_preds)


# =========================================================
# 5) Safe model fitting function
# =========================================================
def safe_logit(formula, data):
    try:
        return smf.logit(formula, data=data).fit(disp=0)
    except Exception as e:
        print(f"Standard logit failed ({e}) -> trying regularized fit")
        return smf.logit(formula, data=data).fit_regularized(disp=0)

formula_X = " + ".join(keep_preds)


# =========================================================
# 6) Mediation models
# =========================================================
model_a = safe_logit(f"emo_use ~ {formula_X}", dfm)
model_c = safe_logit(f"side_effect_any ~ {formula_X}", dfm)
model_cp = safe_logit(f"side_effect_any ~ {formula_X} + emo_use", dfm)

print("\n=== Path a ===")
print(model_a.summary())

print("\n=== Total effect (c) ===")
print(model_c.summary())

print("\n=== Direct effect (c') ===")
print(model_cp.summary())


# =========================================================
# 7) Comparison table
# =========================================================
rows = []

compare_predictors = ["age_years", "education_years", "employed"]

# add gender dummy/dummies if present in fitted model
gender_terms = [p for p in model_c.params.index if p.startswith("C(gender_cat_label)")]

compare_predictors_full = compare_predictors + gender_terms

pretty_names = {
    "age_years": "age_years",
    "education_years": "education_years",
    "employed": "employed",
    "C(gender_cat_label)[T.male]": "male vs female",
    "C(gender_cat_label)[T.other]": "other vs female",
}

for p in compare_predictors_full:
    c = model_c.params.get(p, np.nan)
    cp = model_cp.params.get(p, np.nan)

    rows.append({
        "predictor": pretty_names.get(p, p),
        "OR_total": np.exp(c) if pd.notna(c) else np.nan,
        "OR_direct": np.exp(cp) if pd.notna(cp) else np.nan,
        "delta_logit": (c - cp) if pd.notna(c) and pd.notna(cp) else np.nan
    })

if "emo_use" in model_cp.params.index:
    rows.append({
        "predictor": "emo_use (b path)",
        "OR_total": np.nan,
        "OR_direct": np.exp(model_cp.params["emo_use"]),
        "delta_logit": np.nan
    })

compare = pd.DataFrame(rows)

print("\n=== Mediation comparison ===")
print(compare.round(4).to_string(index=False))


# =========================================================
# 8) Bootstrap (robust)
# =========================================================
B = 2000
rng = np.random.default_rng(42)

boot_predictors = compare_predictors_full.copy()
ab_store = {p: [] for p in boot_predictors}

ok = 0
fail = 0

for _ in range(B):
    idx = rng.integers(0, len(dfm), len(dfm))
    bs = dfm.iloc[idx]

    try:
        a_fit = safe_logit(f"emo_use ~ {formula_X}", bs)
        cp_fit = safe_logit(f"side_effect_any ~ {formula_X} + emo_use", bs)

        b = cp_fit.params.get("emo_use", np.nan)

        for p in boot_predictors:
            a = a_fit.params.get(p, np.nan)
            ab_store[p].append(a * b if pd.notna(a) and pd.notna(b) else np.nan)

        ok += 1

    except Exception:
        for p in boot_predictors:
            ab_store[p].append(np.nan)
        fail += 1

boot_rows = []
for p, vals in ab_store.items():
    vals = np.array(vals, dtype=float)
    vals = vals[~np.isnan(vals)]

    boot_rows.append({
        "predictor": pretty_names.get(p, p),
        "ab_mean": vals.mean() if len(vals) else np.nan,
        "ci_low": np.quantile(vals, 0.025) if len(vals) else np.nan,
        "ci_high": np.quantile(vals, 0.975) if len(vals) else np.nan,
        "n_valid_boot": len(vals)
    })

boot_table = pd.DataFrame(boot_rows)

print("\n=== Bootstrap summary ===")
print(f"Successful draws: {ok}")
print(f"Failed draws: {fail}")

print("\n=== Bootstrap indirect effects ===")
print(boot_table.round(4).to_string(index=False))

In [ ]:

# =========================================================
# 3) Complete-case sample
# =========================================================
X = ["age_years", "gender_cat_label", "education_years", "employed",  "phq4_sum", "behandlung", "diagnose_selbst", "eq5d_daily"]
needed = ["emo_use", "side_effect_any"] + X

dfm = dfm.dropna(subset=needed).copy()

# enforce types
dfm["emo_use"] = pd.to_numeric(dfm["emo_use"], errors="coerce")
dfm["side_effect_any"] = pd.to_numeric(dfm["side_effect_any"], errors="coerce")
dfm["employed"] = pd.to_numeric(dfm["employed"], errors="coerce")
dfm["age_years"] = pd.to_numeric(dfm["age_years"], errors="coerce")
dfm["education_years"] = pd.to_numeric(dfm["education_years"], errors="coerce")


dfm = dfm.dropna(subset=needed).copy()

dfm["emo_use"] = dfm["emo_use"].astype(int)
dfm["side_effect_any"] = dfm["side_effect_any"].astype(int)
dfm["employed"] = dfm["employed"].astype(int)

# clean gender
dfm["gender_cat_label"] = pd.Categorical(dfm["gender_cat_label"])
dfm["gender_cat_label"] = dfm["gender_cat_label"].cat.remove_unused_categories()

print(f"\nComplete-case N: {len(dfm)}")
print("Gender distribution:")
print(dfm["gender_cat_label"].value_counts())


# =========================================================
# 4) Drop problematic predictors
# =========================================================
base_preds = ["age_years", "education_years", "employed", "phq4_sum", "behandlung", "diagnose_selbst", "eq5d_daily"]

# keep only predictors with variation
keep_preds = [p for p in base_preds if dfm[p].nunique(dropna=True) > 1]

# gender only if >1 category present
if dfm["gender_cat_label"].nunique(dropna=True) > 1:
    keep_preds.append("C(gender_cat_label)")
else:
    print("Dropping gender (no variation)")

print("\nFinal predictors:", keep_preds)


# =========================================================
# 5) Safe model fitting function
# =========================================================
def safe_logit(formula, data):
    try:
        return smf.logit(formula, data=data).fit(disp=0)
    except Exception as e:
        print(f"Standard logit failed ({e}) -> trying regularized fit")
        return smf.logit(formula, data=data).fit_regularized(disp=0)

formula_X = " + ".join(keep_preds)


# =========================================================
# 6) Mediation models
# =========================================================
model_a = safe_logit(f"emo_use ~ {formula_X}", dfm)
model_c = safe_logit(f"side_effect_any ~ {formula_X}", dfm)
model_cp = safe_logit(f"side_effect_any ~ {formula_X} + emo_use", dfm)

print("\n=== Path a ===")
print(model_a.summary())

print("\n=== Total effect (c) ===")
print(model_c.summary())

print("\n=== Direct effect (c') ===")
print(model_cp.summary())


# =========================================================
# 7) Comparison table
# =========================================================
rows = []

compare_predictors = ["age_years", "education_years", "employed", "phq4_sum", "behandlung", "diagnose_selbst", "eq5d_daily"]

# add gender dummy/dummies if present in fitted model
gender_terms = [p for p in model_c.params.index if p.startswith("C(gender_cat_label)")]

compare_predictors_full = compare_predictors + gender_terms

pretty_names = {
    "age_years": "age_years",
    "education_years": "education_years",
    "employed": "employed",
    "C(gender_cat_label)[T.male]": "male vs female",
    "C(gender_cat_label)[T.other]": "other vs female",
}

for p in compare_predictors_full:
    c = model_c.params.get(p, np.nan)
    cp = model_cp.params.get(p, np.nan)

    rows.append({
        "predictor": pretty_names.get(p, p),
        "OR_total": np.exp(c) if pd.notna(c) else np.nan,
        "OR_direct": np.exp(cp) if pd.notna(cp) else np.nan,
        "delta_logit": (c - cp) if pd.notna(c) and pd.notna(cp) else np.nan
    })

if "emo_use" in model_cp.params.index:
    rows.append({
        "predictor": "emo_use (b path)",
        "OR_total": np.nan,
        "OR_direct": np.exp(model_cp.params["emo_use"]),
        "delta_logit": np.nan
    })

compare = pd.DataFrame(rows)

print("\n=== Mediation comparison ===")
print(compare.round(4).to_string(index=False))


# =========================================================
# 8) Bootstrap (robust)
# =========================================================
B = 2000
rng = np.random.default_rng(42)

boot_predictors = compare_predictors_full.copy()
ab_store = {p: [] for p in boot_predictors}

ok = 0
fail = 0

for _ in range(B):
    idx = rng.integers(0, len(dfm), len(dfm))
    bs = dfm.iloc[idx]

    try:
        a_fit = safe_logit(f"emo_use ~ {formula_X}", bs)
        cp_fit = safe_logit(f"side_effect_any ~ {formula_X} + emo_use", bs)

        b = cp_fit.params.get("emo_use", np.nan)

        for p in boot_predictors:
            a = a_fit.params.get(p, np.nan)
            ab_store[p].append(a * b if pd.notna(a) and pd.notna(b) else np.nan)

        ok += 1

    except Exception:
        for p in boot_predictors:
            ab_store[p].append(np.nan)
        fail += 1

boot_rows = []
for p, vals in ab_store.items():
    vals = np.array(vals, dtype=float)
    vals = vals[~np.isnan(vals)]

    boot_rows.append({
        "predictor": pretty_names.get(p, p),
        "ab_mean": vals.mean() if len(vals) else np.nan,
        "ci_low": np.quantile(vals, 0.025) if len(vals) else np.nan,
        "ci_high": np.quantile(vals, 0.975) if len(vals) else np.nan,
        "n_valid_boot": len(vals)
    })

boot_table = pd.DataFrame(boot_rows)

print("\n=== Bootstrap summary ===")
print(f"Successful draws: {ok}")
print(f"Failed draws: {fail}")

print("\n=== Bootstrap indirect effects ===")
print(boot_table.round(4).to_string(index=False))

### Prediction of Medical Use Intensity

In [ ]:

# --------------------------------------------------
# 1) Start from AI users only
# --------------------------------------------------
df_med_int = df_reg.loc[df_reg["ai_use_any"] == 1].copy()

# --------------------------------------------------
# 2) Medical-use columns
# --------------------------------------------------
medical_cols = [
    "ai_health_symptom_check",
    "ai_health_explain_med_docs",
    "ai_health_fitness_nutrition",
]
medical_cols = [c for c in medical_cols if c in df_med_int.columns]

if len(medical_cols) != 3:
    print("Warning: expected 3 medical-use columns, found:", len(medical_cols))
    print("Columns found:", medical_cols)

# --------------------------------------------------
# 3) Additional predictors (same as binary med_use model)
# --------------------------------------------------


df_med_int["gender_cat_label"] = df_med_int["gender_cat_label"].where(
    df_med_int["gender_cat_label"].isin(["female", "male"]),
    np.nan
)

# --------------------------------------------------
# 4) Recode medical item intensities
#    1-4 stay as intensity
#    5 = never -> 0
#    everything else -> NaN
# --------------------------------------------------
for c in medical_cols:
    df_med_int[c] = pd.to_numeric(df_med_int[c], errors="coerce")
    df_med_int[f"{c}_int"] = np.where(
        df_med_int[c].isin([1, 2, 3, 4]), df_med_int[c],
        np.where(df_med_int[c] == 5, 0, np.nan)
    )

int_cols = [f"{c}_int" for c in medical_cols]

# At least one actual medical-use response (1-4)
has_medical_use = df_med_int[int_cols].isin([1, 2, 3, 4]).any(axis=1)

# Sum score across the 3 items
# min_count=3 => score is NaN if any of the 3 items is missing
df_med_int["med_use_intensity_score"] = df_med_int[int_cols].sum(axis=1, min_count=3)

# --------------------------------------------------
# 5) Complete-case regression sample
# --------------------------------------------------
required = [
    "med_use_intensity_score",
    "age_years",
    "gender_cat_label",
    "education_years",
    "employed",

]

df_reg_int = df_med_int.loc[has_medical_use].copy()

print("Diagnostic: row loss for medical-use intensity regression")
print(f"  Starting N (AI users):                {len(df_med_int):>6}")
print(f"  With >=1 medical-use response (1-4): {int(has_medical_use.sum()):>6}")

remaining = df_reg_int.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")

print(f"  Final complete cases:                 {len(remaining):>6}")
print()

df_reg_int = df_reg_int.dropna(subset=required).copy()

# Final types
for c in [
    "med_use_intensity_score", "age_years", "education_years", "employed",

]:
    df_reg_int[c] = pd.to_numeric(df_reg_int[c], errors="coerce")

df_reg_int = df_reg_int.dropna(subset=required).copy()

df_reg_int["employed"] = df_reg_int["employed"].astype(int)


df_reg_int["gender_cat_label"] = df_reg_int["gender_cat_label"].astype("category")
df_reg_int["gender_cat_label"] = df_reg_int["gender_cat_label"].cat.remove_unused_categories()

print("Intensity regression among medical AI users:")
print("  Outcome: med_use_intensity_score")
print("  Coding: 1-4 = intensity, 5 = never -> 0, score = sum(3 items)")
print(f"  Analysis N (complete cases): {len(df_reg_int):,}")
print()

# --------------------------------------------------
# 6) Descriptives
# --------------------------------------------------
label_map = {1: "Selten", 2: "Manchmal", 3: "Oft", 4: "Immer"}

pooled = df_reg_int[int_cols].stack()
pooled = pooled[pooled.isin([1, 2, 3, 4])]
counts = pooled.value_counts().sort_index()

print("Intensity response frequencies (pooled across the 3 items):")
for k in [1, 2, 3, 4]:
    print(f"  {label_map[k]}: {int(counts.get(k, 0))}")
print()

print("Per-item intensity frequencies:")
for c in int_cols:
    vc = df_reg_int[c].value_counts().sort_index()
    print(f"  {c}: " + ", ".join([f"{label_map[k]}={int(vc.get(k, 0))}" for k in [1, 2, 3, 4]]))
print()

print("Outcome descriptives:")
print(df_reg_int["med_use_intensity_score"].describe().round(3).to_string())
print()

# --------------------------------------------------
# 7) Linear regression (OLS with HC3 robust SE)
# --------------------------------------------------
preds_int = [
    "age_years", "gender_cat_label", "education_years", "employed",
]

if len(df_reg_int) < 10:
    print("Too few complete cases to fit model safely.")

else:
    keep_int = [p for p in preds_int if df_reg_int[p].nunique(dropna=True) > 1]
    drop_int = [p for p in preds_int if p not in keep_int]

    if drop_int:
        print("Dropped constant predictor(s): " + ", ".join(drop_int))

    if not keep_int:
        print("All predictors are constant; model not fit.")

    else:
        formula_terms = []
        for p in keep_int:
            if p == "gender_cat_label":
                formula_terms.append("C(gender_cat_label)")
            else:
                formula_terms.append(p)

        formula_int = "med_use_intensity_score ~ " + " + ".join(formula_terms)
        print("Model:", formula_int)

        try:
            result_int = smf.ols(formula_int, data=df_reg_int).fit(cov_type="HC3")
            print(result_int.summary())

            coef = result_int.params
            conf = result_int.conf_int()
            pvals = result_int.pvalues

            report = pd.DataFrame({
                "b": coef,
                "CI_low": conf[0],
                "CI_high": conf[1],
                "p": pvals
            })

            print("\nCoefficients with 95% CI:")
            print(report.round(4).to_string())

        except Exception as e:
            print(f"Model fit failed: {e}")

In [ ]:

# --------------------------------------------------
# 1) Start from AI users only
# --------------------------------------------------
df_med_int = df_reg.loc[df_reg["ai_use_any"] == 1].copy()

# --------------------------------------------------
# 2) Medical-use columns
# --------------------------------------------------
medical_cols = [
    "ai_health_symptom_check",
    "ai_health_explain_med_docs",
    "ai_health_fitness_nutrition",
]
medical_cols = [c for c in medical_cols if c in df_med_int.columns]

if len(medical_cols) != 3:
    print("Warning: expected 3 medical-use columns, found:", len(medical_cols))
    print("Columns found:", medical_cols)

# --------------------------------------------------
# 3) Additional predictors (same as binary med_use model)
# --------------------------------------------------
df_med_int["phq4_sum"] = pd.to_numeric(df_med_int["h_phq4_total"], errors="coerce")
df_med_int["behandlung"] = pd.to_numeric(df_med_int["h_psycho_treatment"], errors="coerce")
df_med_int["diagnose_selbst"] = pd.to_numeric(df_med_int["h_dx_mental_ever"], errors="coerce")
df_med_int["eq5d_daily"] = pd.to_numeric(df_med_int["eq5d_usual_activities"], errors="coerce")

df_med_int["gender_cat_label"] = df_med_int["gender_cat_label"].where(
    df_med_int["gender_cat_label"].isin(["female", "male"]),
    np.nan
)

# --------------------------------------------------
# 4) Recode medical item intensities
#    1-4 stay as intensity
#    5 = never -> 0
#    everything else -> NaN
# --------------------------------------------------
for c in medical_cols:
    df_med_int[c] = pd.to_numeric(df_med_int[c], errors="coerce")
    df_med_int[f"{c}_int"] = np.where(
        df_med_int[c].isin([1, 2, 3, 4]), df_med_int[c],
        np.where(df_med_int[c] == 5, 0, np.nan)
    )

int_cols = [f"{c}_int" for c in medical_cols]

# At least one actual medical-use response (1-4)
has_medical_use = df_med_int[int_cols].isin([1, 2, 3, 4]).any(axis=1)

# Sum score across the 3 items
# min_count=3 => score is NaN if any of the 3 items is missing
df_med_int["med_use_intensity_score"] = df_med_int[int_cols].sum(axis=1, min_count=3)

# --------------------------------------------------
# 5) Complete-case regression sample
# --------------------------------------------------
required = [
    "med_use_intensity_score",
    "age_years",
    "gender_cat_label",
    "education_years",
    "employed",
    "phq4_sum",
    "behandlung",
    "diagnose_selbst",
    "eq5d_daily",
]

df_reg_int = df_med_int.loc[has_medical_use].copy()

print("Diagnostic: row loss for medical-use intensity regression")
print(f"  Starting N (AI users):                {len(df_med_int):>6}")
print(f"  With >=1 medical-use response (1-4): {int(has_medical_use.sum()):>6}")

remaining = df_reg_int.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")

print(f"  Final complete cases:                 {len(remaining):>6}")
print()

df_reg_int = df_reg_int.dropna(subset=required).copy()

# Final types
for c in [
    "med_use_intensity_score", "age_years", "education_years", "employed",
    "phq4_sum", "behandlung", "diagnose_selbst", "eq5d_daily"
]:
    df_reg_int[c] = pd.to_numeric(df_reg_int[c], errors="coerce")

df_reg_int = df_reg_int.dropna(subset=required).copy()

df_reg_int["employed"] = df_reg_int["employed"].astype(int)
df_reg_int["behandlung"] = df_reg_int["behandlung"].astype(int)
df_reg_int["diagnose_selbst"] = df_reg_int["diagnose_selbst"].astype(int)

df_reg_int["gender_cat_label"] = df_reg_int["gender_cat_label"].astype("category")
df_reg_int["gender_cat_label"] = df_reg_int["gender_cat_label"].cat.remove_unused_categories()

print("Intensity regression among medical AI users:")
print("  Outcome: med_use_intensity_score")
print("  Coding: 1-4 = intensity, 5 = never -> 0, score = sum(3 items)")
print(f"  Analysis N (complete cases): {len(df_reg_int):,}")
print()

# --------------------------------------------------
# 6) Descriptives
# --------------------------------------------------
label_map = {1: "Selten", 2: "Manchmal", 3: "Oft", 4: "Immer"}

pooled = df_reg_int[int_cols].stack()
pooled = pooled[pooled.isin([1, 2, 3, 4])]
counts = pooled.value_counts().sort_index()

print("Intensity response frequencies (pooled across the 3 items):")
for k in [1, 2, 3, 4]:
    print(f"  {label_map[k]}: {int(counts.get(k, 0))}")
print()

print("Per-item intensity frequencies:")
for c in int_cols:
    vc = df_reg_int[c].value_counts().sort_index()
    print(f"  {c}: " + ", ".join([f"{label_map[k]}={int(vc.get(k, 0))}" for k in [1, 2, 3, 4]]))
print()

print("Outcome descriptives:")
print(df_reg_int["med_use_intensity_score"].describe().round(3).to_string())
print()

# --------------------------------------------------
# 7) Linear regression (OLS with HC3 robust SE)
# --------------------------------------------------
preds_int = [
    "age_years", "gender_cat_label", "education_years", "employed",
    "phq4_sum", "behandlung", "diagnose_selbst", "eq5d_daily"
]

if len(df_reg_int) < 10:
    print("Too few complete cases to fit model safely.")

else:
    keep_int = [p for p in preds_int if df_reg_int[p].nunique(dropna=True) > 1]
    drop_int = [p for p in preds_int if p not in keep_int]

    if drop_int:
        print("Dropped constant predictor(s): " + ", ".join(drop_int))

    if not keep_int:
        print("All predictors are constant; model not fit.")

    else:
        formula_terms = []
        for p in keep_int:
            if p == "gender_cat_label":
                formula_terms.append("C(gender_cat_label)")
            else:
                formula_terms.append(p)

        formula_int = "med_use_intensity_score ~ " + " + ".join(formula_terms)
        print("Model:", formula_int)

        try:
            result_int = smf.ols(formula_int, data=df_reg_int).fit(cov_type="HC3")
            print(result_int.summary())

            coef = result_int.params
            conf = result_int.conf_int()
            pvals = result_int.pvalues

            report = pd.DataFrame({
                "b": coef,
                "CI_low": conf[0],
                "CI_high": conf[1],
                "p": pvals
            })

            print("\nCoefficients with 95% CI:")
            print(report.round(4).to_string())

        except Exception as e:
            print(f"Model fit failed: {e}")

### Prediction of Emotional Ai Use Intensity

In [ ]:

# --------------------------------------------------
# 1) Start from AI users only
# --------------------------------------------------
df_emo_int = df_reg.loc[df_reg["ai_use_any"] == 1].copy()

# --------------------------------------------------
# 2) Emotional-use columns
# --------------------------------------------------
emotional_cols = [    "ai_psych_sit_anxiety_worry",
    "ai_psych_sit_self_doubt_low_mood",
    "ai_psych_sit_loneliness",
    "ai_psych_sit_relationship_conflict",
    "ai_psych_sit_self_reflection",
    "ai_psych_sit_grief_loss",
    "ai_psych_sit_impulsive_or_substance",
    "ai_psych_sit_acute_crisis",
   "ai_psych_sit_other_selected",] 

emotional_cols = [c for c in emotional_cols if c in df_emo_int.columns]

# --------------------------------------------------
# 3) Additional predictors (same as binary emo_Use model)
# --------------------------------------------------


df_emo_int["gender_cat_label"] = df_emo_int["gender_cat_label"].where(
    df_emo_int["gender_cat_label"].isin(["female", "male"]),
    np.nan
)

# --------------------------------------------------
# 4) Recode medical item intensities
#    1-4 stay as intensity
#    5 = never -> 0
#    everything else -> NaN
# --------------------------------------------------
for c in emotional_cols:
    df_emo_int[c] = pd.to_numeric(df_emo_int[c], errors="coerce")
    df_emo_int[f"{c}_int"] = np.where(
        df_emo_int[c].isin([1, 2, 3, 4]), df_emo_int[c],
        np.where(df_emo_int[c] == 5, 0, np.nan)
    )

int_cols = [f"{c}_int" for c in emotional_cols]

# At least one actual emotional-use response (1-4)
has_emotional_use = df_emo_int[int_cols].isin([1, 2, 3, 4]).any(axis=1)

# Sum score across the 3 items
# min_count=3 => score is NaN if any of the 3 items is missing
df_emo_int["emo_use_intensity_score"] = df_emo_int[int_cols].sum(axis=1, min_count=3)

# --------------------------------------------------
# 5) Complete-case regression sample
# --------------------------------------------------
required = [
    "emo_use_intensity_score",
    "age_years",
    "gender_cat_label",
    "education_years",
    "employed",

]

df_reg_int = df_emo_int.loc[has_emotional_use].copy()

print("Diagnostic: row loss for emotional-use intensity regression")
print(f"  Starting N (AI users):                {len(df_emo_int):>6}")
print(f"  With >=1 emotioanl-use response (1-4): {int(has_emotional_use.sum()):>6}")

remaining = df_reg_int.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")

print(f"  Final complete cases:                 {len(remaining):>6}")
print()

df_reg_int = df_reg_int.dropna(subset=required).copy()

# Final types
for c in [
    "emo_use_intensity_score", "age_years", "education_years", "employed",

]:
    df_reg_int[c] = pd.to_numeric(df_reg_int[c], errors="coerce")

df_reg_int = df_reg_int.dropna(subset=required).copy()

df_reg_int["employed"] = df_reg_int["employed"].astype(int)


df_reg_int["gender_cat_label"] = df_reg_int["gender_cat_label"].astype("category")
df_reg_int["gender_cat_label"] = df_reg_int["gender_cat_label"].cat.remove_unused_categories()

print("Intensity regression among emotional AI users:")
print("  Outcome: emo_use_intensity_score")
print("  Coding: 1-4 = intensity, 5 = never -> 0, score = sum(3 items)")
print(f"  Analysis N (complete cases): {len(df_reg_int):,}")
print()

# --------------------------------------------------
# 6) Descriptives
# --------------------------------------------------
label_map = {1: "Selten", 2: "Manchmal", 3: "Oft", 4: "Immer"}

pooled = df_reg_int[int_cols].stack()
pooled = pooled[pooled.isin([1, 2, 3, 4])]
counts = pooled.value_counts().sort_index()

print("Intensity response frequencies (pooled across the 3 items):")
for k in [1, 2, 3, 4]:
    print(f"  {label_map[k]}: {int(counts.get(k, 0))}")
print()

print("Per-item intensity frequencies:")
for c in int_cols:
    vc = df_reg_int[c].value_counts().sort_index()
    print(f"  {c}: " + ", ".join([f"{label_map[k]}={int(vc.get(k, 0))}" for k in [1, 2, 3, 4]]))
print()

print("Outcome descriptives:")
print(df_reg_int["emo_use_intensity_score"].describe().round(3).to_string())
print()

# --------------------------------------------------
# 7) Linear regression (OLS with HC3 robust SE)
# --------------------------------------------------
preds_int = [
    "age_years", "gender_cat_label", "education_years", "employed",
]

if len(df_reg_int) < 10:
    print("Too few complete cases to fit model safely.")

else:
    keep_int = [p for p in preds_int if df_reg_int[p].nunique(dropna=True) > 1]
    drop_int = [p for p in preds_int if p not in keep_int]

    if drop_int:
        print("Dropped constant predictor(s): " + ", ".join(drop_int))

    if not keep_int:
        print("All predictors are constant; model not fit.")

    else:
        formula_terms = []
        for p in keep_int:
            if p == "gender_cat_label":
                formula_terms.append("C(gender_cat_label)")
            else:
                formula_terms.append(p)

        formula_int = "emo_use_intensity_score ~ " + " + ".join(formula_terms)
        print("Model:", formula_int)

        try:
            result_int = smf.ols(formula_int, data=df_reg_int).fit(cov_type="HC3")
            print(result_int.summary())

            coef = result_int.params
            conf = result_int.conf_int()
            pvals = result_int.pvalues

            report = pd.DataFrame({
                "b": coef,
                "CI_low": conf[0],
                "CI_high": conf[1],
                "p": pvals
            })

            print("\nCoefficients with 95% CI:")
            print(report.round(4).to_string())

        except Exception as e:
            print(f"Model fit failed: {e}")

In [ ]:

# --------------------------------------------------
# 1) Start from AI users only
# --------------------------------------------------
df_emo_int = df_reg.loc[df_reg["ai_use_any"] == 1].copy()

# --------------------------------------------------
# 2) Emotional-use columns
# --------------------------------------------------
emotional_cols = [
    "ai_psych_sit_anxiety_worry",
    "ai_psych_sit_self_doubt_low_mood",
    "ai_psych_sit_loneliness",
    "ai_psych_sit_relationship_conflict",
    "ai_psych_sit_self_reflection",
    "ai_psych_sit_grief_loss",
    "ai_psych_sit_impulsive_or_substance",
    "ai_psych_sit_acute_crisis",
    "ai_psych_sit_other_selected",
]
emotional_cols = [c for c in emotional_cols if c in df_emo_int.columns]

# --------------------------------------------------
# 3) Predictors
# --------------------------------------------------
df_emo_int["gender_cat_label"] = df_emo_int["gender_cat_label"].where(
    df_emo_int["gender_cat_label"].isin(["female", "male"]),
    np.nan
)

# --------------------------------------------------
# 4) Recode emotional item intensities
#    1-4 stay as intensity
#    5 = never -> 0
#    everything else -> NaN
# --------------------------------------------------
for c in emotional_cols:
    df_emo_int[c] = pd.to_numeric(df_emo_int[c], errors="coerce")
    df_emo_int[f"{c}_int"] = np.where(
        df_emo_int[c].isin([1, 2, 3, 4]), df_emo_int[c],
        np.where(df_emo_int[c] == 5, 0, np.nan)
    )

int_cols = [f"{c}_int" for c in emotional_cols]

# At least one actual emotional-use response (1-4)
has_emotional_use = df_emo_int[int_cols].isin([1, 2, 3, 4]).any(axis=1)

# Sum score across all emotional-use items
# min_count=len(int_cols) => score is NaN if any included item is missing
df_emo_int["emo_use_intensity_score"] = df_emo_int[int_cols].sum(
    axis=1,
    min_count=len(int_cols)
)

df_emo_int["phq4_sum"] = pd.to_numeric(df_emo_int["h_phq4_total"], errors="coerce")
df_emo_int["behandlung"] = pd.to_numeric(df_emo_int["h_psycho_treatment"], errors="coerce")
df_emo_int["diagnose_selbst"] = pd.to_numeric(df_emo_int["h_dx_mental_ever"], errors="coerce")
df_emo_int["eq5d_daily"] = pd.to_numeric(df_emo_int["eq5d_usual_activities"], errors="coerce")
# --------------------------------------------------
# 5) Complete-case regression sample
# --------------------------------------------------
required = [
    "emo_use_intensity_score",
    "age_years",
    "gender_cat_label",
    "education_years",
    "employed",
    "phq4_sum",
    "behandlung",
    "diagnose_selbst",
    "eq5d_daily",
]

df_reg_int = df_emo_int.loc[has_emotional_use].copy()

print("Diagnostic: row loss for emotional-use intensity regression (extended)")
print(f"  Starting N (AI users):                    {len(df_emo_int):>6}")
print(f"  With >=1 emotional-use response (1-4):   {int(has_emotional_use.sum()):>6}")

remaining = df_reg_int.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")

print(f"  Final complete cases:                     {len(remaining):>6}")
print()

df_reg_int = df_reg_int.dropna(subset=required).copy()

# --------------------------------------------------
# 6) Final types
# --------------------------------------------------
for c in [
    "emo_use_intensity_score",
    "age_years",
    "education_years",
    "employed",
    "phq4_sum",
    "behandlung",
    "diagnose_selbst",
    "eq5d_daily",
]:
    df_reg_int[c] = pd.to_numeric(df_reg_int[c], errors="coerce")

df_reg_int = df_reg_int.dropna(subset=required).copy()

df_reg_int["employed"] = df_reg_int["employed"].astype(int)
df_reg_int["behandlung"] = df_reg_int["behandlung"].astype(int)
df_reg_int["diagnose_selbst"] = df_reg_int["diagnose_selbst"].astype(int)

df_reg_int["gender_cat_label"] = df_reg_int["gender_cat_label"].astype("category")
df_reg_int["gender_cat_label"] = df_reg_int["gender_cat_label"].cat.remove_unused_categories()

print("Extended intensity regression among emotional AI users:")
print("  Outcome: emo_use_intensity_score")
print("  Coding: 1-4 = intensity, 5 = never -> 0")
print(f"  Number of emotional-use items included: {len(int_cols)}")
print(f"  Analysis N (complete cases): {len(df_reg_int):,}")
print()

# --------------------------------------------------
# 7) Descriptives
# --------------------------------------------------
label_map = {1: "Selten", 2: "Manchmal", 3: "Oft", 4: "Immer"}

pooled = df_reg_int[int_cols].stack()
pooled = pooled[pooled.isin([1, 2, 3, 4])]
counts = pooled.value_counts().sort_index()

print("Intensity response frequencies (pooled across emotional-use items):")
for k in [1, 2, 3, 4]:
    print(f"  {label_map[k]}: {int(counts.get(k, 0))}")
print()

print("Per-item intensity frequencies:")
for c in int_cols:
    vc = df_reg_int[c].value_counts().sort_index()
    print(f"  {c}: " + ", ".join([f"{label_map[k]}={int(vc.get(k, 0))}" for k in [1, 2, 3, 4]]))
print()

print("Outcome descriptives:")
print(df_reg_int["emo_use_intensity_score"].describe().round(3).to_string())
print()

# --------------------------------------------------
# 8) Linear regression (OLS with HC3 robust SE)
# --------------------------------------------------
preds_int = [
    "age_years",
    "gender_cat_label",
    "education_years",
    "employed",
    "phq4_sum",
    "behandlung",
    "diagnose_selbst",
    "eq5d_daily",
]

if len(df_reg_int) < 10:
    print("Too few complete cases to fit model safely.")

else:
    keep_int = [p for p in preds_int if df_reg_int[p].nunique(dropna=True) > 1]
    drop_int = [p for p in preds_int if p not in keep_int]

    if drop_int:
        print("Dropped constant predictor(s): " + ", ".join(drop_int))

    if not keep_int:
        print("All predictors are constant; model not fit.")

    else:
        formula_terms = []
        for p in keep_int:
            if p == "gender_cat_label":
                formula_terms.append("C(gender_cat_label)")
            else:
                formula_terms.append(p)

        formula_int = "emo_use_intensity_score ~ " + " + ".join(formula_terms)
        print("Model:", formula_int)

        try:
            result_int = smf.ols(formula_int, data=df_reg_int).fit(cov_type="HC3")
            print(result_int.summary())

            coef = result_int.params
            conf = result_int.conf_int()
            pvals = result_int.pvalues

            report = pd.DataFrame({
                "b": coef,
                "CI_low": conf[0],
                "CI_high": conf[1],
                "p": pvals
            })

            print("\nCoefficients with 95% CI:")
            print(report.round(4).to_string())

        except Exception as e:
            print(f"Model fit failed: {e}")

### Prediction of Side Effect Intensity

In [ ]:

# --------------------------------------------------
# 1) Start from AI users only
# --------------------------------------------------
df_emo_int = df_reg.loc[df_reg["ai_use_any"] == 1].copy()

df_emo_int["ai_friends_relationship_change_negative"] = df_emo_int["ai_friends_relationship_change"].map({1: 3,2: 2,3: 1,4: 0,5: 0, 6: 0,7: 0})

df_emo_int["ai_feel_dependent"] = df_emo_int["ai_feel_dependent"] - 1
df_emo_int["ai_hard_to_decide_alone"] = df_emo_int["ai_hard_to_decide_alone"] - 1

side_effect_cols = [
    "ai_friends_relationship_change_negative",
    "ai_feel_dependent",
    "ai_hard_to_decide_alone"
]

# make sure variables are numeric
df_emo_int[side_effect_cols] = df_emo_int[side_effect_cols].apply(
    pd.to_numeric, errors="coerce"
)


# sum of side effect intensities
side_effect_sum = df_emo_int[side_effect_cols].sum(axis=1, skipna=True)

# number of endorsed side effects (scores > 0)
n_endorsed = (df_emo_int[side_effect_cols] > 0).sum(axis=1)

# intensity score = sum / number of endorsed side effects
df_emo_int["side_effect_intensity"] = side_effect_sum.div(n_endorsed)

# optional: set rows with no endorsed side effect to 0 instead of NaN
df_emo_int.loc[n_endorsed == 0, "side_effect_intensity"] = 0

In [ ]:



# --------------------------------------------------
# 3) Additional predictors (same as binary emo_Use model)
# --------------------------------------------------


df_emo_int["gender_cat_label"] = df_emo_int["gender_cat_label"].where(
    df_emo_int["gender_cat_label"].isin(["female", "male"]),
    np.nan
)

# --------------------------------------------------
# 5) Complete-case regression sample
# --------------------------------------------------
required = [
    "side_effect_intensity",
    "age_years",
    "gender_cat_label",
    "education_years",
    "employed",

]

df_reg_int = df_emo_int.copy()

print("Diagnostic: row loss for side effect intensity regression")
print(f"  Starting N (AI users):                {len(df_emo_int):>6}")

remaining = df_reg_int.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")

print(f"  Final complete cases:                 {len(remaining):>6}")
print()

df_reg_int = df_reg_int.dropna(subset=required).copy()

# Final types
for c in [
    "side_effect_intensity", "age_years", "education_years", "employed",

]:
    df_reg_int[c] = pd.to_numeric(df_reg_int[c], errors="coerce")

df_reg_int = df_reg_int.dropna(subset=required).copy()

df_reg_int["employed"] = df_reg_int["employed"].astype(int)


df_reg_int["gender_cat_label"] = df_reg_int["gender_cat_label"].astype("category")
df_reg_int["gender_cat_label"] = df_reg_int["gender_cat_label"].cat.remove_unused_categories()

print("Intensity regression among emotional AI users:")
print("  Outcome: side_effect_intensity")
print("  Coding: 1-4 = intensity, 5 = never -> 0, score = sum(3 items)")
print(f"  Analysis N (complete cases): {len(df_reg_int):,}")
print()

# --------------------------------------------------
# 6) Descriptives
# --------------------------------------------------

print("Outcome descriptives:")
print(df_reg_int["side_effect_intensity"].describe().round(3).to_string())
print()

# --------------------------------------------------
# 7) Linear regression (OLS with HC3 robust SE)
# --------------------------------------------------
preds_int = [
    "age_years", "gender_cat_label", "education_years", "employed",
]

if len(df_reg_int) < 10:
    print("Too few complete cases to fit model safely.")

else:
    keep_int = [p for p in preds_int if df_reg_int[p].nunique(dropna=True) > 1]
    drop_int = [p for p in preds_int if p not in keep_int]

    if drop_int:
        print("Dropped constant predictor(s): " + ", ".join(drop_int))

    if not keep_int:
        print("All predictors are constant; model not fit.")

    else:
        formula_terms = []
        for p in keep_int:
            if p == "gender_cat_label":
                formula_terms.append("C(gender_cat_label)")
            else:
                formula_terms.append(p)

        formula_int = "side_effect_intensity ~ " + " + ".join(formula_terms)
        print("Model:", formula_int)

        try:
            result_int = smf.ols(formula_int, data=df_reg_int).fit(cov_type="HC3")
            print(result_int.summary())

            coef = result_int.params
            conf = result_int.conf_int()
            pvals = result_int.pvalues

            report = pd.DataFrame({
                "b": coef,
                "CI_low": conf[0],
                "CI_high": conf[1],
                "p": pvals
            })

            print("\nCoefficients with 95% CI:")
            print(report.round(4).to_string())

        except Exception as e:
            print(f"Model fit failed: {e}")

In [ ]:


df_emo_int["phq4_sum"] = pd.to_numeric(df_emo_int["h_phq4_total"], errors="coerce")
df_emo_int["behandlung"] = pd.to_numeric(df_emo_int["h_psycho_treatment"], errors="coerce")
df_emo_int["diagnose_selbst"] = pd.to_numeric(df_emo_int["h_dx_mental_ever"], errors="coerce")
df_emo_int["eq5d_daily"] = pd.to_numeric(df_emo_int["eq5d_usual_activities"], errors="coerce")
# --------------------------------------------------
# 5) Complete-case regression sample
# --------------------------------------------------
required = [
    "side_effect_intensity",
    "age_years",
    "gender_cat_label",
    "education_years",
    "employed",
    "phq4_sum",
    "behandlung",
    "diagnose_selbst",
    "eq5d_daily",
]

df_reg_int = df_emo_int.loc[has_emotional_use].copy()

print("Diagnostic: row loss for emotional-use intensity regression (extended)")
print(f"  Starting N (AI users):                    {len(df_emo_int):>6}")

remaining = df_reg_int.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")

print(f"  Final complete cases:                     {len(remaining):>6}")
print()

df_reg_int = df_reg_int.dropna(subset=required).copy()

# --------------------------------------------------
# 6) Final types
# --------------------------------------------------
for c in [
    "side_effect_intensity",
    "age_years",
    "education_years",
    "employed",
    "phq4_sum",
    "behandlung",
    "diagnose_selbst",
    "eq5d_daily",
]:
    df_reg_int[c] = pd.to_numeric(df_reg_int[c], errors="coerce")

df_reg_int = df_reg_int.dropna(subset=required).copy()

df_reg_int["employed"] = df_reg_int["employed"].astype(int)
df_reg_int["behandlung"] = df_reg_int["behandlung"].astype(int)
df_reg_int["diagnose_selbst"] = df_reg_int["diagnose_selbst"].astype(int)

df_reg_int["gender_cat_label"] = df_reg_int["gender_cat_label"].astype("category")
df_reg_int["gender_cat_label"] = df_reg_int["gender_cat_label"].cat.remove_unused_categories()

print("Extended intensity regression among emotional AI users:")
print("  Outcome: side_effect_intensity")
print("  Coding: 1-3 = intensity, 0 = never ")
print(f"  Number of emotional-use items included: {len(int_cols)}")
print(f"  Analysis N (complete cases): {len(df_reg_int):,}")
print()

# --------------------------------------------------
# 7) Descriptives
# --------------------------------------------------

print("Outcome descriptives:")
print(df_reg_int["side_effect_intensity"].describe().round(3).to_string())
print()

# --------------------------------------------------
# 8) Linear regression (OLS with HC3 robust SE)
# --------------------------------------------------
preds_int = [
    "age_years",
    "gender_cat_label",
    "education_years",
    "employed",
    "phq4_sum",
    "behandlung",
    "diagnose_selbst",
    "eq5d_daily",
]

if len(df_reg_int) < 10:
    print("Too few complete cases to fit model safely.")

else:
    keep_int = [p for p in preds_int if df_reg_int[p].nunique(dropna=True) > 1]
    drop_int = [p for p in preds_int if p not in keep_int]

    if drop_int:
        print("Dropped constant predictor(s): " + ", ".join(drop_int))

    if not keep_int:
        print("All predictors are constant; model not fit.")

    else:
        formula_terms = []
        for p in keep_int:
            if p == "gender_cat_label":
                formula_terms.append("C(gender_cat_label)")
            else:
                formula_terms.append(p)

        formula_int = "side_effect_intensity ~ " + " + ".join(formula_terms)
        print("Model:", formula_int)

        try:
            result_int = smf.ols(formula_int, data=df_reg_int).fit(cov_type="HC3")
            print(result_int.summary())

            coef = result_int.params
            conf = result_int.conf_int()
            pvals = result_int.pvalues

            report = pd.DataFrame({
                "b": coef,
                "CI_low": conf[0],
                "CI_high": conf[1],
                "p": pvals
            })

            print("\nCoefficients with 95% CI:")
            print(report.round(4).to_string())

        except Exception as e:
            print(f"Model fit failed: {e}")

## Sensitivity Analysis: Regression

In [ ]:

df_reg_emo = df_reg.loc[df_reg["ai_use_any"] == 1].copy()
# --------------------------------------------------
# 3) Build additional predictors
# --------------------------------------------------
df_reg_emo["behandlung"] = pd.to_numeric(df_reg_emo["h_psycho_treatment"], errors="coerce")
df_reg_emo["diagnose_selbst"] = pd.to_numeric(df_reg_emo["h_dx_mental_ever"], errors="coerce")
df_reg_emo["eq5d_daily"] = pd.to_numeric(df_reg_emo["eq5d_usual_activities"], errors="coerce")

# keep only female/male; everything else -> NaN
df_reg_emo["gender_cat_label"] = df_reg_emo["gender_cat_label"].where(
    df_reg_emo["gender_cat_label"].isin(["female", "male"]),
    np.nan
)

# convert predictors BEFORE complete-case restriction
for c in [
    "emo_use", "age_years", "education_years", "employed",
]:
    df_reg_emo[c] = pd.to_numeric(df_reg_emo[c], errors="coerce")

# --------------------------------------------------
# 4) Complete-case regression sample
# --------------------------------------------------
required = [
    "emo_use", "age_years", "gender_cat_label", "education_years", "employed",

]

print("Diagnostic: row loss for emo_use extended regression (AI users only)")
print(f"  Starting N (AI users):          {len(df_reg_emo):>6}")

remaining = df_reg_emo.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")

print(f"  Final complete cases:           {len(remaining):>6}")
print()

df_reg_emo = df_reg_emo.dropna(subset=required).copy()

# final types after CC restriction
df_reg_emo["emo_use"] = df_reg_emo["emo_use"].astype(int)
df_reg_emo["employed"] = df_reg_emo["employed"].astype(int)

df_reg_emo["gender_cat_label"] = df_reg_emo["gender_cat_label"].astype("category")
df_reg_emo["gender_cat_label"] = df_reg_emo["gender_cat_label"].cat.remove_unused_categories()

print(f"  Analysis N (final complete cases): {len(df_reg_emo):,}")
print("  emo_use counts:")
print(df_reg_emo["emo_use"].value_counts(dropna=False).sort_index())
print("  gender counts:")
print(df_reg_emo["gender_cat_label"].value_counts(dropna=False))
print()

# --------------------------------------------------
# 5) Logistic regression
# --------------------------------------------------
preds = [
    "age_years", "gender_cat_label", "education_years", "employed",

]

print("Model: emo_use ~ age_years + C(gender_cat_label) + education_years + employed")

if len(df_reg_emo) < 10:
    print("Too few complete cases to fit model safely.")

elif df_reg_emo["emo_use"].nunique() < 2:
    print("emo_use has only one class after filtering; model not fit.")

else:
    # drop constant raw predictors first
    keep_preds = [p for p in preds if df_reg_emo[p].nunique(dropna=True) > 1]
    dropped_raw = [p for p in preds if p not in keep_preds]

    if dropped_raw:
        print("Dropped constant raw predictor(s): " + ", ".join(dropped_raw))

    if not keep_preds:
        print("All predictors are constant; model not fit.")

    else:
        formula_terms = []
        for p in keep_preds:
            if p == "gender_cat_label":
                formula_terms.append("C(gender_cat_label)")
            else:
                formula_terms.append(p)

        formula_fit = "emo_use ~ " + " + ".join(formula_terms)
        print(f"Fitting: {formula_fit}")

        # --------------------------------------------------
        # Build design matrix explicitly
        # --------------------------------------------------
        y, X = patsy.dmatrices(formula_fit, data=df_reg_emo, return_type="dataframe")
        y = np.ravel(y)

        # drop constant design-matrix columns except intercept
        const_cols = [c for c in X.columns if c != "Intercept" and X[c].nunique(dropna=False) <= 1]
        if const_cols:
            print("Dropped constant design-matrix column(s): " + ", ".join(const_cols))
            X = X.drop(columns=const_cols)

        # drop exact duplicate columns
        dup_cols = []
        cols = X.columns.tolist()
        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                if X.iloc[:, i].equals(X.iloc[:, j]):
                    dup_cols.append(cols[j])

        dup_cols = list(dict.fromkeys(dup_cols))
        if dup_cols:
            print("Dropped duplicate design-matrix column(s): " + ", ".join(dup_cols))
            X = X.drop(columns=dup_cols)

        print(f"  Design matrix shape: {X.shape}")
        print(f"  Design matrix rank:  {np.linalg.matrix_rank(X.to_numpy())}")

        # --------------------------------------------------
        # Try standard logit first
        # --------------------------------------------------
        fit_type = None
        result_med = None

        try:
            model = sm.Logit(y, X)
            result_med = model.fit(disp=0, maxiter=200)
            fit_type = "standard"
            print("Standard logistic regression fit succeeded.\n")

        except Exception as e:
            print(f"Standard logistic regression failed: {type(e).__name__}: {e}")
            print("Trying penalized logistic regression instead.\n")

            # penalized fallback avoids singular-matrix crash
            model = sm.Logit(y, X)
            result_med = model.fit_regularized(method="l1", alpha=0.01, disp=0)
            fit_type = "penalized"

        # --------------------------------------------------
        # Output
        # --------------------------------------------------
        try:
            print(result_med.summary())
        except Exception:
            print("Model fit completed, but no printable summary was available.")
            print()

        coef = result_med.params

        report = pd.DataFrame({
            "Coef": coef,
            "OR": np.exp(coef),
        })

        # add CI and p-values where available
        try:
            conf = result_med.conf_int()
            if isinstance(conf, pd.DataFrame):
                conf.columns = ["CI_low", "CI_high"]
                report["OR_95%_CI_low"] = np.exp(conf["CI_low"])
                report["OR_95%_CI_high"] = np.exp(conf["CI_high"])
            else:
                report["OR_95%_CI_low"] = np.exp(conf[:, 0])
                report["OR_95%_CI_high"] = np.exp(conf[:, 1])
        except Exception:
            report["OR_95%_CI_low"] = np.nan
            report["OR_95%_CI_high"] = np.nan

        try:
            report["p"] = result_med.pvalues
        except Exception:
            report["p"] = np.nan

        print(f"Odds ratios and 95% CI (emo_use model; {fit_type} fit):")
        print(report.round(4).to_string())

In [ ]:
# --------------------------------------------------
# 4) Complete-case regression sample
# --------------------------------------------------
required = [
    "emo_use", "age_years", "gender_cat_label", "education_years", "employed",
    "P", "behandlung", "diagnose_selbst", "eq5d_daily",
]

print("Diagnostic: row loss for emo_use extended regression (AI users only)")
print(f"  Starting N (AI users):          {len(df_reg_emo):>6}")

remaining = df_reg_emo.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")

print(f"  Final complete cases:           {len(remaining):>6}")
print()

df_reg_emo = df_reg_emo.dropna(subset=required).copy()

# final types after CC restriction
df_reg_emo["emo_use"] = df_reg_emo["emo_use"].astype(int)
df_reg_emo["employed"] = df_reg_emo["employed"].astype(int)
df_reg_emo["behandlung"] = df_reg_emo["behandlung"].astype(int)

df_reg_emo["gender_cat_label"] = df_reg_emo["gender_cat_label"].astype("category")
df_reg_emo["gender_cat_label"] = df_reg_emo["gender_cat_label"].cat.remove_unused_categories()

print(f"  Analysis N (final complete cases): {len(df_reg_emo):,}")
print("  emo_use counts:")
print(df_reg_emo["emo_use"].value_counts(dropna=False).sort_index())
print("  gender counts:")
print(df_reg_emo["gender_cat_label"].value_counts(dropna=False))
print()

# --------------------------------------------------
# 5) Logistic regression
# --------------------------------------------------
preds = [
    "age_years", "gender_cat_label", "education_years", "employed",
    "P", "behandlung", "diagnose_selbst", "eq5d_daily",

]

print("Model: emo_use ~ age_years + C(gender_cat_label) + education_years + employed + P + behandlung + diagnose_selbst + eq5d daily")

if len(df_reg_emo) < 10:
    print("Too few complete cases to fit model safely.")

elif df_reg_emo["emo_use"].nunique() < 2:
    print("emo_use has only one class after filtering; model not fit.")

else:
    # drop constant raw predictors first
    keep_preds = [p for p in preds if df_reg_emo[p].nunique(dropna=True) > 1]
    dropped_raw = [p for p in preds if p not in keep_preds]

    if dropped_raw:
        print("Dropped constant raw predictor(s): " + ", ".join(dropped_raw))

    if not keep_preds:
        print("All predictors are constant; model not fit.")

    else:
        formula_terms = []
        for p in keep_preds:
            if p == "gender_cat_label":
                formula_terms.append("C(gender_cat_label)")
            else:
                formula_terms.append(p)

        formula_fit = "emo_use ~ " + " + ".join(formula_terms)
        print(f"Fitting: {formula_fit}")

        # --------------------------------------------------
        # Build design matrix explicitly
        # --------------------------------------------------
        y, X = patsy.dmatrices(formula_fit, data=df_reg_emo, return_type="dataframe")
        y = np.ravel(y)

        # drop constant design-matrix columns except intercept
        const_cols = [c for c in X.columns if c != "Intercept" and X[c].nunique(dropna=False) <= 1]
        if const_cols:
            print("Dropped constant design-matrix column(s): " + ", ".join(const_cols))
            X = X.drop(columns=const_cols)

        # drop exact duplicate columns
        dup_cols = []
        cols = X.columns.tolist()
        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                if X.iloc[:, i].equals(X.iloc[:, j]):
                    dup_cols.append(cols[j])

        dup_cols = list(dict.fromkeys(dup_cols))
        if dup_cols:
            print("Dropped duplicate design-matrix column(s): " + ", ".join(dup_cols))
            X = X.drop(columns=dup_cols)

        print(f"  Design matrix shape: {X.shape}")
        print(f"  Design matrix rank:  {np.linalg.matrix_rank(X.to_numpy())}")

        # --------------------------------------------------
        # Try standard logit first
        # --------------------------------------------------
        fit_type = None
        result_med = None

        try:
            model = sm.Logit(y, X)
            result_med = model.fit(disp=0, maxiter=200)
            fit_type = "standard"
            print("Standard logistic regression fit succeeded.\n")

        except Exception as e:
            print(f"Standard logistic regression failed: {type(e).__name__}: {e}")
            print("Trying penalized logistic regression instead.\n")

            # penalized fallback avoids singular-matrix crash
            model = sm.Logit(y, X)
            result_med = model.fit_regularized(method="l1", alpha=0.01, disp=0)
            fit_type = "penalized"

        # --------------------------------------------------
        # Output
        # --------------------------------------------------
        try:
            print(result_med.summary())
        except Exception:
            print("Model fit completed, but no printable summary was available.")
            print()

        coef = result_med.params

        report = pd.DataFrame({
            "Coef": coef,
            "OR": np.exp(coef),
        })

        # add CI and p-values where available
        try:
            conf = result_med.conf_int()
            if isinstance(conf, pd.DataFrame):
                conf.columns = ["CI_low", "CI_high"]
                report["OR_95%_CI_low"] = np.exp(conf["CI_low"])
                report["OR_95%_CI_high"] = np.exp(conf["CI_high"])
            else:
                report["OR_95%_CI_low"] = np.exp(conf[:, 0])
                report["OR_95%_CI_high"] = np.exp(conf[:, 1])
        except Exception:
            report["OR_95%_CI_low"] = np.nan
            report["OR_95%_CI_high"] = np.nan

        try:
            report["p"] = result_med.pvalues
        except Exception:
            report["p"] = np.nan

        print(f"Odds ratios and 95% CI (emo_use model; {fit_type} fit):")
        print(report.round(4).to_string())

###  Side Effects based on HITOP P instead of PHQ-4

In [ ]:

# --------------------------------------------------
# 1) Base sample: AI users
# --------------------------------------------------
df_sub = df_reg.loc[df_reg["ai_use"] == 1].copy()

# --------------------------------------------------
# 2) Side-effect coding (numeric only)
# --------------------------------------------------
side_effect_cols = {
    "ai_feel_dependent": "AI dependency",
    "ai_hard_to_decide_alone": "Decision difficulty",
    "ai_friends_relationship_change": "Social impact",
}

SOCIAL_YES_NUM = {1, 2, 3}

def to_yes_no(v, col):
    if pd.isna(v):
        return np.nan

    num = pd.to_numeric(v, errors="coerce")
    if pd.isna(num):
        return np.nan

    # -----------------------------
    # SPECIAL CASE: social variable
    # -----------------------------
    if col == "ai_friends_relationship_change":
        return 1 if int(num) in SOCIAL_YES_NUM else 0

    # -----------------------------
    # GENERAL CASE: agreement = 2–4
    # -----------------------------
    return 1 if 2 <= num <= 4 else 0


# apply coding
for col in side_effect_cols:
    ycol = f"{col}_yes"
    if col in df_sub.columns:
        df_sub[ycol] = df_sub[col].apply(lambda x: to_yes_no(x, col))
    else:
        df_sub[ycol] = np.nan

# -----------------------------
# Counts per side-effect item
# -----------------------------
print("Per-side-effect Yes/No/Missing counts:")

for col, label in side_effect_cols.items():
    ycol = f"{col}_yes"

    n_yes = int((df_sub[ycol] == 1).sum())
    n_no = int((df_sub[ycol] == 0).sum())
    n_miss = int(df_sub[ycol].isna().sum())

    n_ans = n_yes + n_no
    pct_yes = (100 * n_yes / n_ans) if n_ans > 0 else np.nan

    if n_ans > 0:
        print(f"  {label:<35} Yes={n_yes:>6}  No={n_no:>6}  Missing={n_miss:>6}  Yes%={pct_yes:.1f}")
    else:
        print(f"  {label:<35} Yes={n_yes:>6}  No={n_no:>6}  Missing={n_miss:>6}  Yes%=n/a")


# --------------------------------------------------
# 3) Person-level outcome: ANY side effect
# --------------------------------------------------
ycols = [f"{c}_yes" for c in side_effect_cols.keys()]

answered_any = df_sub[ycols].notna().any(axis=1)
any_yes = df_sub[ycols].eq(1).any(axis=1)

df_sub["side_effect_any"] = np.where(
    ~answered_any,
    np.nan,
    np.where(any_yes, 1, 0)
)

print("Outcome distribution:")
print(df_sub["side_effect_any"].value_counts(dropna=False))


# --------------------------------------------------
# 4) Gender: ONLY female/male
# --------------------------------------------------
df_sub["gender_cat_label"] = df_sub["gender_cat_label"].where(
    df_sub["gender_cat_label"].isin(["female", "male"])
)


# --------------------------------------------------
# 5) Complete-case regression sample
# --------------------------------------------------
required = [
    "side_effect_any",
    "age_years",
    "gender_cat_label",
    "education_years",
    "employed",
]

print("\nDiagnostic: row loss for side_effect regression")
print(f"  Starting N (AI users):          {len(df_sub):>6}")

remaining = df_sub.copy()
for c in required:
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} lost: {lost:>6}")

print(f"  Final complete cases:           {len(remaining):>6}")
print()

df_reg_se = df_sub.dropna(subset=required).copy()


# --------------------------------------------------
# 6) Ensure types
# --------------------------------------------------
df_reg_se["side_effect_any"] = df_reg_se["side_effect_any"].astype(int)
df_reg_se["employed"] = pd.to_numeric(df_reg_se["employed"], errors="coerce").astype(int)
df_reg_se["age_years"] = pd.to_numeric(df_reg_se["age_years"], errors="coerce")
df_reg_se["education_years"] = pd.to_numeric(df_reg_se["education_years"], errors="coerce")
df_reg_se["behandlung"] = pd.to_numeric(df_reg_se["h_psycho_treatment"], errors="coerce")
df_reg_se["diagnose_selbst"] = pd.to_numeric(df_reg_se["h_dx_mental_ever"], errors="coerce")
df_reg_se["eq5d_daily"] = pd.to_numeric(df_reg_se["eq5d_usual_activities"], errors="coerce")

# --------------------------------------------------
# 7) Drop constant predictors
# --------------------------------------------------
preds = ["age_years", "gender_cat_label", "education_years", "employed"]
preds = [p for p in preds if df_reg_se[p].nunique(dropna=True) > 1]


# --------------------------------------------------
# 8) Build formula
# --------------------------------------------------
formula_terms = []
for p in preds:
    if p == "gender_cat_label":
        formula_terms.append("C(gender_cat_label)")
    else:
        formula_terms.append(p)

formula_fit = "side_effect_any ~ " + " + ".join(formula_terms)

print(f"Fitting: {formula_fit}")
print(f"N = {len(df_reg_se)}")


# --------------------------------------------------
# 9) Logistic regression (robust)
# --------------------------------------------------
try:
    result_se = smf.logit(formula_fit, data=df_reg_se).fit(disp=0)
    print(result_se.summary())

    coef = result_se.params
    conf = result_se.conf_int()
    conf.columns = ["CI_low", "CI_high"]

    report = pd.DataFrame({
        "Coef": coef,
        "OR": np.exp(coef),
        "OR_95%_CI_low": np.exp(conf["CI_low"]),
        "OR_95%_CI_high": np.exp(conf["CI_high"]),
        "p": result_se.pvalues,
    })

    print("\nOdds ratios and 95% CI:")
    print(report.round(4).to_string())

except Exception as e:
    print("Model failed:", e)

    # Debugging (if singular matrix etc.)
    print("\nDEBUG outcome distribution:")
    print(df_reg_se["side_effect_any"].value_counts())

    print("\nDEBUG cross-tabs:")
    print(pd.crosstab(df_reg_se["gender_cat_label"], df_reg_se["side_effect_any"]))


# --------------------------------------------------
# 5) Complete-case regression sample
# --------------------------------------------------
required = [
    "side_effect_any",
    "age_years",
    "gender_cat_label",
    "education_years",
    "employed",
    "P", 
    "behandlung", 
    "diagnose_selbst", 
    "eq5d_daily"]

print("\nDiagnostic: row loss for side_effect regression")
print(f"  Starting N (AI users):          {len(df_reg_se):>6}")

remaining = df_reg_se.copy()
for c in required:
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} lost: {lost:>6}")

print(f"  Final complete cases:           {len(remaining):>6}")
print()

df_reg_se = df_reg_se.dropna(subset=required).copy()


# --------------------------------------------------
# 6) Ensure types
# --------------------------------------------------
df_reg_se["side_effect_any"] = df_reg_se["side_effect_any"].astype(int)
df_reg_se["employed"] = pd.to_numeric(df_reg_se["employed"], errors="coerce").astype(int)
df_reg_se["age_years"] = pd.to_numeric(df_reg_se["age_years"], errors="coerce")
df_reg_se["education_years"] = pd.to_numeric(df_reg_se["education_years"], errors="coerce")




# --------------------------------------------------
# 7) Drop constant predictors
# --------------------------------------------------
preds = ["age_years", "gender_cat_label", "education_years", "employed",
          "P", "behandlung", "diagnose_selbst", "eq5d_daily",]
preds = [p for p in preds if df_reg_se[p].nunique(dropna=True) > 1]


# --------------------------------------------------
# 8) Build formula
# --------------------------------------------------
formula_terms = []
for p in preds:
    if p == "gender_cat_label":
        formula_terms.append("C(gender_cat_label)")
    else:
        formula_terms.append(p)

formula_fit = "side_effect_any ~ " + " + ".join(formula_terms)

print(f"Fitting: {formula_fit}")
print(f"N = {len(df_reg_se)}")


# --------------------------------------------------
# 9) Logistic regression (robust)
# --------------------------------------------------
try:
    result_se = smf.logit(formula_fit, data=df_reg_se).fit(disp=0)
    print(result_se.summary())

    coef = result_se.params
    conf = result_se.conf_int()
    conf.columns = ["CI_low", "CI_high"]

    report = pd.DataFrame({
        "Coef": coef,
        "OR": np.exp(coef),
        "OR_95%_CI_low": np.exp(conf["CI_low"]),
        "OR_95%_CI_high": np.exp(conf["CI_high"]),
        "p": result_se.pvalues,
    })

    print("\nOdds ratios and 95% CI:")
    print(report.round(4).to_string())

except Exception as e:
    print("Model failed:", e)

    # Debugging (if singular matrix etc.)
    print("\nDEBUG outcome distribution:")
    print(df_reg_se["side_effect_any"].value_counts())

    print("\nDEBUG cross-tabs:")
    print(pd.crosstab(df_reg_se["gender_cat_label"], df_reg_se["side_effect_any"]))

### Sensitivity Analysis: Categorical Education

In [ ]:

df_reg_emo = df_reg.loc[df_reg["ai_use_any"] == 1].copy()
# --------------------------------------------------
# 3) Build additional predictors
# --------------------------------------------------
df_reg_emo["phq4_sum"] = pd.to_numeric(df_reg_emo["h_phq4_total"], errors="coerce")
df_reg_emo["behandlung"] = pd.to_numeric(df_reg_emo["h_psycho_treatment"], errors="coerce")
df_reg_emo["diagnose_selbst"] = pd.to_numeric(df_reg_emo["h_dx_mental_ever"], errors="coerce")
df_reg_emo["eq5d_daily"] = pd.to_numeric(df_reg_emo["eq5d_usual_activities"], errors="coerce")

# keep only female/male; everything else -> NaN
df_reg_emo["gender_cat_label"] = df_reg_emo["gender_cat_label"].where(
    df_reg_emo["gender_cat_label"].isin(["female", "male"]),
    np.nan
)

# convert predictors BEFORE complete-case restriction
for c in [
    "emo_use", "age_years", "employed",
]:
    df_reg_emo[c] = pd.to_numeric(df_reg_emo[c], errors="coerce")
edu5_order = ["low", "medium", "high", "tertiary_adv", "doctorate", "unknown"]
df_reg_emo["edu5"] = pd.Categorical(df_reg_emo["edu5"], categories=edu5_order, ordered=True)

# --------------------------------------------------
# 4) Complete-case regression sample
# --------------------------------------------------
required = [
    "emo_use", "age_years", "gender_cat_label", "edu5", "employed",

]

print("Diagnostic: row loss for emo_use extended regression (AI users only)")
print(f"  Starting N (AI users):          {len(df_reg_emo):>6}")

remaining = df_reg_emo.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")

print(f"  Final complete cases:           {len(remaining):>6}")
print()

df_reg_emo = df_reg_emo.dropna(subset=required).copy()
df_reg_emo["edu5"] = df_reg_emo["edu5"].cat.remove_unused_categories()

# final types after CC restriction
df_reg_emo["emo_use"] = df_reg_emo["emo_use"].astype(int)
df_reg_emo["employed"] = df_reg_emo["employed"].astype(int)

df_reg_emo["gender_cat_label"] = df_reg_emo["gender_cat_label"].astype("category")
df_reg_emo["gender_cat_label"] = df_reg_emo["gender_cat_label"].cat.remove_unused_categories()

print(f"  Analysis N (final complete cases): {len(df_reg_emo):,}")
print("  emo_use counts:")
print(df_reg_emo["emo_use"].value_counts(dropna=False).sort_index())
print("  gender counts:")
print(df_reg_emo["gender_cat_label"].value_counts(dropna=False))
print()

# --------------------------------------------------
# 5) Logistic regression
# --------------------------------------------------
preds = [
    "age_years", "gender_cat_label", "edu5", "employed",

]

print("Model: emo_use ~ age_years + C(gender_cat_label) + edu5 + employed")

if len(df_reg_emo) < 10:
    print("Too few complete cases to fit model safely.")

elif df_reg_emo["emo_use"].nunique() < 2:
    print("emo_use has only one class after filtering; model not fit.")

else:
    # drop constant raw predictors first
    keep_preds = [p for p in preds if df_reg_emo[p].nunique(dropna=True) > 1]
    dropped_raw = [p for p in preds if p not in keep_preds]

    if dropped_raw:
        print("Dropped constant raw predictor(s): " + ", ".join(dropped_raw))

    if not keep_preds:
        print("All predictors are constant; model not fit.")

    else:
        formula_terms = []
        for p in keep_preds:
            if p == "gender_cat_label":
                formula_terms.append('C(gender_cat_label)')
            elif p == "edu5":
                formula_terms.append('C(edu5, Treatment(reference="medium"))')
            else:
                formula_terms.append(p)

        formula_fit = "emo_use ~ " + " + ".join(formula_terms)
        print(f"Fitting: {formula_fit}")

        # --------------------------------------------------
        # Build design matrix explicitly
        # --------------------------------------------------
        y, X = patsy.dmatrices(formula_fit, data=df_reg_emo, return_type="dataframe")
        y = np.ravel(y)

        # drop constant design-matrix columns except intercept
        const_cols = [c for c in X.columns if c != "Intercept" and X[c].nunique(dropna=False) <= 1]
        if const_cols:
            print("Dropped constant design-matrix column(s): " + ", ".join(const_cols))
            X = X.drop(columns=const_cols)

        # drop exact duplicate columns
        dup_cols = []
        cols = X.columns.tolist()
        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                if X.iloc[:, i].equals(X.iloc[:, j]):
                    dup_cols.append(cols[j])

        dup_cols = list(dict.fromkeys(dup_cols))
        if dup_cols:
            print("Dropped duplicate design-matrix column(s): " + ", ".join(dup_cols))
            X = X.drop(columns=dup_cols)

        print(f"  Design matrix shape: {X.shape}")
        print(f"  Design matrix rank:  {np.linalg.matrix_rank(X.to_numpy())}")

        # --------------------------------------------------
        # Try standard logit first
        # --------------------------------------------------
        fit_type = None
        result_med = None

        try:
            model = sm.Logit(y, X)
            result_med = model.fit(disp=0, maxiter=200)
            fit_type = "standard"
            print("Standard logistic regression fit succeeded.\n")

        except Exception as e:
            print(f"Standard logistic regression failed: {type(e).__name__}: {e}")
            print("Trying penalized logistic regression instead.\n")

            # penalized fallback avoids singular-matrix crash
            model = sm.Logit(y, X)
            result_med = model.fit_regularized(method="l1", alpha=0.01, disp=0)
            fit_type = "penalized"

        # --------------------------------------------------
        # Output
        # --------------------------------------------------
        try:
            print(result_med.summary())
        except Exception:
            print("Model fit completed, but no printable summary was available.")
            print()

        coef = result_med.params

        report = pd.DataFrame({
            "Coef": coef,
            "OR": np.exp(coef),
        })

        # add CI and p-values where available
        try:
            conf = result_med.conf_int()
            if isinstance(conf, pd.DataFrame):
                conf.columns = ["CI_low", "CI_high"]
                report["OR_95%_CI_low"] = np.exp(conf["CI_low"])
                report["OR_95%_CI_high"] = np.exp(conf["CI_high"])
            else:
                report["OR_95%_CI_low"] = np.exp(conf[:, 0])
                report["OR_95%_CI_high"] = np.exp(conf[:, 1])
        except Exception:
            report["OR_95%_CI_low"] = np.nan
            report["OR_95%_CI_high"] = np.nan

        try:
            report["p"] = result_med.pvalues
        except Exception:
            report["p"] = np.nan

        print(f"Odds ratios and 95% CI (emo_use model; {fit_type} fit):")
        print(report.round(4).to_string())

In [ ]:
df_reg_emo["side_effect_any"].value_counts()

### Sensitivity Analysis: Side Effects based on Categorical Education

In [ ]:

df_reg_emo = df_reg.loc[df_reg["ai_use_any"] == 1].copy()
# --------------------------------------------------
# 3) Build additional predictors
# --------------------------------------------------
df_reg_emo["phq4_sum"] = pd.to_numeric(df_reg_emo["h_phq4_total"], errors="coerce")
df_reg_emo["behandlung"] = pd.to_numeric(df_reg_emo["h_psycho_treatment"], errors="coerce")
df_reg_emo["diagnose_selbst"] = pd.to_numeric(df_reg_emo["h_dx_mental_ever"], errors="coerce")
df_reg_emo["eq5d_daily"] = pd.to_numeric(df_reg_emo["eq5d_usual_activities"], errors="coerce")

# keep only female/male; everything else -> NaN
df_reg_emo["gender_cat_label"] = df_reg_emo["gender_cat_label"].where(
    df_reg_emo["gender_cat_label"].isin(["female", "male"]),
    np.nan
)

# convert predictors BEFORE complete-case restriction
for c in [
    "side_effect_any", "age_years", "employed",
]:
    df_reg_emo[c] = pd.to_numeric(df_reg_emo[c], errors="coerce")
edu5_order = ["low", "medium", "high", "tertiary_adv", "doctorate", "unknown"]
df_reg_emo["edu5"] = pd.Categorical(df_reg_emo["edu5"], categories=edu5_order, ordered=True)

# --------------------------------------------------
# 4) Complete-case regression sample
# --------------------------------------------------
required = [
    "side_effect_any", "age_years", "gender_cat_label", "edu5", "employed",

]

print("Diagnostic: row loss for side_effect_any extended regression (AI users only)")
print(f"  Starting N (AI users):          {len(df_reg_emo):>6}")

remaining = df_reg_emo.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")

print(f"  Final complete cases:           {len(remaining):>6}")
print()

df_reg_emo = df_reg_emo.dropna(subset=required).copy()
df_reg_emo["edu5"] = df_reg_emo["edu5"].cat.remove_unused_categories()

# final types after CC restriction
df_reg_emo["side_effect_any"] = df_reg_emo["side_effect_any"].astype(int)
df_reg_emo["employed"] = df_reg_emo["employed"].astype(int)

df_reg_emo["gender_cat_label"] = df_reg_emo["gender_cat_label"].astype("category")
df_reg_emo["gender_cat_label"] = df_reg_emo["gender_cat_label"].cat.remove_unused_categories()

print(f"  Analysis N (final complete cases): {len(df_reg_emo):,}")
print(" side_effect_any counts:")
print(df_reg_emo["side_effect_any"].value_counts(dropna=False).sort_index())
print("  gender counts:")
print(df_reg_emo["gender_cat_label"].value_counts(dropna=False))
print()

# --------------------------------------------------
# 5) Logistic regression
# --------------------------------------------------
preds = [
    "age_years", "gender_cat_label", "edu5", "employed",

]

print("Model: side_effect_any ~ age_years + C(gender_cat_label) + edu5 + employed")

if len(df_reg_emo) < 10:
    print("Too few complete cases to fit model safely.")

elif df_reg_emo["side_effect_any"].nunique() < 2:
    print("side_effect_any has only one class after filtering; model not fit.")

else:
    # drop constant raw predictors first
    keep_preds = [p for p in preds if df_reg_emo[p].nunique(dropna=True) > 1]
    dropped_raw = [p for p in preds if p not in keep_preds]

    if dropped_raw:
        print("Dropped constant raw predictor(s): " + ", ".join(dropped_raw))

    if not keep_preds:
        print("All predictors are constant; model not fit.")

    else:
        formula_terms = []
        for p in keep_preds:
            if p == "gender_cat_label":
                formula_terms.append('C(gender_cat_label)')
            elif p == "edu5":
                formula_terms.append('C(edu5, Treatment(reference="medium"))')
            else:
                formula_terms.append(p)

        formula_fit = "side_effect_any ~ " + " + ".join(formula_terms)
        print(f"Fitting: {formula_fit}")

        # --------------------------------------------------
        # Build design matrix explicitly
        # --------------------------------------------------
        y, X = patsy.dmatrices(formula_fit, data=df_reg_emo, return_type="dataframe")
        y = np.ravel(y)

        # drop constant design-matrix columns except intercept
        const_cols = [c for c in X.columns if c != "Intercept" and X[c].nunique(dropna=False) <= 1]
        if const_cols:
            print("Dropped constant design-matrix column(s): " + ", ".join(const_cols))
            X = X.drop(columns=const_cols)

        # drop exact duplicate columns
        dup_cols = []
        cols = X.columns.tolist()
        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                if X.iloc[:, i].equals(X.iloc[:, j]):
                    dup_cols.append(cols[j])

        dup_cols = list(dict.fromkeys(dup_cols))
        if dup_cols:
            print("Dropped duplicate design-matrix column(s): " + ", ".join(dup_cols))
            X = X.drop(columns=dup_cols)

        print(f"  Design matrix shape: {X.shape}")
        print(f"  Design matrix rank:  {np.linalg.matrix_rank(X.to_numpy())}")

        # --------------------------------------------------
        # Try standard logit first
        # --------------------------------------------------
        fit_type = None
        result_med = None

        try:
            model = sm.Logit(y, X)
            result_med = model.fit(disp=0, maxiter=200)
            fit_type = "standard"
            print("Standard logistic regression fit succeeded.\n")

        except Exception as e:
            print(f"Standard logistic regression failed: {type(e).__name__}: {e}")
            print("Trying penalized logistic regression instead.\n")

            # penalized fallback avoids singular-matrix crash
            model = sm.Logit(y, X)
            result_med = model.fit_regularized(method="l1", alpha=0.01, disp=0)
            fit_type = "penalized"

        # --------------------------------------------------
        # Output
        # --------------------------------------------------
        try:
            print(result_med.summary())
        except Exception:
            print("Model fit completed, but no printable summary was available.")
            print()

        coef = result_med.params

        report = pd.DataFrame({
            "Coef": coef,
            "OR": np.exp(coef),
        })

        # add CI and p-values where available
        try:
            conf = result_med.conf_int()
            if isinstance(conf, pd.DataFrame):
                conf.columns = ["CI_low", "CI_high"]
                report["OR_95%_CI_low"] = np.exp(conf["CI_low"])
                report["OR_95%_CI_high"] = np.exp(conf["CI_high"])
            else:
                report["OR_95%_CI_low"] = np.exp(conf[:, 0])
                report["OR_95%_CI_high"] = np.exp(conf[:, 1])
        except Exception:
            report["OR_95%_CI_low"] = np.nan
            report["OR_95%_CI_high"] = np.nan

        try:
            report["p"] = result_med.pvalues
        except Exception:
            report["p"] = np.nan

        print(f"Odds ratios and 95% CI (side_effect_any model; {fit_type} fit):")
        print(report.round(4).to_string())

### Dose Effect Analysis

In [ ]:

# --------------------------------------------------
# 1) Start from AI users only
# --------------------------------------------------
df_emo_int = df_reg.loc[df_reg["ai_use_any"] == 1].copy()

# --------------------------------------------------
# 2) Emotional-use columns
# --------------------------------------------------
emotional_cols = [
    "ai_psych_sit_anxiety_worry",
    "ai_psych_sit_self_doubt_low_mood",
    "ai_psych_sit_loneliness",
    "ai_psych_sit_relationship_conflict",
    "ai_psych_sit_self_reflection",
    "ai_psych_sit_grief_loss",
    "ai_psych_sit_impulsive_or_substance",
    "ai_psych_sit_acute_crisis",
    "ai_psych_sit_other_selected",
]

emotional_cols = [c for c in emotional_cols if c in df_emo_int.columns]

# --------------------------------------------------
# 3) Additional predictors
# --------------------------------------------------
df_emo_int["gender_cat_label"] = df_emo_int["gender_cat_label"].where(
    df_emo_int["gender_cat_label"].isin(["female", "male"]),
    np.nan
)

# --------------------------------------------------
# 4) Recode emotional item intensities
#    1-4 stay as intensity
#    5 = never -> 0
#    everything else -> NaN
# --------------------------------------------------
for c in emotional_cols:
    df_emo_int[c] = pd.to_numeric(df_emo_int[c], errors="coerce")
    df_emo_int[f"{c}_int"] = np.where(
        df_emo_int[c].isin([1, 2, 3, 4]), df_emo_int[c],
        np.where(df_emo_int[c] == 5, 0, np.nan)
    )

int_cols = [f"{c}_int" for c in emotional_cols]

# Any emotional AI use: at least one item with actual use intensity > 0
has_emotional_use = df_emo_int[int_cols].gt(0).any(axis=1)

# Mean intensity across answered items
# 0 = explicit non-use, 1-4 = use intensity, NaN = missing
df_emo_int["emo_intensity_sum"] = df_emo_int[int_cols].sum(axis=1, skipna=True)

# If someone answered none of the emotional-use items at all, set mean to NaN
answered_n = df_emo_int[int_cols].notna().sum(axis=1)
df_emo_int.loc[answered_n == 0, "emo_intensity_sum"] = np.nan

# Keep only people with any emotional AI use
df_reg_int = df_emo_int.loc[has_emotional_use].copy()

# --------------------------------------------------
# 5) Complete-case regression sample
# --------------------------------------------------
required = [
    "side_effect_any",
    "emo_intensity_sum",
    "age_years",
    "gender_cat_label",
    "education_years",
    "employed",
]

print("Diagnostic: row loss for emotional AI-use intensity -> side effects model")
print(f"  Starting N (AI users):                       {len(df_emo_int):>6}")
print(f"  With >=1 emotional-use response (1-4):      {int(has_emotional_use.sum()):>6}")

remaining = df_reg_int.copy()
for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<25} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")

print(f"  Final complete cases:                        {len(remaining):>6}")
print()

df_reg_int = df_reg_int.dropna(subset=required).copy()

# Final types
for c in ["side_effect_any", "emo_intensity_sum", "age_years", "education_years", "employed"]:
    df_reg_int[c] = pd.to_numeric(df_reg_int[c], errors="coerce")

df_reg_int = df_reg_int.dropna(subset=required).copy()

df_reg_int["side_effect_any"] = df_reg_int["side_effect_any"].astype(int)
df_reg_int["employed"] = df_reg_int["employed"].astype(int)

df_reg_int["gender_cat_label"] = df_reg_int["gender_cat_label"].astype("category")
df_reg_int["gender_cat_label"] = df_reg_int["gender_cat_label"].cat.remove_unused_categories()

print("Logistic regression among emotional AI users:")
print("  Outcome: side_effect_any")
print("  Predictor: emo_intensity_sum")
print("  Coding: 1-4 = intensity, 5 = never -> 0, mean across answered emotional-use items")
print(f"  Analysis N (complete cases): {len(df_reg_int):,}")
print()

# --------------------------------------------------
# 6) Descriptives
# --------------------------------------------------
label_map = {1: "Selten", 2: "Manchmal", 3: "Oft", 4: "Immer"}

pooled = df_reg_int[int_cols].stack()
pooled = pooled[pooled.isin([1, 2, 3, 4])]
counts = pooled.value_counts().sort_index()

print("Intensity response frequencies (pooled across emotional-use items):")
for k in [1, 2, 3, 4]:
    print(f"  {label_map[k]}: {int(counts.get(k, 0))}")
print()

print("Per-item intensity frequencies:")
for c in int_cols:
    vc = df_reg_int[c].value_counts().sort_index()
    print(f"  {c}: " + ", ".join([f"{label_map[k]}={int(vc.get(k, 0))}" for k in [1, 2, 3, 4]]))
print()

print("Predictor descriptives:")
print(df_reg_int["emo_intensity_sum"].describe().round(3).to_string())
print()

print("Outcome frequencies:")
print(df_reg_int["side_effect_any"].value_counts(dropna=False).sort_index().to_string())
print()

# --------------------------------------------------
# 7) Logistic regression
# --------------------------------------------------
preds_int = [
    "emo_intensity_sum",
    "age_years",
    "gender_cat_label",
    "education_years",
    "employed",
]

if len(df_reg_int) < 10:
    print("Too few complete cases to fit model safely.")

else:
    keep_int = [p for p in preds_int if df_reg_int[p].nunique(dropna=True) > 1]
    drop_int = [p for p in preds_int if p not in keep_int]

    if drop_int:
        print("Dropped constant predictor(s): " + ", ".join(drop_int))

    if "emo_intensity_sum" not in keep_int:
        print("emo_intensity_sum is constant or missing; model not fit.")

    else:
        formula_terms = []
        for p in keep_int:
            if p == "gender_cat_label":
                formula_terms.append("C(gender_cat_label)")
            else:
                formula_terms.append(p)

        formula_int = "side_effect_any ~ " + " + ".join(formula_terms)
        print("Model:", formula_int)

        try:
            result_int = smf.logit(formula_int, data=df_reg_int).fit(disp=0)
            print(result_int.summary())

            coef = result_int.params
            conf = result_int.conf_int()
            conf.columns = ["CI_low_logit", "CI_high_logit"]
            pvals = result_int.pvalues

            report = pd.DataFrame({
                "OR": np.exp(coef),
                "CI_low": np.exp(conf["CI_low_logit"]),
                "CI_high": np.exp(conf["CI_high_logit"]),
                "p": pvals
            })

            print("\nOdds ratios with 95% CI:")
            print(report.round(4).to_string())

        except Exception as e:
            print(f"Model fit failed: {e}")

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# --------------------------------------------------
# 1) Start from AI users only
# --------------------------------------------------
df_work = df_reg.loc[df_reg["ai_use_any"] == 1].copy()

# --------------------------------------------------
# 2) Emotional-use columns
# --------------------------------------------------
emotional_cols = [
    "ai_psych_sit_anxiety_worry",
    "ai_psych_sit_self_doubt_low_mood",
    "ai_psych_sit_loneliness",
    "ai_psych_sit_relationship_conflict",
    "ai_psych_sit_self_reflection",
    "ai_psych_sit_grief_loss",
    "ai_psych_sit_impulsive_or_substance",
    "ai_psych_sit_acute_crisis",
    "ai_psych_sit_other_selected",
]
emotional_cols = [c for c in emotional_cols if c in df_work.columns]

# --------------------------------------------------
# 3) Additional predictors
# --------------------------------------------------
df_work["gender_cat_label"] = df_work["gender_cat_label"].where(
    df_work["gender_cat_label"].isin(["female", "male"]),
    np.nan
)

# --------------------------------------------------
# 4) Recode emotional item intensities
#    1-4 stay as intensity
#    5 = never -> 0
#    everything else -> NaN
# --------------------------------------------------
for c in emotional_cols:
    df_work[c] = pd.to_numeric(df_work[c], errors="coerce")
    df_work[f"{c}_int"] = np.where(
        df_work[c].isin([1, 2, 3, 4]), df_work[c],
        np.where(df_work[c] == 5, 0, np.nan)
    )

int_cols = [f"{c}_int" for c in emotional_cols]

# at least one actual emotional AI use (>0)
has_emotional_use = df_work[int_cols].gt(0).any(axis=1)

# require known intensity on all emotional-use items, as in your comparison block
known_intensity = df_work[int_cols].notna().all(axis=1)

# sum across emotional-use items
df_work["emo_intensity_sum"] = df_work[int_cols].sum(axis=1)

# emotional-use sample
df_emo = df_work.loc[has_emotional_use & known_intensity, ["emo_intensity_sum"]].copy()

print(f"(A) Emotional intensity sample (AI users, has emotional use + known intensity): {len(df_emo):,}")

# --------------------------------------------------
# 5) Side-effect intensity
#    Build mean intensity among experienced side effects only
# --------------------------------------------------

# Social relationship change:
# original coding assumed:
# 1 = much worse, 2 = somewhat worse, 3 = a little worse,
# 4-7 = neutral / better / no worsening
df_work["ai_friends_relationship_change_negative"] = df_work["ai_friends_relationship_change"].map({
    1: 3,
    2: 2,
    3: 1,
    4: 0,
    5: 0,
    6: 0,
    7: 0
})

# These two are assumed to run from 1-4, so recode to 0-3
df_work["ai_feel_dependent_int"] = pd.to_numeric(df_work["ai_feel_dependent"], errors="coerce") - 1
df_work["ai_hard_to_decide_alone_int"] = pd.to_numeric(df_work["ai_hard_to_decide_alone"], errors="coerce") - 1

side_effect_cols = [
    "ai_friends_relationship_change_negative",
    "ai_feel_dependent_int",
    "ai_hard_to_decide_alone_int"
]

# numeric safety
df_work[side_effect_cols] = df_work[side_effect_cols].apply(pd.to_numeric, errors="coerce")

# any answered side-effect item?
answered_any_se = df_work[side_effect_cols].notna().any(axis=1)
df_se = df_work.loc[answered_any_se].copy()

# number of actually experienced side effects (>0)
df_se["n_side_effects_experienced"] = (df_se[side_effect_cols] > 0).sum(axis=1)

# keep only rows with at least one experienced side effect
df_se = df_se.loc[df_se["n_side_effects_experienced"] > 0].copy()

# sum and mean intensity
df_se["side_effect_intensity_sum"] = df_se[side_effect_cols].fillna(0).sum(axis=1)
df_se["side_effect_intensity_mean"] = (
    df_se["side_effect_intensity_sum"] / df_se["n_side_effects_experienced"]
)

df_se_int = df_se[["side_effect_intensity_mean"]].copy()

print(f"(B) Side-effect intensity sample (AI users, answered any SE + >=1 experienced): {len(df_se_int):,}")

# --------------------------------------------------
# 6) Intersect both samples
# --------------------------------------------------
work = (
    df_work.drop(columns=["emo_intensity_sum"], errors="ignore")
           .join(df_emo, how="inner")
           .join(df_se_int, how="inner")
)

# --------------------------------------------------
# 7) Complete-case regression sample
# --------------------------------------------------
required = [
    "side_effect_intensity_mean",
    "emo_intensity_sum",
    "age_years",
    "gender_cat_label",
    "education_years",
    "employed",
]

remaining = work.copy()

print("\nDiagnostic: row loss for emotional AI-use intensity -> side-effect intensity model")
print(f"  Starting N (AI users):                               {len(df_work):>6}")
print(f"  With emotional use + known intensity:               {len(df_emo):>6}")
print(f"  With >=1 experienced side effect:                   {len(df_se_int):>6}")
print(f"  In intersection before complete-case filtering:     {len(work):>6}")

for c in required:
    n_valid = int(remaining[c].notna().sum())
    n_miss = int(remaining[c].isna().sum())
    before = len(remaining)
    remaining = remaining.dropna(subset=[c])
    lost = before - len(remaining)
    print(f"  {c:<28} valid: {n_valid:>6}  missing: {n_miss:>6}  (lost {lost:>6})")

print(f"  Final complete cases:                               {len(remaining):>6}")
print()

df_reg_int = work.dropna(subset=required).copy()

# type safety
for c in ["side_effect_intensity_mean", "emo_intensity_sum", "age_years", "education_years"]:
    df_reg_int[c] = pd.to_numeric(df_reg_int[c], errors="coerce")

df_reg_int["employed"] = pd.to_numeric(df_reg_int["employed"], errors="coerce")
df_reg_int = df_reg_int.dropna(subset=required).copy()
df_reg_int["employed"] = df_reg_int["employed"].astype(int)

df_reg_int["gender_cat_label"] = df_reg_int["gender_cat_label"].astype("category")
df_reg_int["gender_cat_label"] = df_reg_int["gender_cat_label"].cat.remove_unused_categories()

print("OLS regression among emotional AI users with at least one experienced side effect:")
print("  Outcome: side_effect_intensity_mean")
print("  Predictor: emo_intensity_sum")
print(f"  Analysis N (complete cases): {len(df_reg_int):,}")
print()

# --------------------------------------------------
# 8) Descriptives
# --------------------------------------------------
label_map = {1: "Selten", 2: "Manchmal", 3: "Oft", 4: "Immer"}

print("Predictor descriptives:")
print(df_reg_int["emo_intensity_sum"].describe().round(3).to_string())
print()

print("Outcome descriptives:")
print(df_reg_int["side_effect_intensity_mean"].describe().round(3).to_string())
print()

df_reg_int["n_emotional_domains_used"] = df_reg_int[int_cols].gt(0).sum(axis=1)
print("Number of emotional-use domains endorsed per person:")
print(df_reg_int["n_emotional_domains_used"].value_counts().sort_index().to_string())
print()

pooled_use_only = df_reg_int[int_cols].where(df_reg_int[int_cols].gt(0)).stack()
counts = pooled_use_only.value_counts().sort_index()
total_endorsed = int(counts.sum())

print("Item-level intensity frequencies across all endorsed emotional-use responses:")
for k in [1, 2, 3, 4]:
    n_k = int(counts.get(k, 0))
    pct_k = 100 * n_k / total_endorsed if total_endorsed > 0 else np.nan
    print(f"  {label_map[k]}: {n_k} ({pct_k:.1f}%)")
print(f"  Total endorsed item responses: {total_endorsed}")
print()

print("Per-item intensity frequencies:")
for c in int_cols:
    vc = df_reg_int[c].value_counts().sort_index()
    use_n = int(df_reg_int[c].gt(0).sum())
    parts = []
    for k in [1, 2, 3, 4]:
        n_k = int(vc.get(k, 0))
        pct_k = 100 * n_k / use_n if use_n > 0 else np.nan
        parts.append(f"{label_map[k]}={n_k} ({pct_k:.1f}%)")
    print(f"  {c}: " + ", ".join(parts))
print()

# --------------------------------------------------
# 9) OLS regression with HC3 robust SE
# --------------------------------------------------
preds_int = [
    "emo_intensity_sum",
    "age_years",
    "gender_cat_label",
    "education_years",
    "employed",
]

if len(df_reg_int) < 10:
    print("Too few complete cases to fit model safely.")

else:
    keep_int = [p for p in preds_int if df_reg_int[p].nunique(dropna=True) > 1]
    drop_int = [p for p in preds_int if p not in keep_int]

    if drop_int:
        print("Dropped constant predictor(s): " + ", ".join(drop_int))

    if "emo_intensity_sum" not in keep_int:
        print("emo_intensity_sum is constant or missing; model not fit.")

    else:
        formula_terms = []
        for p in keep_int:
            if p == "gender_cat_label":
                formula_terms.append("C(gender_cat_label)")
            else:
                formula_terms.append(p)

        formula_int = "side_effect_intensity_mean ~ " + " + ".join(formula_terms)
        print("Model:", formula_int)

        try:
            result_int = smf.ols(formula_int, data=df_reg_int).fit(cov_type="HC3")
            print(result_int.summary())

            coef = result_int.params
            conf = result_int.conf_int()
            pvals = result_int.pvalues

            report = pd.DataFrame({
                "b": coef,
                "CI_low": conf[0],
                "CI_high": conf[1],
                "p": pvals
            })

            print("\nCoefficients with 95% CI:")
            print(report.round(4).to_string())

        except Exception as e:
            print(f"Model fit failed: {e}")